# Moving-block holdout bootstrap for XANES reference selection

Given one measured spectrum and a pool of reference spectra, which combination of references
best explains it, and how sure can we be? This notebook develops the machinery behind MrFitty's
answer on the `example/arsenic` data, and then interrogates each choice it makes.

Both questions are answered by one estimator: a **moving-block holdout bootstrap**. The
bootstrap half answers "how sure" by refitting — rather than trusting a single fit it
manufactures many slightly different versions of the same spectrum, the same fitted curve plus a
reshuffled copy of the leftover noise, and takes the spread of the results as the measure of
confidence. The **holdout** half borrows from cross-validation: each refit is scored only at
energies it was not fitted to, so a model earns its score by predicting spectrum it has not seen.

Both halves have to respect one property of this data. The residuals of a XANES fit are strongly
autocorrelated — where the fitted curve runs above the measurement it stays above it across a
stretch of adjacent energies, because what the model missed is a smooth feature many points wide
rather than independent point-to-point noise. An ordinary bootstrap assumes independent
residuals and understates the uncertainty. Resampling in contiguous **blocks** carries that
dependence into the bootstrap, and holding out contiguous **blocks** keeps the scored energies
from being near-duplicates of the fitted ones next door. "Moving" block means blocks may start
anywhere and may overlap, rather than coming from one fixed partition.

Why prediction error rather than goodness of fit: adding a reference can only improve the fit,
since three references cannot fit worse than two, so residual size cannot say how many
references are justified. Prediction error can — a reference that is only absorbing noise helps
at the energies it was fitted to and hurts at the ones it was not.

## How to read this

**Part 1** builds the pipeline, one section per step, and runs it on five unknowns.
**Part 2** is six studies, each interrogating one choice the pipeline makes. Every study asks
its question twice: once against **synthetic data, where the right answer is known and recovery
can be scored**, and once against the **five real unknowns, where it matters**. Each ends in a
**Findings** write-up that states its numbers in the prose, so the conclusions read without
running anything.

The studies, in order:

1. [Linear against cubic spline interpolation](#study-interpolation) — which one rebuilds a
   XANES curve from fewer points, and whether the choice reaches the selection.
2. [Choosing the block length](#study-block-length) — whether the estimator recovers a
   dependence length that is known, and what to make of one unknown asking for 32.
3. [Resample length against holdout length](#study-resample-length) — the two were one number
   until now, and they want opposite things.
4. [What prediction error is measuring](#study-whiteline) — mostly whether the holdout took the
   energies the references disagree about, and whether that half of the test is fair.
5. [Which interval, and what counts as equally good](#study-selection) — the estimator the tie
   rules need, and what the rules do with it.
6. [Correlation against cosine reference distance](#study-distance) — which recovers a planted
   grouping, and whether the two trees disagree about anything that matters.

## What this costs to run

A single combination search — 2,324 reference combinations at 1,000 bootstrap iterations — takes
about 24 seconds. Part 1 does five of them. The studies in Part 2 are built on synthetic
replicates, each of which is a full search, so the expensive ones run into tens of minutes on 32
cores; each says what it costs above the cell that runs it.

Study results are cached as Parquet under `notebooks/study_results/`, which is **not** committed
— a fresh clone recomputes everything on its first run. Pass `recompute=True` to force a redraw.
Fit files are cached under `notebooks/fit_cache/`, also uncommitted, at about 24 MB each.

## Where the derivations went

This notebook is a rewrite. Several conclusions it states in a paragraph were argued at length in
[`moving_block_holdout_bootstrap_development.ipynb`](moving_block_holdout_bootstrap_development.ipynb),
which is frozen and no longer maintained. The largest is the development of the holdout selector
through five versions; this notebook keeps only the last and cites that comparison. Where a
finding here supersedes one there, it says so.

In [ ]:
# Everything the notebook imports, in one place. The originals accumulated these at the top of
# whichever cell first needed them, which made it impossible to see the dependency surface.
import base64
import collections
import contextlib
import datetime
import fnmatch
import functools
import hashlib
import io
import itertools
import json
import os
import tempfile
import time
import warnings

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.cluster.hierarchy as hc
import scipy.optimize
import scipy.stats
from collections import Counter
from itertools import combinations
from joblib import Parallel, delayed
from math import comb
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import NullFormatter
from matplotlib.transforms import blended_transform_factory
from scipy.interpolate import make_interp_spline
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from sklearn.metrics import adjusted_rand_score

import mrfitty
from mrfitty.base import ReferenceSpectrum
from mrfitty.linear_model import OlsWithStats

%matplotlib inline

# All cores. Set to 1 to run every sweep in this process instead, which is what the
# determinism checks and any debugging want.
N_JOBS = -1

# Where cached work lives. Neither directory is committed: a fit file is about 24 MB, and the
# study summaries are cheap to regenerate and would go stale against the code.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__')) or '.'
FIT_CACHE_DIR = os.path.join(NOTEBOOK_DIR, 'fit_cache')
STUDY_RESULTS_DIR = os.path.join(NOTEBOOK_DIR, 'study_results')

In [ ]:
# Naming the figures.
#
# Every figure this notebook draws carries, in its bottom right corner, the name of the
# function that drew it. A figure pasted into a message or pulled out of a report is then
# traceable to the call that makes it, which matters here because most of the figures are
# assembled from several panel functions and the one worth calling is not guessable from
# what is drawn.
#
# None of this lives in the plotting functions. They carry a @names_its_figures line and
# nothing else; the stamping happens below.
import functools

FIGURE_SOURCE_STYLE = {'fontsize': 7, 'color': '0.55', 'ha': 'right', 'va': 'bottom'}
_figure_source_stack = []


def _stamp_figure_source(figure, source):
    """Write the producing function's name in the corner of a figure, once."""
    if getattr(figure, '_source_stamped', False):
        return
    figure._source_stamped = True
    figure.text(0.998, 0.004, f'{source}()', **FIGURE_SOURCE_STYLE)


# functools.wraps leaves the original reachable as __wrapped__, so re-running this cell
# rewraps the original rather than wrapping the wrapper and growing a chain of them.
_unstamped_figure = getattr(plt.figure, '__wrapped__', plt.figure)


@functools.wraps(_unstamped_figure)
def _figure_naming_its_source(*args, **kwargs):
    figure = _unstamped_figure(*args, **kwargs)
    if _figure_source_stack:
        _stamp_figure_source(figure, _figure_source_stack[-1])
    return figure


# plt.subplots builds its figure by calling the module-level plt.figure, so replacing that
# one name catches both ways of making a figure, and any helper that goes through either.
plt.figure = _figure_naming_its_source


def names_its_figures(fn):
    """Mark a plotting function, so every figure it creates is labelled with its name.

    The label goes on at creation rather than on the way out, which is what lets one
    mechanism cover both kinds of plotting function here: the ones that hand their figures
    back for the caller to display, and the older ones that draw their own and return
    nothing.

    A figure is named by the innermost marked function on the stack -- the one that actually
    created it. So a function that assembles a report out of figures from another marked
    function leaves their names alone, and each figure says what would redraw it rather than
    what happened to be running at the time.

    Stamping is idempotent, so re-marking a function or drawing into an existing figure
    cannot double up the label.
    """
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        _figure_source_stack.append(fn.__name__)
        try:
            return fn(*args, **kwargs)
        finally:
            _figure_source_stack.pop()
    return wrapper

In [ ]:
def run_arms(arms, n_jobs=N_JOBS):
    """Evaluate independent (callable, kwargs) tasks, returning results in task order.

    Task order rather than completion order, so that printed tables and stored figures do
    not depend on how the work happened to be scheduled.

    Parameters
    ----------
    arms   : list of (callable, kwargs dict) -- each evaluated as callable(**kwargs)
    n_jobs : int -- passed to joblib; -1 uses every core, 1 runs in this process, which
             is what the determinism check and any debugging want

    Returns
    -------
    list -- one result per task, in the order given
    """
    if n_jobs == 1:
        return [arm_fn(**kwargs) for arm_fn, kwargs in arms]
    return Parallel(n_jobs=n_jobs)(delayed(arm_fn)(**kwargs) for arm_fn, kwargs in arms)

In [ ]:
# ---------------------------------------------------------------------------
# Tests for the figure naming above.
#
# The mechanism is a patched plt.figure plus a stack, which is worth pinning down: the
# properties that make it safe are that it names the innermost marked function, that it
# leaves figures nobody claimed alone, and that it cannot stamp the same figure twice.
# ---------------------------------------------------------------------------

def _figure_stamps(figure):
    return [text.get_text() for text in figure.texts if text.get_text().endswith('()')]


def test_a_marked_function_names_its_figures():
    @names_its_figures
    def plot_one():
        figure, _ = plt.subplots()
        return figure

    figure = plot_one()
    try:
        assert _figure_stamps(figure) == ['plot_one()']
    finally:
        plt.close(figure)


def test_the_innermost_marked_function_names_the_figure():
    """A report assembled from another marked function must not relabel its figures."""
    @names_its_figures
    def plot_inner():
        return plt.figure()

    @names_its_figures
    def plot_outer():
        return plot_inner(), plt.figure()

    inner, outer = plot_outer()
    try:
        assert _figure_stamps(inner) == ['plot_inner()'], 'the one that drew it owns it'
        assert _figure_stamps(outer) == ['plot_outer()']
    finally:
        plt.close(inner)
        plt.close(outer)


def test_a_figure_nobody_claimed_is_left_alone():
    """Figures made outside any marked function -- in a cell, or a test -- stay unstamped."""
    figure, _ = plt.subplots()
    try:
        assert _figure_stamps(figure) == []
    finally:
        plt.close(figure)


def test_a_figure_is_named_once_however_often_it_is_handled():
    """Passing a figure back through another marked function must not double the label."""
    @names_its_figures
    def plot_first():
        return plt.figure()

    @names_its_figures
    def plot_again(figure):
        _stamp_figure_source(figure, 'plot_again')
        return figure

    figure = plot_again(plot_first())
    try:
        assert _figure_stamps(figure) == ['plot_first()']
    finally:
        plt.close(figure)


def test_the_stack_unwinds_when_a_plot_raises():
    """A failed plot must not leave its name on whatever is drawn next."""
    @names_its_figures
    def plot_broken():
        raise ValueError('no figure for you')

    try:
        plot_broken()
    except ValueError:
        pass
    assert _figure_source_stack == [], 'the stack should be empty again'


_figure_source_test_fns = [
    test_a_marked_function_names_its_figures,
    test_the_innermost_marked_function_names_the_figure,
    test_a_figure_nobody_claimed_is_left_alone,
    test_a_figure_is_named_once_however_often_it_is_handled,
    test_the_stack_unwinds_when_a_plot_raises,
]
for _fn in _figure_source_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_figure_source_test_fns)} tests passed')

# Part 1 — The pipeline

Eight steps, one section each, ending with all five unknowns fitted. Everything here is
machinery; the choices it makes are interrogated in Part 2.

1. [The spectra](#pipeline-spectra)
2. [The design matrix](#pipeline-design-matrix)
3. [The reference pool](#pipeline-reference-pool)
4. [Fitting, and the residuals](#pipeline-fit)
5. [Drawing the holdout blocks](#pipeline-holdout)
6. [The bootstrap](#pipeline-bootstrap)
7. [Choosing between combinations](#pipeline-selection)
8. [Fit files and the study cache](#pipeline-fit-files) — then [all five](#pipeline-all-five)

<a id="pipeline-spectra"></a>

## 1. The spectra

Twenty-four reference spectra and five unknowns, read from `example/arsenic`.

The unknowns are five of the sixteen available, chosen to span the chemistry rather than to
sample it evenly. All sixteen are mixtures of two arsenic species — reduced arsenic absorbing
near 11869.7 eV and arsenate near 11875.2 eV — and they lie along one mixing line between those
two, so most of the sixteen are near-duplicates of one another. Three samples are represented
(`OTT3_55`, `Ott3_73`, `Ott3_74`) across fifteen distinct spots; spots within a sample differ
more than spots across samples do.

These five take both ends of the line, the middle, and a second sample at one end:

| unknown | why |
|---|---|
| `OTT3_55_spot0` | the primary. Every finding recorded in the development notebook is about this one, so it is kept for continuity. |
| `Ott3_73_AsXANES_spot5_000` | most arsenate-dominated |
| `Ott3_73_AsXANES_spot1_avg` | most reduced-dominated |
| `Ott3_73_AsXANES_spot6_000` | very nearly half and half |
| `Ott3_74_AsXANES_spot0` | arsenate again, from a different sample |

Every unknown fully contains the references' common window of 11830–12097 eV, with at least
52 eV of headroom at the low end, so none of them constrains the fit.

In [ ]:
src_path, _ = os.path.split(mrfitty.__path__[0])
sample_data_dir_path = os.path.join(src_path, 'example', 'arsenic')

reference_spectra_list = sorted(
    ReferenceSpectrum.read_all([os.path.join(sample_data_dir_path, 'reference/*.e')])[0],
    key=lambda s: s.file_name,
)
unknown_spectra_list = sorted(
    ReferenceSpectrum.read_all([os.path.join(sample_data_dir_path, 'unknown/*.e')])[0],
    key=lambda s: s.file_name,
)
print(f'{len(reference_spectra_list)} references, {len(unknown_spectra_list)} unknowns available')


def filter_spectra_by_name(spectra_list, *patterns):
    """Spectra whose file_name matches any of the glob-style patterns, in the order given.

    ReferenceSpectrum.read_all hands back an unordered set, so something like this is needed
    whatever else changes.
    """
    matches = []
    for pattern in patterns:
        hits = [s for s in spectra_list if fnmatch.fnmatch(s.file_name, pattern)]
        if not hits:
            raise ValueError(
                f'no spectrum matched {pattern!r}. Available: '
                f'{[s.file_name for s in spectra_list]}'
            )
        matches.extend(sorted(hits, key=lambda s: s.file_name))
    return matches


# The five unknowns, in the order the tables and figures report them: primary first, then
# arsenate end, reduced end, the middle, and the second sample.
UNKNOWN_NAMES = (
    'OTT3_55_spot0.e',
    'Ott3_73_AsXANES_spot5_000.e',
    'Ott3_73_AsXANES_spot1_avg.e',
    'Ott3_73_AsXANES_spot6_000.e',
    'Ott3_74_AsXANES_spot0.e',
)
PRIMARY_UNKNOWN = UNKNOWN_NAMES[0]

reference_spectra = filter_spectra_by_name(reference_spectra_list, '*')
unknown_spectra = filter_spectra_by_name(unknown_spectra_list, *UNKNOWN_NAMES)
primary_spectrum = unknown_spectra[0]
ref_names = [r.file_name for r in reference_spectra]

print(f'\nfitting {len(unknown_spectra)} unknowns against {len(reference_spectra)} references')
for spectrum in unknown_spectra:
    marker = '  <- primary' if spectrum.file_name == PRIMARY_UNKNOWN else ''
    print(f'  {spectrum.file_name}{marker}')

<a id="pipeline-design-matrix"></a>

## 2. The design matrix

Every reference is measured on its own energy grid, so each must be resampled onto the
unknown's grid before anything can be fitted. `interpolate_references_at_sample_energies`
does that over the energy range common to the sample and every reference — no extrapolation —
and returns the design matrix `A` (one column per reference), the response vector `b` (the
sample's normalized absorption), and which spectra bound each end of the range.

The interpolant is a parameter rather than a constant, because which one to use is a choice
the pipeline makes and [a study below](#study-interpolation) interrogates it. The default is
the cubic spline, matching `mrfitty.base.ReferenceSpectrum`.

In [ ]:
def make_linear_interpolant(energies, norm):
    """Piecewise-linear interpolant through (energies, norm)."""
    return make_interp_spline(energies, norm, k=1)


def make_cubic_spline_interpolant(energies, norm):
    """Interpolating cubic spline through (energies, norm)."""
    return make_interp_spline(energies, norm, k=3)


def interpolate_references_at_sample_energies(
    reference_spectra, sample_spectrum, make_interpolant=make_cubic_spline_interpolant,
    verbose=True,
):
    """Interpolate reference spectra onto the sample spectrum's energy grid.

    The usable energy range is the intersection of the sample spectrum's range
    and every reference spectrum's range, so no extrapolation occurs.  Each
    reference is resampled by building an interpolant through its own measured
    energies and evaluating it at the sample energies.

    Parameters
    ----------
    reference_spectra : list of ReferenceSpectrum
        Pool of reference spectra to interpolate.  Each must expose a
        ``data_df`` attribute indexed by energy (eV) with a ``norm`` column.
    sample_spectrum : Spectrum or ReferenceSpectrum
        The unknown spectrum to be fitted.  Must expose a ``data_df`` attribute
        whose index contains energy values (eV) and whose ``norm`` column
        contains the normalized fluorescence values used as the regression
        response vector.
    verbose : bool, optional
        Report the range, the limiting spectra and each reference's interpolated
        extent. Off when fitting several unknowns in a row, where it is 24 lines of
        per-reference detail each time.
    make_interpolant : callable, optional
        Factory called as ``make_interpolant(energies, norm)`` returning a
        callable that evaluates the reference at arbitrary energies.  Defaults
        to ``make_cubic_spline_interpolant``, matching the cubic spline
        ``ReferenceSpectrum`` builds internally.  Pass
        ``make_linear_interpolant`` to resample linearly instead.  This choice
        sits upstream of the design matrix, the fit coefficients, and the
        prediction error, so it can affect which reference combination is
        selected -- see the interpolation method comparison at the end of this
        notebook.

    Returns
    -------
    valid_energies : ndarray, shape (n,)
        Energy values (eV) at which interpolation was performed — the
        intersection of the sample spectrum's range and all reference ranges.
        Depends only on the measured energy ranges, not on ``make_interpolant``.
    A : ndarray, shape (n, n_refs)
        Design matrix A for linear regression: column i holds reference i
        interpolated at ``valid_energies``.
    b : ndarray, shape (n,)
        Sample spectrum normalized fluorescence values at ``valid_energies``.
        This is the response vector for linear regression against A.
    low_limiters : list
        The spectrum object(s) whose lower bound sets ``valid_energies[0]``
        (the highest lower bound).  More than one when several tie exactly.
        The sample spectrum is a candidate alongside the references.
    high_limiters : list
        The spectrum object(s) whose upper bound sets ``valid_energies[-1]``
        (the lowest upper bound).  More than one when several tie exactly.
    """
    say = print if verbose else (lambda *args, **kwargs: None)
    energies = sample_spectrum.data_df.index.values
    say(f'sample_spectrum: {sample_spectrum.file_name}')
    say(f'energy range: {energies[0]:.2f}–{energies[-1]:.2f} eV ({len(energies)} points)')
    say(f'references: {len(reference_spectra)}')
    say(f'interpolation: {make_interpolant.__name__}')

    # restrict to energies covered by the sample_spectrum AND every reference to avoid extrapolation.
    # collect (spectrum, low, high) for the sample and every reference so we can report and return
    # which spectra limit each end of the common range.
    spectrum_ranges = [
        (s, s.data_df.index.values[0], s.data_df.index.values[-1])
        for s in (sample_spectrum, *reference_spectra)
    ]

    # energy_min is set by the highest lower bound; energy_max by the lowest upper bound.
    # report every spectrum tied at each limiting energy, not just the first.
    energy_min = max(low for _, low, _ in spectrum_ranges)
    energy_max = min(high for _, _, high in spectrum_ranges)
    low_limiters = [s for s, low, _ in spectrum_ranges if low == energy_min]
    high_limiters = [s for s, _, high in spectrum_ranges if high == energy_max]

    say(f'lowest valid energy {energy_min:.2f} eV limited by '
          f'{", ".join(s.file_name for s in low_limiters)}')
    say(f'highest valid energy {energy_max:.2f} eV limited by '
          f'{", ".join(s.file_name for s in high_limiters)}')

    valid_mask = (energies >= energy_min) & (energies <= energy_max)
    valid_energies = energies[valid_mask]
    n_excluded = (~valid_mask).sum()
    if n_excluded:
        say(f'excluded {n_excluded} energies outside common range '
              f'({energy_min:.2f}–{energy_max:.2f} eV)')
    say(f'interpolating at {len(valid_energies)} energies '
          f'({valid_energies[0]:.2f}–{valid_energies[-1]:.2f} eV)')

    A = np.zeros((len(valid_energies), len(reference_spectra)))
    for i, ref in enumerate(reference_spectra):
        # build the interpolant from the reference's own measured points rather than
        # using a pre-built one, so make_interpolant actually selects the method
        reference_interpolant = make_interpolant(
            ref.data_df.index.values, ref.data_df['norm'].values,
        )
        A[:, i] = reference_interpolant(valid_energies)
        say(f'  {ref.file_name}: norm [{A[:, i].min():.4f}, {A[:, i].max():.4f}]')

    b = sample_spectrum.data_df['norm'].values[valid_mask]

    return valid_energies, A, b, low_limiters, high_limiters

In [ ]:
# ---------------------------------------------------------------------------
# Test fixtures: a minimal stand-in for ReferenceSpectrum / Spectrum
# ---------------------------------------------------------------------------
# interpolate_references_at_sample_energies() only ever touches two things on
# the spectra it is handed:
#   * .file_name              - a label used in the printed / returned report
#   * .data_df                - a pandas DataFrame indexed by energy (eV) with a
#                               'norm' column of normalized fluorescence values
# It builds its own interpolant from .data_df via the make_interpolant argument,
# so a real spectrum's pre-built .interpolant is never read and the fixture does
# not need to supply one.
# FakeSpectrum supplies exactly those two, so the tests can drive the function
# with tiny, fully-controlled inputs instead of reading real .e files from disk.
import numpy as np
import pandas as pd


class FakeSpectrum:
    def __init__(self, file_name, energies, norm):
        # store energy/norm as float arrays so integer test grids behave like real data
        energies = np.asarray(energies, dtype=float)
        norm = np.asarray(norm, dtype=float)
        self.file_name = file_name
        # energy is the index; 'norm' is the single column the function reads
        self.data_df = pd.DataFrame({'norm': norm}, index=energies)


def test_returns_intersection_range_shapes_and_values():
    # Sample grid spans 0..10 on integer eV; its 'norm' is a simple ramp (10*E)
    # so the expected response vector b is trivial to read off.
    sample = FakeSpectrum('sample', np.arange(0, 11), np.arange(0, 11) * 10.0)

    # The references sit on grids shifted a fraction of an eV off the sample grid,
    # so none of their nodes line up with the sample energies and the function
    # must genuinely interpolate them.  Each reference's 'norm' is still a linear
    # function of energy, and linear interpolation reproduces a linear function
    # exactly, so the expected values at the sample energies stay simple.
    offset = 0.3
    ref_a_energies = np.arange(2, 11) - offset   # 1.7 .. 9.7 -> highest lower bound (1.7 eV)
    ref_a = FakeSpectrum('ref_a', ref_a_energies, 2.0 * ref_a_energies)   # norm = 2*E
    ref_b_energies = np.arange(0, 9) + offset    # 0.3 .. 8.3 -> lowest upper bound (8.3 eV)
    ref_b = FakeSpectrum('ref_b', ref_b_energies, ref_b_energies + 1.0)   # norm = E + 1

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # Common range is the intersection [max(lows), min(highs)] = [1.7, 8.3]; the
    # valid energies are the SAMPLE grid points falling inside it, i.e. 2..8.
    np.testing.assert_array_equal(valid_energies, np.array([2, 3, 4, 5, 6, 7, 8], dtype=float))

    # A: one row per valid energy, one column per reference, in the input order.
    assert A.shape == (7, 2)
    # Column 0 is ref_a interpolated onto valid_energies (2*E), column 1 is ref_b (E+1).
    # These hold despite the offset grids because linear interp of linear data is exact.
    np.testing.assert_allclose(A[:, 0], 2.0 * valid_energies)
    np.testing.assert_allclose(A[:, 1], valid_energies + 1.0)

    # b is the sample's 'norm' (10*E) sampled at the valid energies - no interpolation.
    np.testing.assert_allclose(b, 10.0 * valid_energies)


def test_identifies_low_and_high_limiters():
    # 'norm' values are irrelevant here, so use zeros; only the energy bounds matter.
    sample = FakeSpectrum('sample', np.arange(0, 11), np.zeros(11))
    ref_a = FakeSpectrum('ref_a', np.arange(2, 11), np.zeros(9))   # starts latest (2 eV)
    ref_b = FakeSpectrum('ref_b', np.arange(0, 9), np.zeros(9))    # ends earliest (8 eV)

    _, _, _, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # ref_a alone has the highest lower bound, so it alone limits the low end;
    # ref_b alone has the lowest upper bound, so it alone limits the high end.
    assert low_limiters == [ref_a]
    assert high_limiters == [ref_b]


def test_reports_all_tied_limiters():
    sample = FakeSpectrum('sample', np.arange(0, 11), np.zeros(11))
    # ref_a and ref_c share the exact same starting energy (2 eV) ...
    ref_a = FakeSpectrum('ref_a', np.arange(2, 11), np.zeros(9))
    ref_c = FakeSpectrum('ref_c', np.arange(2, 11), np.zeros(9))
    # ... and ref_b and ref_d share the exact same ending energy (8 eV).
    ref_b = FakeSpectrum('ref_b', np.arange(0, 9), np.zeros(9))
    ref_d = FakeSpectrum('ref_d', np.arange(0, 9), np.zeros(9))

    _, _, _, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_c, ref_b, ref_d], sample)

    # Every spectrum tied at a limiting energy is reported (not just the first),
    # in the order the function scans them: sample first, then the references as given.
    assert low_limiters == [ref_a, ref_c]
    assert high_limiters == [ref_b, ref_d]


def test_sample_spectrum_can_be_the_limiter():
    # The sample is NARROWER than both references (3..7 vs 0..10), so the sample
    # itself limits BOTH ends and none of its grid points are excluded.
    sample = FakeSpectrum('sample', np.arange(3, 8), np.zeros(5))
    ref_a = FakeSpectrum('ref_a', np.arange(0, 11), np.zeros(11))
    ref_b = FakeSpectrum('ref_b', np.arange(0, 11), np.zeros(11))

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # No sample energies fall outside the common range, so the full grid is kept.
    np.testing.assert_array_equal(valid_energies, np.arange(3, 8, dtype=float))
    assert A.shape == (5, 2)
    # The sample is the tightest spectrum at each end, so it is the sole limiter.
    assert low_limiters == [sample]
    assert high_limiters == [sample]


def test_identical_ranges_exclude_nothing():
    # When the sample and reference share the same grid, the whole grid is valid
    # and the design matrix has exactly one column.
    grid = np.arange(0, 6)
    sample = FakeSpectrum('sample', grid, np.arange(0, 6) * 1.0)
    ref_a = FakeSpectrum('ref_a', grid, np.arange(0, 6) * 1.0)

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a], sample)

    np.testing.assert_array_equal(valid_energies, grid.astype(float))
    assert A.shape == (6, 1)
    # With identical bounds, both the sample and the reference tie at each edge.
    assert low_limiters == [sample, ref_a]
    assert high_limiters == [sample, ref_a]


def test_interpolation_method_is_selectable():
    # Every other fixture in this file uses linear (or zero) 'norm' data, which both
    # a linear and a cubic interpolant reproduce exactly -- so those tests cannot tell
    # the two methods apart. This one uses data with genuine curvature.
    #
    # The reference sits on a COARSE grid (every 2 eV) and is sampled on a FINER grid
    # (every 1 eV), so half the sample energies fall strictly between reference nodes
    # and must actually be interpolated. That is the same situation as the real data,
    # where several references are measured every 1.05 eV against a 0.5 eV sample grid.
    def cubic_norm(energy):
        return 0.01 * energy ** 3 - 0.2 * energy ** 2 + energy + 1.0

    sample_energies = np.arange(0, 21, 1)
    sample = FakeSpectrum('sample', sample_energies, np.zeros(len(sample_energies)))
    reference_energies = np.arange(0, 21, 2)
    reference = FakeSpectrum('curved_ref', reference_energies, cubic_norm(reference_energies))

    _, A_cubic, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_cubic_spline_interpolant,
    )
    _, A_linear, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_linear_interpolant,
    )

    # An interpolating cubic spline through samples of a cubic polynomial reproduces
    # that polynomial exactly, so the cubic arm is right to floating-point tolerance.
    np.testing.assert_allclose(A_cubic[:, 0], cubic_norm(sample_energies), atol=1e-10)

    # The linear arm chords across each 2 eV gap and so must NOT match, and the
    # disagreement has to be large enough to matter rather than a rounding artifact.
    assert not np.allclose(A_linear[:, 0], cubic_norm(sample_energies), atol=1e-10)
    assert np.abs(A_linear[:, 0] - A_cubic[:, 0]).max() > 0.05

    # Both methods interpolate rather than extrapolate, so they agree exactly wherever
    # a sample energy coincides with a reference node -- the difference is confined to
    # the points between nodes.
    on_node = np.isin(sample_energies, reference_energies)
    np.testing.assert_allclose(A_linear[on_node, 0], A_cubic[on_node, 0], atol=1e-10)
    assert np.abs(A_linear[~on_node, 0] - A_cubic[~on_node, 0]).max() > 0.05


def test_default_interpolation_is_cubic():
    # Pins the default. Everything in this notebook that was produced before
    # make_interpolant existed -- the v1-v5 holdout comparison, the fits, the
    # prediction error tables -- was computed with the cubic spline
    # ReferenceSpectrum builds internally, so the default has to stay cubic for
    # those results to remain reproducible.
    def cubic_norm(energy):
        return 0.01 * energy ** 3 - 0.2 * energy ** 2 + energy + 1.0

    sample_energies = np.arange(0, 21, 1)
    sample = FakeSpectrum('sample', sample_energies, np.zeros(len(sample_energies)))
    reference_energies = np.arange(0, 21, 2)
    reference = FakeSpectrum('curved_ref', reference_energies, cubic_norm(reference_energies))

    _, A_default, _, _, _ = interpolate_references_at_sample_energies([reference], sample)
    _, A_cubic, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_cubic_spline_interpolant,
    )

    np.testing.assert_array_equal(A_default, A_cubic)

_interpolation_test_fns = [
    test_returns_intersection_range_shapes_and_values,
    test_identifies_low_and_high_limiters,
    test_reports_all_tied_limiters,
    test_sample_spectrum_can_be_the_limiter,
    test_identical_ranges_exclude_nothing,
    test_interpolation_method_is_selectable,
    test_default_interpolation_is_cubic,
]
for _fn in _interpolation_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_interpolation_test_fns)} tests passed')

In [ ]:
@names_its_figures
def plot_interpolated_references(
    valid_energies, A, b, low_limiters, high_limiters,
    reference_spectra, sample_spectrum, ax=None,
):
    """Visualize the output of interpolate_references_at_sample_energies.

    Built to stay legible from 3 references to 30+.  The sample is bold black;
    references are colored by the *role* they play, not one hue each, so the
    legend never explodes:
      * references that limit the low edge  -> blue
      * references that limit the high edge -> orange
      * references that limit both edges    -> green
      * every other reference               -> faint gray context (one legend row)
    Color is never the only cue: the legend names each role and its count, and
    the red dashed bound lines name the limiting spectra (or their count when many
    tie at the same energy).

    Parameters
    ----------
    valid_energies, A, b, low_limiters, high_limiters
        The five values returned by interpolate_references_at_sample_energies.
    reference_spectra : list of ReferenceSpectrum
        The same reference pool passed to that function; ``A``'s columns follow
        this order.
    sample_spectrum : Spectrum or ReferenceSpectrum
        The sample whose response vector ``b`` is plotted.
    ax : matplotlib Axes, optional
        Axis to draw on. A new figure/axis is created when omitted.

    Returns
    -------
    ax : matplotlib Axes
        The axis the plot was drawn on.
    """
    from collections import Counter
    from matplotlib.lines import Line2D

    if ax is None:
        _, ax = plt.subplots(figsize=(11, 5))

    low_ids = {id(s) for s in low_limiters}
    high_ids = {id(s) for s in high_limiters}
    role_color = {'low': 'tab:blue', 'high': 'tab:orange', 'low+high': 'tab:green'}

    def role_of(ref):
        r = [name for name, ids in (('low', low_ids), ('high', high_ids)) if id(ref) in ids]
        return '+'.join(r) if r else None

    # context (non-limiter) references first, faint gray, underneath everything
    n_other = 0
    for i, ref in enumerate(reference_spectra):
        if role_of(ref) is None:
            ax.plot(valid_energies, A[:, i], color='0.45', alpha=0.5, linewidth=0.8, zorder=1)
            n_other += 1

    # limiting references, colored by role; all co-limiters share the role color
    role_counts = Counter()
    for i, ref in enumerate(reference_spectra):
        role = role_of(ref)
        if role is None:
            continue
        ax.plot(valid_energies, A[:, i], color=role_color[role], alpha=0.75, linewidth=1.3, zorder=3)
        role_counts[role] += 1

    # the full sample spectrum at ALL its energies: the portion outside the common
    # range (where no references are interpolated) is drawn dimmed and dashed so
    # every sample energy is visible, ...
    sample_energies = sample_spectrum.data_df.index.values
    sample_norm = sample_spectrum.data_df['norm'].values
    faint_sample, = ax.plot(sample_energies, sample_norm, color='black', alpha=0.5,
                            linewidth=1, linestyle='--', zorder=2)
    # ... while the in-range response vector b is drawn bold and solid on top.
    sample_line, = ax.plot(valid_energies, b, color='black', linewidth=2, zorder=4)
    handles = [sample_line, faint_sample]
    labels = [f'{sample_spectrum.file_name} (sample, in range)',
              'sample (all energies)']

    # one legend row per non-empty limiter role (names live on the bound lines below)
    for role in ('low', 'high', 'low+high'):
        n = role_counts.get(role, 0)
        if n:
            handles.append(Line2D([], [], color=role_color[role], alpha=0.75, linewidth=1.3))
            labels.append(f'{role} limiter{"" if n == 1 else "s"} ({n})')

    # one proxy entry stands in for every faded reference
    if n_other:
        handles.append(Line2D([], [], color='0.45', alpha=0.7, linewidth=0.8))
        labels.append(f'other references ({n_other})')

    # exact common-range bounds (red dashed) and the sampled-energy extent (gray dotted).
    # every low/high limiter shares the limiting energy, so read it off the first one;
    # summarise the limiter names so a big tie does not blow out the legend width.
    def summarize(spectra):
        names = [s.file_name for s in spectra]
        return ', '.join(names) if len(names) <= 2 else f'{len(names)} spectra'

    energy_min = low_limiters[0].data_df.index.values[0]
    energy_max = high_limiters[0].data_df.index.values[-1]
    lo = ax.axvline(energy_min, color='tab:red', linestyle='--', linewidth=1, zorder=2)
    hi = ax.axvline(energy_max, color='tab:red', linestyle='--', linewidth=1, zorder=2)
    ax.axvline(valid_energies[0], color='gray', linestyle=':', linewidth=1, zorder=2)
    ax.axvline(valid_energies[-1], color='gray', linestyle=':', linewidth=1, zorder=2)
    handles += [lo, hi, Line2D([], [], color='gray', linestyle=':', linewidth=1)]
    labels += [
        f'low bound {energy_min:.1f} eV (limited by {summarize(low_limiters)})',
        f'high bound {energy_max:.1f} eV (limited by {summarize(high_limiters)})',
        f'sampled extent [{valid_energies[0]:.1f}, {valid_energies[-1]:.1f}] eV',
    ]

    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Normalized Fluorescence')
    ax.set_title(f'Interpolated references vs {sample_spectrum.file_name} '
                 f'({len(valid_energies)} energies, {len(reference_spectra)} references)')
    # legend outside the axes on the right so it never covers the data
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1.02, 1),
              fontsize='small', borderaxespad=0.0)
    return ax


# The design matrix for the primary unknown, and the references that build it.
primary_energies, primary_A, primary_b, primary_low_limiters, primary_high_limiters = \
    interpolate_references_at_sample_energies(
        reference_spectra=reference_spectra, sample_spectrum=primary_spectrum,
    )

display(plot_interpolated_references(
    primary_energies, primary_A, primary_b,
    primary_low_limiters, primary_high_limiters,
    reference_spectra, primary_spectrum,
))

<a id="pipeline-reference-pool"></a>

## 3. The reference pool, and where a combination sits in it

Twenty-four references are not twenty-four independent explanations. Several are
near-duplicates — two arsenopyrites, several sorbed arsenates — so a combination drawn from one
cluster of them says something much weaker than one spanning the pool, and the prediction error
alone cannot tell those apart.

`cluster_reference_spectra` builds the tree and sets its cutoff by resampling: the references
are shuffled within each energy to destroy the relationships between them while keeping each
energy's distribution, and the cutoff is the 95th percentile of the merge heights those
surrogates produce. A merge tighter than that is tighter than chance.

The dendrogram functions come in three kinds, by what is laid over the tree: nothing,
combinations marked by membership, or the whole pool shaded by a weight.

In [ ]:
def permute_within_rows(A, rng):
    """Shuffle the reference values within each energy row, independently row by row.

    This builds the comparison case the significance cutoff below is measured against --
    the *null model*, meaning a version of this data in which the thing being looked for
    is absent by construction. The thing being looked for here is resemblance between
    references, so each energy keeps exactly the values it measured while which reference
    holds which value is randomized. Every reference still looks like a plausible spectrum
    row by row, but any tendency for two of them to rise and fall together is gone.
    Clustering such a randomized copy shows how close references come to each other for no
    reason at all, which is the yardstick a real merge has to beat.

    `Generator.permuted(A, axis=1)` does this and returns a copy. It replaces the
    DataFrame idiom in the older clustering notebooks,
    `df.values[i, :] = shuffle(df.values[i, :])`, which is unreliable: `.values` may hand
    back a copy under pandas copy-on-write, in which case the assignment silently does
    nothing and the "randomized" copy is the real data again. The cutoff is then measured
    against the very merge heights it is supposed to judge, and certifies whatever it is
    given. `mrfitty/combination_fit.py` fixes that with `.iloc`; working on a numpy copy
    avoids the question.
    """
    return rng.permuted(A, axis=1)


def phase_randomize_columns(A, rng):
    """Surrogate references that keep each reference's own character but not its relatives.

    Another way to build the randomized copies, in place of `permute_within_rows`, done the
    standard surrogate-data way:
    Fourier transform each reference, replace the phases with uniform random ones, transform
    back. Every reference keeps its own power spectrum -- and so its mean, its variance and
    its smoothness -- while what it shares with the other references is destroyed.

    The intent was a stricter comparison than shuffling values within an energy: one that asks
    "are these two references more alike than two arbitrary spectra with this character?"
    rather than "is there any shared structure at all?". Measured, it is the *easier* of the
    two to beat -- see the surrogate comparison below, which is why `cluster_reference_spectra`
    still defaults to `permute_within_rows`. It is kept because the comparison is worth
    being able to re-run on another reference set.
    """
    n_energies = A.shape[0]
    spectrum = np.fft.rfft(A, axis=0)
    phases = rng.uniform(0.0, 2.0 * np.pi, size=spectrum.shape)
    phases[0, :] = 0.0                   # DC stays real, so each reference keeps its mean
    if n_energies % 2 == 0:
        phases[-1, :] = 0.0              # and so does Nyquist, where the length gives one
    return np.fft.irfft(np.abs(spectrum) * np.exp(1j * phases), n=n_energies, axis=0)


def cluster_reference_spectra(
    A, ref_names, rng, metric='correlation', method='complete',
    resample_count=1000, percentile=95.0, surrogate=permute_within_rows, verbose=True,
):
    """Hierarchically cluster the reference spectra that are the columns of A.

    Parameters
    ----------
    A : ndarray, shape (n_energies, n_refs)
        The design matrix from `interpolate_references_at_sample_energies`. Its columns
        are the references, so the clustering transposes it -- `pdist` treats *rows* as
        the observations. That A is free of NaN is load-bearing rather than incidental:
        `pdist` propagates a single NaN across a reference's entire distance row, and
        `linkage` then fails opaquely. A is NaN-free because it is built on the energy
        range common to the sample and every reference.
    ref_names : list of str
        Reference file names, parallel to A's columns.
    rng : numpy Generator
        Drives the randomized comparison copies only. The tree itself is deterministic,
        so a different seed moves the cutoff and nothing else.
    metric : str
        Any `pdist` metric. 'correlation' (the default, and what
        `mrfitty/combination_fit.py` uses) centers each reference before comparing, so it
        measures shape alone; 'cosine' does not center, so it measures the angle from the
        origin -- closer to the collinearity that makes two references interchangeable in
        a non-negative fit. Which one to use is settled by the correlation-vs-cosine
        comparison that follows the fits below.
    method : str
        Linkage method, passed to `scipy.cluster.hierarchy.linkage`.
    resample_count, percentile : int, float
        How the significance cutoff is built. `resample_count` times, the references are
        randomized against each other by `permute_within_rows` and reclustered; every
        merge height those randomized trees produce is pooled, and the cutoff is the
        `percentile`th of that pool. Read the result as "two references this close would
        hardly ever happen by chance": at the 95th percentile, only one merge in twenty of
        the randomized ones was that tight, so merges below the cutoff are the ones worth
        calling clusters. 1,000 replicates take about a tenth of a second at this size, so
        there is no reason to run fewer -- and no reason to reach for `run_arms`, whose
        process round trip would cost more than the work.
    surrogate : callable
        `surrogate(A, rng) -> ndarray`, how a randomized copy is built. The default,
        `permute_within_rows`, shuffles values between references at each energy;
        `phase_randomize_columns` is the alternative, and the comparison below measures what
        the choice costs.

    Returns
    -------
    dict carrying the tree and everything drawn or reported from it, so that nothing is
    recomputed at draw time: 'distances' (condensed), 'Z', 'cutoff_distance',
    'chance_merge_heights' (every merge height the randomized copies produced -- the
    distribution the cutoff is one percentile of), 'cophenetic_correlation',
    'cophenetic_distances', 'labels' (flat clusters at the cutoff), 'n_clusters', the
    parameters, and 'first_permutation_digest' -- see below.
    """
    A = np.ascontiguousarray(A, dtype=float)
    ref_names = list(ref_names)
    if A.shape[1] != len(ref_names):
        raise ValueError(f'A has {A.shape[1]} columns but {len(ref_names)} reference names were given')
    if A.shape[1] < 2:
        raise ValueError('clustering needs at least two references')
    if not np.isfinite(A).all():
        raise ValueError('A holds non-finite values, which pdist would propagate across whole '
                         'distance rows; build A over the common energy range first')

    # Catch the degenerate columns each metric cannot handle, naming the reference, rather
    # than letting pdist return NaN and linkage fail somewhere further down.
    if metric == 'correlation':
        degenerate = [ref_names[i] for i in np.flatnonzero(A.std(axis=0) == 0)]
        reason = 'is constant, so its correlation with anything is undefined'
    elif metric == 'cosine':
        degenerate = [ref_names[i] for i in np.flatnonzero(np.linalg.norm(A, axis=0) == 0)]
        reason = 'is all zeros, so its angle to anything is undefined'
    else:
        degenerate, reason = [], ''
    if degenerate:
        raise ValueError(f'{", ".join(degenerate)} {reason} under metric={metric!r}')

    def condensed_distances(matrix):
        # pdist clusters rows, so the design matrix is transposed to put references there.
        # The clip removes the ~1e-17 negatives 'correlation' can return for two nearly
        # identical references, which would otherwise become negative merge heights.
        return np.clip(pdist(np.ascontiguousarray(matrix.T), metric=metric), 0.0, None)

    distances = condensed_distances(A)
    Z = hc.linkage(distances, method=method)

    # Recluster `resample_count` randomized copies and pool every merge height they
    # produce. That pool is what "by chance" means for this particular reference set, and
    # the cutoff is one percentile of it.
    chance_merge_heights = np.empty(resample_count * (A.shape[1] - 1))
    first_permutation_digest = None
    for i in range(resample_count):
        permuted = surrogate(A, rng)
        if i == 0:
            # 40 bytes that let a comparison assert -- rather than assume -- that two arms
            # drew the same permutations, the way compare_interpolation_methods asserts its
            # holdout masks are shared.
            first_permutation_digest = hashlib.sha1(permuted.tobytes()).hexdigest()
        shuffled_Z = hc.linkage(condensed_distances(permuted), method=method)
        chance_merge_heights[i * (A.shape[1] - 1):(i + 1) * (A.shape[1] - 1)] = shuffled_Z[:, 2]
    cutoff_distance = float(np.percentile(chance_merge_heights, percentile))

    cophenetic_correlation, cophenetic_distances = hc.cophenet(Z, distances)
    labels = hc.fcluster(Z, t=cutoff_distance, criterion='distance')

    if verbose:
        print(f'clustered {len(ref_names)} references by {metric} distance, {method} linkage')
        print(f'  pairwise distance range: {distances.min():.5f}-{distances.max():.5f}')
        print(f'  merge height range:      {Z[:, 2].min():.5f}-{Z[:, 2].max():.5f} (root {Z[-1, 2]:.5f})')
        print(f'  cutoff: {cutoff_distance:.5f} ({percentile:g}th percentile of '
              f'{resample_count} randomized copies from {surrogate.__name__})')
        sizes = sorted((int(size) for size in np.bincount(labels)[1:]), reverse=True)
        print(f'  {labels.max()} cluster(s) at the cutoff, sizes {sizes}')
        print(f'  {(Z[:, 2] < cutoff_distance).sum()} of {Z.shape[0]} merges fall below it')
        print(f'  cophenetic correlation: {cophenetic_correlation:.4f}')
        if cutoff_distance >= Z[-1, 2]:
            print('  NOTE: the cutoff sits above the root, so no merge is significant at this '
                  'percentile -- every reference is its own cluster')

    return {
        'ref_names': ref_names,
        'metric': metric,
        'method': method,
        'percentile': percentile,
        'resample_count': resample_count,
        'surrogate': surrogate.__name__,
        'distances': distances,
        'Z': Z,
        'cutoff_distance': cutoff_distance,
        'chance_merge_heights': chance_merge_heights,
        'cophenetic_correlation': float(cophenetic_correlation),
        'cophenetic_distances': cophenetic_distances,
        'labels': labels,
        'n_clusters': int(labels.max()),
        'first_permutation_digest': first_permutation_digest,
    }

In [ ]:
def smallest_enclosing_subtree(Z, leaf_indices):
    """The smallest subtree of Z containing every leaf in leaf_indices.

    Walking Z once and unioning leaf sets, then taking the smallest set that contains the
    group, rather than scanning nodes by increasing height: the leaf sets of a linkage
    form a laminar family, so the smallest containing node is unique, whereas a
    height-ordered scan assumes merge heights increase monotonically -- true for the
    'complete' default but not for every method `linkage` accepts.

    Returns a dict with 'node' (linkage node id), 'height' (0 for a single leaf, which
    encloses itself), 'leaf_indices', 'size' and 'is_root'.
    """
    target = frozenset(int(i) for i in leaf_indices)
    if not target:
        raise ValueError('no references given to locate')

    n_leaves = Z.shape[0] + 1
    leaf_sets = {i: frozenset((i,)) for i in range(n_leaves)}
    for row, (left, right, _, _) in enumerate(Z):
        leaf_sets[n_leaves + row] = leaf_sets[int(left)] | leaf_sets[int(right)]

    size, node = min((len(leaves), node) for node, leaves in leaf_sets.items() if target <= leaves)
    return {
        'node': node,
        'height': 0.0 if node < n_leaves else float(Z[node - n_leaves, 2]),
        'leaf_indices': sorted(leaf_sets[node]),
        'size': size,
        'is_root': node == 2 * n_leaves - 2,
    }


def _normalize_highlight(highlight, ref_names):
    """{label: sorted leaf indices} from names, indices, or a NaN-padded ref_indices row."""
    if highlight is None:
        return {}
    groups = highlight if isinstance(highlight, dict) else {'highlighted': highlight}
    name_to_index = {name: i for i, name in enumerate(ref_names)}

    normalized = {}
    for label, refs in groups.items():
        # a bare string is iterable, and iterating it would look up one character at a time
        if isinstance(refs, str) or np.isscalar(refs):
            refs = [refs]
        indices = []
        for ref in refs:
            if isinstance(ref, str):
                if ref not in name_to_index:
                    raise ValueError(f'{ref!r} is not one of the clustered references')
                indices.append(name_to_index[ref])
            else:
                # results['ref_indices'] rows are float and NaN-padded to max_M; drop the
                # padding here, because int(nan) raises and astype(int) silently yields a
                # huge negative index
                value = float(ref)
                if np.isfinite(value):
                    indices.append(int(value))
        if not indices:
            raise ValueError(f'highlight group {label!r} names no references')
        normalized[label] = sorted(set(indices))
    return normalized


def best_subsets_by_size(results):
    """{'best M=m': [column indices]} for the lowest-median-PE combination at each size.

    The same selection `plot_best_subset_bootstrap_summaries` makes, reduced to what
    `plot_reference_dendrogram` highlights.
    """
    subsets = {}
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        medians = np.array([np.median(results['bootstrap_pes'][i]) for i in m_indices])
        best = m_indices[int(np.argmin(medians))]
        subsets[f'best M={m}'] = [int(j) for j in results['ref_indices'][best, :m]]
    return subsets

In [ ]:
# Drawing the reference tree.
#
# Three ways of drawing it, and they differ only in what is laid over the tree: nothing, a
# handful of combinations picked out by name, or the whole pool shaded by a weight. The
# tree itself, the cutoff line, the axis limits and the legend are the same either way, so
# they live in _draw_reference_tree and _finish_reference_tree and the three public
# functions are what is left once that is factored out.
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory

HIGHLIGHT_COLORS = ('tab:red', 'tab:blue', 'tab:green', 'tab:purple', 'tab:brown')
HIGHLIGHT_MARKERS = ('o', 's', '^', 'D', 'v')
HIGHLIGHT_LINESTYLES = ('-', (0, (4, 2)), (0, (1, 1.5)), (0, (6, 2, 1, 2)), (0, (3, 1, 1, 1, 1, 1)))

# A weight of 0 would be white and unreadable as text, so the ramp starts part way in and
# only its upper half is ever used. The bars carry the magnitude; the color carries the
# impression, and has to stay legible at the bottom of the range to do that.
LEAF_WEIGHT_CMAP = 'Reds'
LEAF_WEIGHT_FLOOR = 0.45
# The bars live in a gutter opened between the tree and the leaf labels, rather than inside
# the axes where they would sit on top of the branches. The gutter is made by pushing the
# tick labels right, so both measurements are in inches: a full-weight bar is
# LEAF_WEIGHT_BAR_INCHES long, and the labels start LEAF_WEIGHT_GUTTER_PAD further out.
LEAF_WEIGHT_BAR_INCHES = 0.55
LEAF_WEIGHT_GUTTER_PAD = 0.12


def _normalize_leaf_weights(leaf_weights, ref_names):
    """{leaf index: weight} from names or column indices, weights being fractions."""
    if not leaf_weights:
        return {}
    name_to_index = {name: i for i, name in enumerate(ref_names)}
    normalized = {}
    for ref, weight in leaf_weights.items():
        if isinstance(ref, str):
            if ref not in name_to_index:
                raise ValueError(f'{ref!r} is not one of the clustered references')
            index = name_to_index[ref]
        else:
            index = int(ref)
        if not 0.0 <= weight <= 1.0:
            raise ValueError(f'leaf weight for {ref!r} is {weight}, not a fraction')
        normalized[index] = float(weight)
    return normalized


def _draw_reference_tree(clustering, ax, color_threshold, title):
    """The tree, before anything is laid over it.

    Returns the record the annotation steps work against: the axis, where each leaf ended
    up, and the legend entries collected so far. Neither the legend nor the leaf label
    styling happens here, because an annotation has its own entries and styles to add and
    both can only be applied once.
    """
    ref_names = clustering['ref_names']
    Z = clustering['Z']

    if ax is None:
        _, ax = plt.subplots(figsize=(9, max(4.0, 0.34 * len(ref_names))))

    dendrogram = hc.dendrogram(
        Z, ax=ax, orientation='left', labels=ref_names,
        color_threshold=0.0 if color_threshold is None else color_threshold,
        above_threshold_color='0.35',
    )
    # With orientation='left' the x axis carries distance and is inverted (root at the
    # left, leaves at 0), y carries the leaves at 5, 15, 25, ... in drawn order. scipy
    # sizes the x axis from the root height alone, so a cutoff above the root would be
    # drawn off the axes; size it from both and then stop autoscaling.
    ax.set_xlim(max(1.05 * Z[-1, 2], 1.08 * clustering['cutoff_distance']), 0.0)
    ax.set_ylim(0.0, 10.0 * len(ref_names))
    ax.set_autoscale_on(False)

    ax.set_xlabel(f'{clustering["metric"]} distance ({clustering["method"]} linkage)')
    ax.set_title(title or f'{len(ref_names)} reference spectra by {clustering["metric"]} '
                          f'distance — {clustering["n_clusters"]} clusters at the cutoff')

    return {
        'ax': ax,
        'clustering': clustering,
        'leaves': dendrogram['leaves'],
        'leaf_position': {leaf: position
                          for position, leaf in enumerate(dendrogram['leaves'])},
        'root_height': Z[-1, 2],
        # x in axes coordinates, y in data coordinates, so an annotation sits at a fixed
        # depth across the axis while tracking the leaf it belongs to
        'marker_transform': blended_transform_factory(ax.transAxes, ax.transData),
        'handles': [],
        'labels': [],
        'leaf_styles': {},
    }


def _finish_reference_tree(tree, show_cutoff, legend_loc):
    """Apply whatever an annotation recorded, draw the cutoff, and build the one legend."""
    ax, clustering = tree['ax'], tree['clustering']

    # Tick labels ascend with y, the same order as the dendrogram's leaves, so the leaf
    # each one names is read off that rather than by matching the label text.
    for position, tick_label in enumerate(ax.get_ymajorticklabels()):
        style = tree['leaf_styles'].get(tree['leaves'][position])
        if style is None:
            continue
        color, bold = style
        tick_label.set_color(color)
        if bold:
            tick_label.set_fontweight('bold')

    # last in the legend, after whatever the annotation added, because it is a property of
    # the tree rather than of what is being shown on it
    if show_cutoff:
        cutoff = clustering['cutoff_distance']
        ax.axvline(cutoff, color='tab:orange', linestyle='--', linewidth=1.2, zorder=4)
        tree['handles'].append(Line2D([], [], color='tab:orange', linestyle='--',
                                      linewidth=1.2))
        tree['labels'].append(
            f'cutoff {cutoff:.3f} — tighter than {clustering["percentile"]:g}% of merges '
            f'from {clustering["resample_count"]} randomized copies')

    if tree['handles'] and legend_loc is not None:
        if legend_loc == 'below':
            # the leaf labels occupy the right of the axes, so the legend goes underneath
            ax.legend(tree['handles'], tree['labels'], loc='upper left',
                      bbox_to_anchor=(0.0, -0.11), fontsize='small', borderaxespad=0.0)
        elif legend_loc == 'inside':
            # Upper left, and fixed rather than 'best'. The leaf labels and the bars beside
            # them own the right of the axes whatever the tree looks like, and 'best' does
            # not know that -- it counts only the branches, so it happily picks a corner
            # the labels are already using.
            ax.legend(tree['handles'], tree['labels'], loc='upper left', fontsize='small',
                      framealpha=0.9)
        else:
            raise ValueError(f"legend_loc must be 'below', 'inside' or None, "
                             f"not {legend_loc!r}")
    return ax


def _mark_highlight_groups(tree, highlight):
    """Lay the highlighted combinations over the tree; returns {label: enclosing subtree}."""
    ax, clustering = tree['ax'], tree['clustering']
    ref_names, Z = clustering['ref_names'], clustering['Z']
    cutoff, root_height = clustering['cutoff_distance'], tree['root_height']
    cap_width = 0.02 * root_height

    groups, leaf_group_colors = {}, {}
    for k, (label, indices) in enumerate(_normalize_highlight(highlight, ref_names).items()):
        color = HIGHLIGHT_COLORS[k % len(HIGHLIGHT_COLORS)]
        marker = HIGHLIGHT_MARKERS[k % len(HIGHLIGHT_MARKERS)]
        linestyle = HIGHLIGHT_LINESTYLES[k % len(HIGHLIGHT_LINESTYLES)]
        group = smallest_enclosing_subtree(Z, indices)
        groups[label] = group

        positions = [tree['leaf_position'][leaf] for leaf in group['leaf_indices']]
        y_low, y_high = 10 * min(positions), 10 * max(positions) + 10

        # Shade the enclosing subtree, except when it is the whole tree: a full-height band
        # says nothing and only dims the figure. That case is reported by the bracket
        # sitting at the root, and by the legend.
        if group['size'] < len(ref_names):
            ax.axhspan(y_low, y_high, color=color, alpha=0.10, zorder=0)

        # Bracket at the merge height, spanning the subtree. Two combinations can share an
        # enclosing subtree -- on this data the best M=2 and M=3 subsets both reach the
        # root -- so the linestyle, not the color alone, is what tells the brackets apart.
        if group['height'] > 0:
            ax.vlines(group['height'], y_low + 1.5, y_high - 1.5, color=color, linewidth=2.0,
                      linestyles=linestyle, zorder=5)
            for y in (y_low + 1.5, y_high - 1.5):
                ax.plot([group['height'], group['height'] - cap_width], [y, y], color=color,
                        linewidth=2.0, zorder=5)

        # Markers sit inside the axes: with orientation='left' scipy puts the leaf labels
        # outside on the right, so anything past the axes edge would land on the text.
        for leaf in indices:
            leaf_group_colors.setdefault(leaf, []).append(color)
            ax.plot(0.985 - 0.028 * k, 10 * tree['leaf_position'][leaf] + 5, marker=marker,
                    color=color, markersize=6, transform=tree['marker_transform'],
                    clip_on=False, zorder=6)

        extent = ('a single leaf' if group['height'] == 0 else
                  f"subtree height {group['height']:.3f} = {group['height'] / root_height:.0%} "
                  f"of root, {group['size']} of {len(ref_names)} refs")
        # a lone reference is inside every cluster trivially, so only say this of a subtree
        within = ('' if group['height'] == 0 or group['height'] > cutoff
                  else ', within the cutoff')
        tree['handles'].append(Line2D([], [], color=color, marker=marker, linestyle=linestyle,
                                      linewidth=2.0, markersize=6))
        tree['labels'].append(f'{label}: {len(indices)} ref{"" if len(indices) == 1 else "s"}, '
                              f'{extent}{within}')

    # a leaf in two groups gets neither color, because it is not either one of them
    tree['leaf_styles'].update(
        {leaf: (colors[0] if len(colors) == 1 else 'black', True)
         for leaf, colors in leaf_group_colors.items()}
    )
    return groups


def _mark_leaf_weights(tree, leaf_weights, leaf_weight_label):
    """Shade every weighted leaf and give it a bar in the gutter beside the tree."""
    ax = tree['ax']
    weighted = _normalize_leaf_weights(leaf_weights, tree['clustering']['ref_names'])
    if not weighted:
        return

    colormap = plt.get_cmap(LEAF_WEIGHT_CMAP)

    def weight_color(weight):
        return colormap(LEAF_WEIGHT_FLOOR + (1.0 - LEAF_WEIGHT_FLOOR) * weight)

    # The labels are pushed out by the width of a full-weight bar, and the bars grow from
    # the right edge of the axes into the space that makes. They share that edge as a
    # baseline, so the column of them reads as a bar chart standing beside the leaves.
    ax.tick_params(axis='y', pad=72 * (LEAF_WEIGHT_BAR_INCHES + LEAF_WEIGHT_GUTTER_PAD))

    # The bars are drawn in axes coordinates, so their length is converted from inches
    # using the width of the axes. tight_layout may narrow the axes afterwards to fit the
    # labels their new padding pushed out, which shortens the bars with it -- they stay
    # inside the gutter, which is what matters, because the gutter is set in points and
    # does not move with the axes.
    axes_width_inches = max(ax.get_position().width * ax.figure.get_figwidth(), 1e-6)
    bar_axes_fraction = LEAF_WEIGHT_BAR_INCHES / axes_width_inches

    for leaf, weight in weighted.items():
        color = weight_color(weight)
        tree['leaf_styles'][leaf] = (color, False)
        y = 10 * tree['leaf_position'][leaf] + 5
        ax.plot([1.0, 1.0 + bar_axes_fraction * weight], [y, y], color=color, linewidth=4.0,
                solid_capstyle='butt', transform=tree['marker_transform'], clip_on=False,
                zorder=6)

    heaviest = max(weighted.values())
    tree['handles'].append(Line2D([], [], color=weight_color(heaviest), linewidth=4.0))
    tree['labels'].append(leaf_weight_label or
                          f'bar and label colour: weight per reference '
                          f'(longest = {heaviest:.0%})')


@names_its_figures
def plot_reference_dendrogram(clustering, ax=None, color_threshold=None, show_cutoff=True,
                              title=None, legend_loc='below'):
    """Draw the reference tree, with nothing laid over it.

    Parameters
    ----------
    clustering : dict
        The return value of `cluster_reference_spectra`.
    ax : matplotlib.axes.Axes, optional
        Where to draw. A figure sized to the number of references is made if omitted.
    color_threshold : float, optional
        Passed to `scipy`'s dendrogram. The default draws the whole tree in one neutral
        gray so that color belongs to whatever is laid over it; pass
        `clustering['cutoff_distance']` to color the significant clusters instead.
    show_cutoff : bool
        Draw the cutoff distance as a dashed line, and say in the legend what it means.
    title : str, optional
        Replaces the default, which names the metric and the cluster count.
    legend_loc : {'below', 'inside', None}
        Where to put the legend. 'below' hangs it under the axes, which is right for a
        figure of its own but lands on whatever is drawn beneath when this panel is one
        cell of a grid; 'inside' puts it in the upper left, clear of the leaf labels and
        any bars drawn beside them, which own the right of the axes; None suppresses it.

    Returns
    -------
    matplotlib.axes.Axes -- the axis drawn on.
    """
    tree = _draw_reference_tree(clustering, ax, color_threshold, title)
    return _finish_reference_tree(tree, show_cutoff, legend_loc)


@names_its_figures
def plot_highlighted_reference_dendrogram(
    clustering, highlight, ax=None, color_threshold=None, show_cutoff=True, title=None,
    legend_loc='below',
):
    """Draw the reference tree, showing where whole reference combinations sit in it.

    Each highlighted group gets the smallest subtree that contains all of its references,
    shaded and bracketed at the height that subtree merges — which is the question the
    figure exists to answer. A group whose bracket sits low is a combination drawn from
    one cluster of near-interchangeable references; a group whose bracket sits at the root
    is a combination spanning the whole reference set.

    Use `plot_weighted_reference_dendrogram` instead when what matters is how much each
    reference features rather than which combinations it belongs to: a group containing
    every reference brackets the whole tree and says nothing.

    Parameters
    ----------
    highlight : list or dict
        One combination, or `{label: combination}` for several at once. A combination may
        be given as reference names or as column indices into A, so a row of
        `results['ref_indices']` can be passed through unchanged.

    The remaining arguments are `plot_reference_dendrogram`'s and mean the same there.

    Returns
    -------
    (ax, groups) : the axis, and {label: smallest_enclosing_subtree(...)} so a caller can
    report the same numbers the figure draws without recomputing them.
    """
    tree = _draw_reference_tree(clustering, ax, color_threshold, title)
    groups = _mark_highlight_groups(tree, highlight)
    return _finish_reference_tree(tree, show_cutoff, legend_loc), groups


@names_its_figures
def plot_weighted_reference_dendrogram(
    clustering, leaf_weights, leaf_weight_label=None, ax=None, color_threshold=None,
    show_cutoff=True, title=None, legend_loc='below',
):
    """Draw the reference tree with every reference shaded by a weight.

    For marking a whole pool by degree, where `plot_highlighted_reference_dendrogram`
    marks a handful of references by membership. Each weighted leaf gets a bar in a gutter
    opened between the tree and its label, whose length is its fraction, and the label is
    colored on the same scale.

    Parameters
    ----------
    leaf_weights : dict
        `{reference: fraction}`, keyed by name or by column index into A. A reference left
        out is drawn plain, which is how "not used" stays distinguishable from "used
        rarely".
    leaf_weight_label : str, optional
        What the weights mean, for the legend. Defaults to a bare description of the scale.

    The remaining arguments are `plot_reference_dendrogram`'s and mean the same there.

    Returns
    -------
    matplotlib.axes.Axes -- the axis drawn on.
    """
    tree = _draw_reference_tree(clustering, ax, color_threshold, title)
    _mark_leaf_weights(tree, leaf_weights, leaf_weight_label)
    return _finish_reference_tree(tree, show_cutoff, legend_loc)

In [ ]:
# ---------------------------------------------------------------------------
# Tests for the reference clustering and its dendrogram
#
# These take A and ref_names directly rather than spectra, so they need no FakeSpectrum:
# the fixture is a design matrix with known block structure, where which references
# belong together is decided rather than discovered.
#
# The runner is an explicit list, as in the block length tests above, rather than the
# globals() scan used for the interpolation tests -- a second scanning runner would
# re-run those suites and misreport the count.
# ---------------------------------------------------------------------------

def _block_design_matrix(seed=0, n_energies=120, noise=0.01):
    """Six references: three noisy copies of one shape, three of a very different one."""
    rng = np.random.default_rng(seed)
    energies = np.linspace(0, 1, n_energies)
    shape_a = np.exp(-((energies - 0.35) ** 2) / 0.004)
    shape_b = np.exp(-((energies - 0.65) ** 2) / 0.004) + 0.5 * energies
    columns = [shape_a, shape_a, shape_a, shape_b, shape_b, shape_b]
    A = np.column_stack([c + noise * rng.standard_normal(n_energies) for c in columns])
    return A, ['a1', 'a2', 'a3', 'b1', 'b2', 'b3']


def test_references_are_the_observations():
    # The clustering is of references, not energies: 6 references over 120 energies must
    # give a 5-row linkage. Forgetting the transpose would give 119 rows.
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=50, verbose=False)
    assert clustering['Z'].shape == (len(ref_names) - 1, 4)
    assert clustering['distances'].shape == (len(ref_names) * (len(ref_names) - 1) // 2,)
    assert clustering['labels'].shape == (len(ref_names),)


def test_copies_of_one_shape_cluster_together():
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=200, verbose=False)
    labels = clustering['labels']
    assert labels[0] == labels[1] == labels[2], 'the copies of shape a should share a cluster'
    assert labels[3] == labels[4] == labels[5], 'the copies of shape b should share a cluster'
    assert labels[0] != labels[3], 'the two shapes should not share a cluster'
    assert clustering['cophenetic_correlation'] > 0.9

    group = smallest_enclosing_subtree(clustering['Z'], [0, 1, 2])
    assert group['leaf_indices'] == [0, 1, 2], 'no other reference belongs in that subtree'
    assert group['height'] < clustering['cutoff_distance']


def test_randomized_copies_merge_less_tightly_than_the_real_data():
    # The trap the older notebooks' permute_row_elements falls into: if the shuffle
    # silently no-ops, the randomized copies are just the real data again, the cutoff is
    # measured against the very heights it is meant to judge, and it certifies whatever it
    # is given. On data with real blocks the true merges are far tighter than chance.
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=200, verbose=False)
    observed = clustering['Z'][:, 2]
    by_chance = clustering['chance_merge_heights']
    assert by_chance.mean() > observed.mean(), \
        'shuffling must destroy the block structure, not preserve it'
    assert clustering['cutoff_distance'] > observed[:len(observed) - 1].max(), \
        'every within-shape merge should be more than chance'


def test_permutation_preserves_each_energy_row():
    # Pins what is shuffled: values move within an energy row, never between rows.
    A, _ = _block_design_matrix()
    permuted = permute_within_rows(A, np.random.default_rng(0))
    np.testing.assert_array_equal(np.sort(permuted, axis=1), np.sort(A, axis=1))
    assert not np.array_equal(permuted, A), 'the shuffle did nothing at all'


def test_smallest_enclosing_subtree_on_a_hand_built_tree():
    # 4 leaves: (0,1) join at 0.1, (2,3) join at 0.2, the two pairs join at 0.5.
    Z = np.array([[0.0, 1.0, 0.1, 2.0],
                  [2.0, 3.0, 0.2, 2.0],
                  [4.0, 5.0, 0.5, 4.0]])

    group = smallest_enclosing_subtree(Z, [0, 1])
    assert group['leaf_indices'] == [0, 1] and np.isclose(group['height'], 0.1)
    assert not group['is_root']

    # a group spanning both pairs can only be enclosed by the root
    group = smallest_enclosing_subtree(Z, [0, 2])
    assert group['leaf_indices'] == [0, 1, 2, 3] and np.isclose(group['height'], 0.5)
    assert group['is_root']

    # one reference encloses itself, at height 0 -- there is no subtree to shade
    group = smallest_enclosing_subtree(Z, [3])
    assert group['leaf_indices'] == [3] and group['height'] == 0.0 and group['size'] == 1


def test_highlight_accepts_names_indices_and_nan_padding():
    ref_names = ['a1', 'a2', 'a3', 'b1']
    assert (_normalize_highlight({'best M=2': ['a1', 'b1']}, ref_names)
            == _normalize_highlight({'best M=2': [0, 3]}, ref_names)
            == {'best M=2': [0, 3]})

    # a bare list becomes one group; a bare string is one name, not four characters
    assert _normalize_highlight(['a2'], ref_names) == {'highlighted': [1]}
    assert _normalize_highlight('a2', ref_names) == {'highlighted': [1]}

    # the literal shape of a results['ref_indices'] row: float, NaN-padded
    assert _normalize_highlight({'M=2': np.array([0.0, 3.0, np.nan])}, ref_names) == {'M=2': [0, 3]}

    try:
        _normalize_highlight({'oops': ['not_a_reference']}, ref_names)
    except ValueError:
        pass
    else:
        raise AssertionError('an unknown reference name should raise')


def test_correlation_ignores_an_offset_that_cosine_sees():
    # The whole difference between the two metrics: correlation centers each reference
    # first, so adding a constant leaves the tree alone; cosine measures the angle from
    # the origin, so the same constant moves it. Both are blind to a positive rescaling.
    A, ref_names = _block_design_matrix()

    def linkage_for(matrix, metric):
        return cluster_reference_spectra(matrix, ref_names, np.random.default_rng(0),
                                         metric=metric, resample_count=10, verbose=False)['Z']

    offset = A.copy()
    offset[:, 0] += 5.0
    np.testing.assert_allclose(linkage_for(A, 'correlation'), linkage_for(offset, 'correlation'))
    assert not np.allclose(linkage_for(A, 'cosine'), linkage_for(offset, 'cosine'))

    scaled = A.copy()
    scaled[:, 0] *= 3.0
    for metric in ('correlation', 'cosine'):
        np.testing.assert_allclose(linkage_for(A, metric), linkage_for(scaled, metric), atol=1e-12)


def test_cutoff_depends_on_the_generator_but_the_tree_does_not():
    A, ref_names = _block_design_matrix()
    first = cluster_reference_spectra(A, ref_names, np.random.default_rng(7),
                                      resample_count=200, verbose=False)
    again = cluster_reference_spectra(A, ref_names, np.random.default_rng(7),
                                      resample_count=200, verbose=False)
    other = cluster_reference_spectra(A, ref_names, np.random.default_rng(8),
                                      resample_count=200, verbose=False)

    assert first['cutoff_distance'] == again['cutoff_distance'], 'one seed, one cutoff'
    assert first['first_permutation_digest'] == again['first_permutation_digest']
    assert other['cutoff_distance'] != first['cutoff_distance'], 'a different seed should move it'
    np.testing.assert_array_equal(first['Z'], other['Z'])  # the tree itself is deterministic
    assert len(first['chance_merge_heights']) == 200 * (len(ref_names) - 1)


def test_degenerate_and_non_finite_columns_are_rejected():
    A, ref_names = _block_design_matrix()

    constant = A.copy()
    constant[:, 2] = 1.0
    try:
        cluster_reference_spectra(constant, ref_names, np.random.default_rng(0),
                                  metric='correlation', resample_count=5, verbose=False)
    except ValueError as error:
        assert 'a3' in str(error), 'the offending reference should be named'
    else:
        raise AssertionError('a constant column has no correlation distance')

    with_nan = A.copy()
    with_nan[3, 2] = np.nan
    try:
        cluster_reference_spectra(with_nan, ref_names, np.random.default_rng(0),
                                  resample_count=5, verbose=False)
    except ValueError as error:
        assert 'non-finite' in str(error)
    else:
        raise AssertionError('a NaN in A should raise rather than silently poison pdist')


def test_highlight_geometry_matches_the_drawn_dendrogram():
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=100, verbose=False)
    fig, ax = plt.subplots()
    _, groups = plot_highlighted_reference_dendrogram(
        clustering, highlight={'one shape': ['a1', 'a2', 'a3'], 'single': ['b2']}, ax=ax)

    # the shaded band must cover exactly the enclosing subtree's leaves, in drawn order
    drawn = hc.dendrogram(clustering['Z'], no_plot=True)['leaves']
    positions = sorted(drawn.index(leaf) for leaf in groups['one shape']['leaf_indices'])
    assert positions == list(range(positions[0], positions[-1] + 1)), \
        'a subtree draws as one contiguous run of leaves'
    spans = [patch for patch in ax.patches if hasattr(patch, 'get_xy')]
    assert spans, 'the enclosing subtree should be shaded'

    # a single-reference group has no subtree to shade or bracket
    assert groups['single']['height'] == 0.0 and groups['single']['size'] == 1

    # the x axis still runs root -> leaves and still contains the cutoff
    x_left, x_right = ax.get_xlim()
    assert x_left > x_right, 'orientation=left inverts the distance axis'
    assert x_left >= clustering['cutoff_distance'], 'the cutoff must be inside the axes'

    bold = {label.get_text() for label in ax.get_ymajorticklabels()
            if label.get_fontweight() == 'bold'}
    assert bold == {'a1', 'a2', 'a3', 'b2'}, f'highlighted leaves should be bold, got {bold}'
    plt.close(fig)


def test_leaf_weights_shade_every_leaf_by_its_weight():
    """Weights color the whole pool by degree, where a highlight marks a few by membership."""
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=50, verbose=False)
    weights = {name: (i + 1) / len(ref_names) for i, name in enumerate(ref_names)}

    fig, ax = plt.subplots()
    try:
        plot_weighted_reference_dendrogram(clustering, leaf_weights=weights, ax=ax,
                                           legend_loc='inside')
        label_colors = {label.get_text(): label.get_color()
                        for label in ax.get_ymajorticklabels()}
        # heavier weights sit further along the ramp, so they are darker
        lightest = label_colors[min(weights, key=weights.get)]
        heaviest = label_colors[max(weights, key=weights.get)]
        assert sum(heaviest[:3]) < sum(lightest[:3]), 'the heaviest leaf should be the darkest'
        assert len(set(label_colors.values())) == len(ref_names), 'each weight its own shade'
        assert 'longest' in ax.get_legend().get_texts()[0].get_text()

        # the bars sit in a gutter outside the axes, not on top of the branches, and the
        # leaf labels are pushed out past them
        bars = [line for line in ax.lines
                if line.get_transform() is not ax.transData and len(line.get_xdata()) == 2]
        assert bars, 'a weighted leaf should be drawn a bar'
        assert all(min(line.get_xdata()) >= 1.0 for line in bars), \
            'bars should start at the right edge of the axes and grow outwards'
        longest = max(max(line.get_xdata()) for line in bars)
        assert longest > 1.0, 'the heaviest bar should reach into the gutter'
        pad = ax.yaxis.get_major_ticks()[0].get_pad()
        bar_inches = (longest - 1.0) * ax.get_position().width * fig.get_figwidth()
        assert pad / 72 > bar_inches, 'the labels should start beyond the longest bar'
    finally:
        plt.close(fig)

    # names or column indices, the same as highlight takes
    assert _normalize_leaf_weights({ref_names[2]: 0.5}, ref_names) == {2: 0.5}
    assert _normalize_leaf_weights({2: 0.5}, ref_names) == {2: 0.5}
    assert _normalize_leaf_weights(None, ref_names) == {}

    for bad in ({'nope.e': 0.5}, {0: 1.5}, {0: -0.1}):
        try:
            _normalize_leaf_weights(bad, ref_names)
        except ValueError:
            continue
        raise AssertionError(f'{bad!r} should have been refused')


def test_legend_loc_places_or_suppresses_the_legend():
    # The default hangs the legend under the axes, which is right for a figure of its own
    # and wrong inside a grid, where it lands on whatever the next row draws. The summary
    # figures ask for 'inside' instead, so both placements have to actually differ.
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=50, verbose=False)

    def legend_and_axes_extents(**kwargs):
        fig, ax = plt.subplots()
        try:
            plot_highlighted_reference_dendrogram(clustering, highlight=['a1'], ax=ax,
                                                  **kwargs)
            fig.canvas.draw()  # window extents are only meaningful once laid out
            legend = ax.get_legend()
            if legend is None:
                return None
            return legend.get_window_extent(), ax.get_window_extent()
        finally:
            # in a finally, so the bad-value case below closes its figure too rather than
            # leaving a stray blank one to be rendered under %matplotlib inline
            plt.close(fig)

    legend_box, axes_box = legend_and_axes_extents()
    assert legend_box.y1 <= axes_box.y0 + 1, 'the default legend should sit below the axes'

    legend_box, axes_box = legend_and_axes_extents(legend_loc='inside')
    assert legend_box.y0 >= axes_box.y0 - 1, "legend_loc='inside' should sit within the axes"

    assert legend_and_axes_extents(legend_loc=None) is None, 'legend_loc=None should draw none'

    try:
        legend_and_axes_extents(legend_loc='somewhere')
    except ValueError:
        pass
    else:
        raise AssertionError('an unknown legend_loc should raise')


def test_phase_randomize_keeps_each_reference_and_breaks_the_relationships():
    # What the surrogate has to do to be a null at all: every reference keeps its own
    # character -- mean, variance, smoothness, all of which live in its power spectrum --
    # while what it shares with the other references goes away.
    A, ref_names = _block_design_matrix()
    rng = np.random.default_rng(0)
    surrogate = phase_randomize_columns(A, rng)

    assert surrogate.shape == A.shape
    np.testing.assert_allclose(np.abs(np.fft.rfft(surrogate, axis=0)),
                               np.abs(np.fft.rfft(A, axis=0)), atol=1e-10)
    np.testing.assert_allclose(surrogate.mean(axis=0), A.mean(axis=0), atol=1e-10)
    np.testing.assert_allclose(surrogate.std(axis=0), A.std(axis=0), atol=1e-10)
    assert np.isrealobj(surrogate), 'randomizing DC or Nyquist would make the inverse complex'

    def mean_abs_cross_correlation(matrix):
        correlations = np.corrcoef(matrix.T)
        return np.abs(correlations[~np.eye(correlations.shape[0], dtype=bool)]).mean()

    assert mean_abs_cross_correlation(surrogate) < mean_abs_cross_correlation(A), \
        'the surrogate references should resemble each other less than the real ones'

    # a different seed gives a different surrogate; the same seed repeats it exactly
    np.testing.assert_array_equal(
        phase_randomize_columns(A, np.random.default_rng(1)),
        phase_randomize_columns(A, np.random.default_rng(1)))
    assert not np.allclose(surrogate, phase_randomize_columns(A, np.random.default_rng(1)))


def test_surrogate_choice_reaches_the_cutoff():
    # The parameter has to actually be used: the two nulls must give different cutoffs on
    # the same data and seed, and the default must stay permute_within_rows.
    A, ref_names = _block_design_matrix()

    def cutoff_with(**kwargs):
        return cluster_reference_spectra(A, ref_names, np.random.default_rng(3),
                                         resample_count=100, verbose=False, **kwargs)

    default = cutoff_with()
    shuffled = cutoff_with(surrogate=permute_within_rows)
    phase = cutoff_with(surrogate=phase_randomize_columns)

    assert default['cutoff_distance'] == shuffled['cutoff_distance']
    assert default['surrogate'] == 'permute_within_rows'
    assert phase['surrogate'] == 'phase_randomize_columns'
    assert phase['cutoff_distance'] != shuffled['cutoff_distance']


def test_flat_labels_match_the_dendrogram_colors():
    # fcluster at the cutoff and scipy's own color_threshold must agree, which is what
    # lets the plot color clusters by passing the cutoff straight through.
    A, ref_names = _block_design_matrix()
    clustering = cluster_reference_spectra(A, ref_names, np.random.default_rng(0),
                                           resample_count=100, verbose=False)
    drawn = hc.dendrogram(clustering['Z'], no_plot=True,
                          color_threshold=clustering['cutoff_distance'])
    by_color = {}
    for leaf, color in zip(drawn['leaves'], drawn['leaves_color_list']):
        by_color.setdefault(color, set()).add(leaf)
    by_label = {}
    for leaf, label in enumerate(clustering['labels']):
        by_label.setdefault(label, set()).add(leaf)
    assert sorted(map(sorted, by_color.values())) == sorted(map(sorted, by_label.values()))


_clustering_test_fns = [
    test_references_are_the_observations,
    test_copies_of_one_shape_cluster_together,
    test_randomized_copies_merge_less_tightly_than_the_real_data,
    test_permutation_preserves_each_energy_row,
    test_smallest_enclosing_subtree_on_a_hand_built_tree,
    test_highlight_accepts_names_indices_and_nan_padding,
    test_correlation_ignores_an_offset_that_cosine_sees,
    test_cutoff_depends_on_the_generator_but_the_tree_does_not,
    test_degenerate_and_non_finite_columns_are_rejected,
    test_highlight_geometry_matches_the_drawn_dendrogram,
    test_leaf_weights_shade_every_leaf_by_its_weight,
    test_legend_loc_places_or_suppresses_the_legend,
    test_flat_labels_match_the_dendrogram_colors,
    test_phase_randomize_keeps_each_reference_and_breaks_the_relationships,
    test_surrogate_choice_reaches_the_cutoff,
]

for _fn in _clustering_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_clustering_test_fns)} tests passed')

In [ ]:
# Cluster the pool once on the primary unknown's design matrix, and look at both metrics.
# Which distance to use is itself a choice; see the study below.
primary_clusterings = {
    metric: cluster_reference_spectra(primary_A, ref_names, np.random.default_rng(42),
                                      metric=metric, verbose=False)
    for metric in ('correlation', 'cosine')
}

fig, axs = plt.subplots(1, 2, figsize=(22, 9))
for ax, (metric, clustering) in zip(axs, primary_clusterings.items()):
    plot_reference_dendrogram(clustering, ax=ax, legend_loc='inside',
                              color_threshold=clustering['cutoff_distance'],
                              title=f'{metric} distance')
fig.tight_layout()
display(fig)
plt.close(fig)

<a id="pipeline-fit"></a>

## 4. Fitting, and what the residuals look like

`fit_nnls` solves the non-negative least squares problem — mixing fractions cannot be negative
— and the residuals go through an autocorrelation function and a histogram. This is where the
autocorrelation that motivates the whole block scheme becomes visible: neighbouring residuals
are not independent, so a bootstrap that resamples them one at a time would understate the
uncertainty.

In [ ]:
def fit_nnls(A, b):
    coef, _ = scipy.optimize.nnls(A, b)
    fitted = A @ coef
    residuals = fitted - b
    return coef, fitted, residuals


def calculate_acf(residuals):
    n = len(residuals)
    n_lags = min(40, n // 2)
    x = residuals - residuals.mean()
    var = np.dot(x, x) / n
    acf_values = np.array(
        [1.0] + [np.dot(x[:-lag], x[lag:]) / (n * var) for lag in range(1, n_lags + 1)]
    )
    lags = np.arange(n_lags + 1)
    return lags, acf_values


def plot_spectrum_fit(energies, b, fitted, residuals, ax):
    ax.plot(energies, b, label='unknown', color='black')
    ax.plot(energies, fitted, label='fit', color='red', linestyle='--')
    ax.scatter(energies, residuals, color='orange', label='residuals', marker='o', s=10.0)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Normalized Fluorescence')
    ax.legend()


def plot_residuals_histogram(residuals, ax, bins=15):
    mean = residuals.mean()
    std = residuals.std()
    ax.hist(residuals, bins=bins, color="orange", alpha=0.7, edgecolor="white")
    ax.axvline(mean, color="red", linestyle="--", label=f"mean={mean:.4f}")
    ax.axvline(mean + std, color="steelblue", linestyle=":", label=f"+1 std={mean + std:.4f}")
    ax.axvline(mean - std, color="steelblue", linestyle=":", label=f"-1 std={mean - std:.4f}")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend()


def plot_acf(lags, acf_values, n, ax):
    ci_95 = 1.96 / np.sqrt(n)
    ax.bar(lags, acf_values, color='orange', alpha=0.7)
    ax.axhline(ci_95, color='red', linestyle='--', label='95% CI (white noise)')
    ax.axhline(-ci_95, color='red', linestyle='--')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.legend()

In [ ]:
# One fit of the whole pool against the primary unknown, and what its residuals look like.
primary_coef, primary_fitted, primary_residuals = fit_nnls(primary_A, primary_b)
primary_lags, primary_acf = calculate_acf(primary_residuals)

fig, (ax_fit, ax_acf, ax_hist) = plt.subplots(1, 3, figsize=(20, 4.5))
plot_spectrum_fit(primary_energies, primary_b, primary_fitted, primary_residuals, ax_fit)
ax_fit.set_title(primary_spectrum.file_name)
plot_acf(primary_lags, primary_acf, len(primary_residuals), ax_acf)
ax_acf.set_title('Residual autocorrelation')
ax_acf.legend(fontsize=9)
plot_residuals_histogram(primary_residuals, ax_hist)
ax_hist.set_title('Residuals')
ax_hist.legend(fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

print(f'residual std {primary_residuals.std():.5f}, '
      f'lag-1 autocorrelation {primary_acf[1]:.3f} -- which is why the blocks exist')

<a id="pipeline-holdout"></a>

## 5. Drawing the holdout blocks

`select_holdout_blocks` pre-generates, for all iterations at once, the holdout masks — about a
third of the energies, in contiguous blocks — and the block starts used to resample residuals.
Generating them once and sharing them across every reference combination is what makes
prediction errors *paired*, and every comparison downstream depends on that.

It takes **two** block lengths. The holdout blocks and the resample blocks are doing opposite
jobs, and older versions of this code pinned them to one number; [a study below](#study-resample-length)
takes them apart.

In [ ]:
@names_its_figures
def select_holdout_blocks(n, rng, n_bootstrap=1000, block_length_min=6, block_length_max=10,
                          block_length=None, resample_block_length=None):
    """Draw the holdout masks and the resample block starts for every bootstrap iteration.

    The spectrum is tiled end to end with blocks of random length in
    [block_length_min, block_length_max], a third of them are held out, and every other
    iteration's mask is reversed so the truncated final block alternates between the
    high-energy and low-energy end rather than always landing at the high end. That
    alternation is the last of five versions this selector went through; the comparison that
    chose it is in the development notebook under *Development of `select_holdout_blocks`
    v1-v5*, and the short version is that the ranking of the earlier versions turned entirely
    on whether the block length divided the number of energies, which v5 is immune to.

    Two lengths, not one
    --------------------
    `block_length` sets the shortest *holdout* block. `resample_block_length` sets the length
    of the *residual* blocks the bootstrap resamples, and defaults to the same number, which
    is what every version of this selector did before. They are doing opposite jobs: holdout
    blocks want to be short, so the spectrum is scored finely, and resample blocks want to be
    long, so they carry the residual autocorrelation into the bootstrap. Pinning them
    together meant trading one against the other. See *Resample length vs. holdout length*.

    Leaving `resample_block_length` unset reproduces the earlier behaviour exactly, draw for
    draw, because the two quantities it changes -- how many blocks are needed and which
    starts are available -- both collapse to the old ones.

    Returns
    -------
    (holdout_masks, sampled_starts, resample_block_length, n_holdout_blocks)
        `holdout_masks` is (n_bootstrap, n) bool; `sampled_starts` is
        (n_bootstrap, n_blocks_needed) int, indices into the residual series. The third
        element is the length those starts are meant to be read with -- pass it to
        `do_moving_block_holdout_bootstrap`.
    """
    # The random range is slid rather than collapsed, so setting a block length keeps the
    # varying-length behaviour rather than turning the selector into a fixed grid.
    if block_length is not None:
        block_length_min, block_length_max = (
            block_length, block_length + (block_length_max - block_length_min),
        )
    if resample_block_length is None:
        resample_block_length = block_length_min

    n_blocks_needed = int(np.ceil(n / resample_block_length))
    resample_block_starts = np.arange(n - resample_block_length + 1)

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)
    total_holdout_blocks = 0

    for i in range(n_bootstrap):
        block_starts_i, block_lengths_i = [], []
        pos = 0
        while pos < n:
            bl = int(rng.integers(block_length_min, block_length_max + 1))
            bl = min(bl, n - pos)
            block_starts_i.append(pos)
            block_lengths_i.append(bl)
            pos += bl

        n_blocks_i = len(block_lengths_i)
        n_holdout_i = round(n_blocks_i / 3)
        total_holdout_blocks += n_holdout_i

        holdout_block_indices = rng.choice(n_blocks_i, size=n_holdout_i, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            s = block_starts_i[idx]
            holdout_mask[s:s + block_lengths_i[idx]] = True

        if i % 2 == 1:
            holdout_mask = holdout_mask[::-1]

        holdout_masks[i] = holdout_mask

        # A resample block may not overlap the holdout, or the bootstrap would be resampling
        # residuals it is about to score against. Long resample blocks and short holdout
        # blocks pull against each other here: a finely spread holdout leaves short clear
        # runs, and a long block needs a long one. When nothing fits, say so rather than
        # letting rng.choice raise on an empty sequence.
        valid_resample_starts = resample_block_starts[np.array(
            [not holdout_mask[s:s + resample_block_length].any() for s in resample_block_starts]
        )]
        if len(valid_resample_starts) == 0:
            raise ValueError(
                f'no run of {resample_block_length} energies survives the holdout on '
                f'iteration {i}; the resample block length is too long for holdout blocks '
                f'of [{block_length_min}, {block_length_max}]'
            )
        sampled_starts[i] = rng.choice(valid_resample_starts, size=n_blocks_needed, replace=True)

    n_holdout_blocks = round(total_holdout_blocks / n_bootstrap)
    mean_holdout_frac = (total_holdout_blocks * (block_length_min + block_length_max) / 2
                         / (n_bootstrap * n))
    print(f'n={n}, holdout blocks [{block_length_min}, {block_length_max}], '
          f'resample blocks {resample_block_length}, n_blocks_needed={n_blocks_needed}')
    print(f'avg n_holdout_blocks={n_holdout_blocks} (~{mean_holdout_frac:.0%} of data)')

    return holdout_masks, sampled_starts, resample_block_length, n_holdout_blocks


def available_start_fraction(holdout_masks, resample_block_length):
    """Fraction of resample start positions the holdout leaves usable, averaged over draws.

    The feasibility side of the two block lengths. It falls as either length grows -- longer
    resample blocks need longer clear runs, longer holdout blocks eat them -- and when it
    reaches zero the draw above cannot be made at all.
    """
    n = holdout_masks.shape[1]
    starts = np.arange(n - resample_block_length + 1)
    usable = [
        np.mean([not mask[s:s + resample_block_length].any() for s in starts])
        for mask in holdout_masks
    ]
    return float(np.mean(usable))

In [ ]:
# ---------------------------------------------------------------------------
# Tests for the holdout draw.
#
# One of these is a regression pin rather than a property. Separating the resample block
# length from the holdout block length touched the three places the old code used one number
# for both, and the whole argument for the change rests on it being a no-op when the new
# argument is left alone -- so that is pinned to a digest rather than described.
# ---------------------------------------------------------------------------

def _draw(**kwargs):
    with contextlib.redirect_stdout(io.StringIO()):
        return select_holdout_blocks(198, np.random.default_rng(42), n_bootstrap=200, **kwargs)


def test_leaving_the_resample_length_alone_reproduces_the_old_draws():
    """The draws v5 made before the two lengths were separated, to the bit.

    Recorded from the frozen development notebook's `select_holdout_blocks_v5` at this seed,
    size and block length. If this fails, the separation is no longer transparent and every
    number carried over from that notebook is in question.
    """
    holdout_masks, sampled_starts, resample_length, n_holdout_blocks = _draw(block_length=10)
    digest = hashlib.sha256(holdout_masks.tobytes() + sampled_starts.tobytes()).hexdigest()
    assert digest[:16] == '42d5c3adadf02486', f'draws changed: {digest[:16]}'
    assert (resample_length, n_holdout_blocks) == (10, 6)
    assert sampled_starts.shape == (200, 20)


def test_the_resample_length_is_what_was_asked_for():
    """It sets the number of blocks needed and comes back as the length to read them with."""
    for resample_block_length in (4, 8, 16):
        _, sampled_starts, returned, _ = _draw(block_length=10,
                                               resample_block_length=resample_block_length)
        assert returned == resample_block_length
        assert sampled_starts.shape[1] == int(np.ceil(198 / resample_block_length))


def test_no_resample_block_overlaps_the_holdout():
    """The invariant the whole scheme rests on: a block may not be resampled from what it
    is about to be scored against."""
    for holdout, resample in ((10, 10), (3, 12), (14, 4)):
        holdout_masks, sampled_starts, length, _ = _draw(block_length=holdout,
                                                         resample_block_length=resample)
        for mask, starts in zip(holdout_masks[:20], sampled_starts[:20]):
            for start in starts:
                assert not mask[start:start + length].any(), (holdout, resample, start)


def test_a_resample_block_longer_than_any_clear_run_is_refused():
    """Rather than letting rng.choice raise on an empty sequence."""
    try:
        # holdout blocks of [3, 7] leave no clear run of 20 on this grid; see the study below
        _draw(block_length=3, resample_block_length=20)
    except ValueError as error:
        assert 'resample block length' in str(error)
    else:
        raise AssertionError('a resample block nothing can accommodate should be refused')


def test_availability_falls_as_either_length_grows():
    """The feasibility side, and the reason the two lengths cannot be set independently."""
    holdout_masks, _, _, _ = _draw(block_length=10)
    at_4 = available_start_fraction(holdout_masks, 4)
    at_20 = available_start_fraction(holdout_masks, 20)
    assert at_4 > at_20, 'longer resample blocks need longer clear runs'

    short_holdout, _, _, _ = _draw(block_length=3)
    long_holdout, _, _, _ = _draw(block_length=14)
    assert (available_start_fraction(short_holdout, 12)
            < available_start_fraction(long_holdout, 12)), \
        'a finely spread holdout leaves shorter clear runs'


_holdout_test_fns = [
    test_leaving_the_resample_length_alone_reproduces_the_old_draws,
    test_the_resample_length_is_what_was_asked_for,
    test_no_resample_block_overlaps_the_holdout,
    test_a_resample_block_longer_than_any_clear_run_is_refused,
    test_availability_falls_as_either_length_grows,
]
for _fn in _holdout_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_holdout_test_fns)} tests passed')

<a id="pipeline-bootstrap"></a>

## 6. The bootstrap

For each iteration, `do_moving_block_holdout_bootstrap` adds resampled residual blocks to the
fitted spectrum to build a bootstrap spectrum, refits by NNLS on the training positions, and
scores root-mean-square error against the *real* spectrum at the held-out positions. The
spread of those prediction errors is the uncertainty estimate.

`do_ref_subsets_moving_block_holdout_bootstrap` then runs that for every combination of
references up to a given size — 2,324 of them for sizes 1, 2 and 3 over 24 references — on the
one shared set of draws.

In [ ]:
def flat_top_lag_window(t):
    """Politis & Romano flat-top lag window: 1 on |t|<=1/2, tapering to 0 at |t|=1."""
    abs_t = np.abs(t)
    return np.where(abs_t <= 0.5, 1.0, np.where(abs_t <= 1.0, 2.0 * (1.0 - abs_t), 0.0))


def politis_white_block_length(x, k_n=None):
    """Data-driven optimal moving-block-bootstrap block length (Politis & White 2004).

    The notebook's selectors all set block_length = round(n ** (1/3)), which is only
    the *rate* at which the optimal block length grows with n; the constant in front
    of it depends on how much dependence the series actually carries. This estimates
    that constant from the data.

    The estimator has three steps:

      1. Pick a bandwidth M. Find the smallest lag m past which the sample
         autocorrelation stays inside the +/- 2 sqrt(log10(n) / n) band for k_n
         consecutive lags -- i.e. the lag past which the series looks uncorrelated --
         and set M = 2m.
      2. Form flat-top-weighted estimates over lags |k| <= M of the long-run variance
         G_hat_0 = sum R(k) and of the "curvature" g_hat = sum |k| R(k).
      3. b_opt = (2 g_hat^2 / D_hat) ** (1/3) * n ** (1/3), with
         D_hat = (4/3) G_hat_0^2 for the moving block bootstrap.

    Returns
    -------
    dict with the estimate and the intermediate quantities, so the number can be
    audited rather than taken on faith.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    if k_n is None:
        k_n = max(5, int(np.ceil(np.sqrt(np.log10(n)))))

    # autocovariances R(0..n-1) and autocorrelations
    centered = x - x.mean()
    max_lag = min(n - 1, int(np.ceil(np.sqrt(n))) + k_n + 40)
    autocovariance = np.array(
        [np.dot(centered[:n - k], centered[k:]) / n for k in range(max_lag + 1)]
    )
    autocorrelation = autocovariance / autocovariance[0]

    # step 1: smallest m with k_n consecutive insignificant autocorrelations after it
    significance_bound = 2.0 * np.sqrt(np.log10(n) / n)
    m_hat = 0
    for m in range(1, max_lag - k_n + 1):
        window = np.abs(autocorrelation[m + 1:m + 1 + k_n])
        if len(window) == k_n and (window < significance_bound).all():
            m_hat = m
            break
    else:
        # no such run: fall back to the largest lag that is still significant
        significant = np.where(np.abs(autocorrelation[1:]) >= significance_bound)[0]
        m_hat = int(significant[-1]) + 1 if len(significant) else 1

    bandwidth_M = min(2 * m_hat, max_lag)

    # step 2: flat-top-weighted sums over -M..M (symmetric, so double the k>0 terms)
    lags = np.arange(1, bandwidth_M + 1)
    weights = flat_top_lag_window(lags / bandwidth_M) if bandwidth_M > 0 else np.array([])
    long_run_variance = autocovariance[0] + 2.0 * np.sum(weights * autocovariance[lags])
    curvature = 2.0 * np.sum(weights * lags * autocovariance[lags])

    # step 3
    d_hat = (4.0 / 3.0) * long_run_variance ** 2
    if curvature == 0 or d_hat == 0:
        b_opt = 1.0
    else:
        b_opt = (2.0 * curvature ** 2 / d_hat) ** (1.0 / 3.0) * n ** (1.0 / 3.0)

    # Politis & White cap the estimate; without it a near-unit-root series can ask for
    # a block longer than the sample can support.
    b_max = np.ceil(min(3.0 * np.sqrt(n), n / 3.0))
    b_opt_capped = float(np.clip(b_opt, 1.0, b_max))

    return {
        'n': n,
        'k_n': k_n,
        'm_hat': m_hat,
        'bandwidth_M': bandwidth_M,
        'significance_bound': significance_bound,
        'long_run_variance': long_run_variance,
        'variance': autocovariance[0],
        'curvature': curvature,
        'b_opt_raw': float(b_opt),
        'b_opt': b_opt_capped,
        'b_max': float(b_max),
        'rule_of_thumb': float(max(1, round(n ** (1 / 3)))),
    }


def choose_block_length(residual_matrix, percentile=10, verbose=True):
    """Pick one moving-block length for a whole set of reference combinations.

    The holdout draw is shared across every reference combination -- that is what makes
    their prediction errors comparable -- so one block length has to serve all of them.
    politis_white_block_length gives a per-combination answer, and those answers
    disagree wildly (7 to the cap of 43 on the arsenic data), so they have to be reduced
    to a single number.

    That reduction is a *low* quantile rather than the median, and the reason is
    structural. An underfit combination's residuals are not noise: they contain the part
    of the spectrum the model failed to explain, which is smooth and spectrum-shaped.
    Politis-White cannot tell that deterministic misfit from genuine dependence, and
    reads it as very long-range autocorrelation. On the arsenic data every M=1 subset
    pins the cap and rank correlation between fit RMSE and b_opt is +0.54.

    The key point is that this contamination is *one-sided*: unmodeled structure can only
    ever inflate the estimate, never deflate it. So the low order statistics are the
    trustworthy end of the distribution, and the median -- drawn mostly from combinations
    whose residuals are mostly signal -- is meaningless.

    Parameters
    ----------
    residual_matrix : (n_combinations, n) array of full-data residuals, one row per
                      reference combination. This is exactly the all_residuals array
                      do_ref_subsets_moving_block_holdout_bootstrap already builds.
    percentile      : which low quantile to take. The p1/p5/p10/p25 sweep is reported
                      alongside so the choice can be audited rather than trusted.

    Returns
    -------
    dict with 'block_length' (int, the value to use), 'b_opt' (per-combination
    estimates), 'percentile_sweep', 'n_at_cap' and 'rule_of_thumb'.
    """
    residual_matrix = np.atleast_2d(np.asarray(residual_matrix, dtype=float))
    n = residual_matrix.shape[1]

    estimates = np.array([
        politis_white_block_length(row)['b_opt'] for row in residual_matrix
    ])
    b_max = np.ceil(min(3.0 * np.sqrt(n), n / 3.0))

    sweep_percentiles = [1, 5, 10, 25, 50]
    sweep = dict(zip(sweep_percentiles, np.percentile(estimates, sweep_percentiles)))

    block_length = max(1, int(np.round(np.percentile(estimates, percentile))))
    rule_of_thumb = max(1, int(np.round(n ** (1 / 3))))

    if verbose:
        print(f'block length tuned from {len(estimates)} combinations '
              f'(Politis-White, p{percentile}):')
        print('  percentile sweep: ' + '  '.join(f'p{p}={v:.2f}' for p, v in sweep.items()))
        print(f'  {int((estimates >= b_max).sum())} of {len(estimates)} combinations pinned '
              f'the cap ({b_max:.0f}) -- these are underfit, not strongly dependent')
        print(f'  block_length={block_length} (rule of thumb round(n**(1/3))={rule_of_thumb})')

    return {
        'block_length': block_length,
        'b_opt': estimates,
        'percentile': percentile,
        'percentile_sweep': sweep,
        'n_at_cap': int((estimates >= b_max).sum()),
        'b_max': float(b_max),
        'rule_of_thumb': rule_of_thumb,
    }

In [ ]:
# ---------------------------------------------------------------------------
# Tests for politis_white_block_length / choose_block_length
#
# The estimator has no reference implementation here to check against, so the
# tests pin the behavior that makes it trustworthy: it should ask for short
# blocks when there is nothing to preserve, longer blocks as dependence grows,
# never more than the cap, and -- the property the whole aggregate rests on --
# it should not let a structure-dominated series drag the chosen length up.
# ---------------------------------------------------------------------------

def _ar1_series(phi, n=500, seed=0):
    rng = np.random.default_rng(seed)
    innovations = rng.standard_normal(n)
    series = np.zeros(n)
    for i in range(1, n):
        series[i] = phi * series[i - 1] + innovations[i]
    return series


def test_white_noise_wants_short_blocks():
    """Independent data has no dependence to preserve, so blocks should be ~1 point."""
    white_noise = np.random.default_rng(0).standard_normal(198)
    result = politis_white_block_length(white_noise)
    assert result['b_opt'] < 2.0, f"expected b_opt < 2 for white noise, got {result['b_opt']:.3f}"


def test_ar1_block_length_increases_with_dependence():
    """More persistent series need longer blocks to carry their autocorrelation."""
    block_lengths = [politis_white_block_length(_ar1_series(phi))['b_opt']
                     for phi in (0.2, 0.5, 0.8)]
    assert block_lengths == sorted(block_lengths), \
        f'expected block length to increase with phi, got {np.round(block_lengths, 2)}'
    assert block_lengths[0] < block_lengths[-1], \
        f'expected a strict increase from phi=0.2 to phi=0.8, got {np.round(block_lengths, 2)}'


def test_block_length_is_capped():
    """A near-unit-root series would otherwise ask for a block the sample cannot support."""
    result = politis_white_block_length(_ar1_series(0.99, n=198))
    assert result['b_opt'] <= result['b_max'], \
        f"b_opt {result['b_opt']:.2f} exceeded the cap {result['b_max']:.2f}"
    assert result['b_opt'] <= np.ceil(198 / 3)


def test_aggregate_ignores_inflated_estimates():
    """The low quantile must survive rows whose residuals are misfit rather than noise.

    This is the assumption choose_block_length is built on, so it is tested rather
    than asserted in a comment: mixing in structure-dominated rows -- a smooth
    half-cosine, which is what an underfit spectrum's residuals look like -- must
    raise the *median* while leaving the low quantile on the noise-like rows.
    """
    n = 198
    rng = np.random.default_rng(0)
    noise_like = np.array([_ar1_series(0.5, n=n, seed=s) for s in range(30)])
    structure_like = np.array([
        np.cos(np.linspace(0, np.pi, n)) * (1.0 + 0.05 * rng.standard_normal(n))
        for _ in range(30)
    ])

    clean = choose_block_length(noise_like, percentile=10, verbose=False)
    contaminated = choose_block_length(
        np.vstack([noise_like, structure_like]), percentile=10, verbose=False,
    )

    assert contaminated['block_length'] == clean['block_length'], (
        f"low quantile moved from {clean['block_length']} to "
        f"{contaminated['block_length']} when structure-dominated rows were added"
    )
    assert contaminated['percentile_sweep'][50] > clean['percentile_sweep'][50], \
        'expected the median to be inflated by the structure-dominated rows'
    assert contaminated['n_at_cap'] > 0, 'expected the structure-dominated rows to pin the cap'


_block_length_test_fns = [
    test_white_noise_wants_short_blocks,
    test_ar1_block_length_increases_with_dependence,
    test_block_length_is_capped,
    test_aggregate_ignores_inflated_estimates,
]
for _fn in _block_length_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_block_length_test_fns)} tests passed')

In [ ]:
def do_moving_block_holdout_bootstrap(A, b, fitted, residuals, holdout_masks, sampled_starts, block_length):
    """Compute holdout prediction error for each pre-drawn bootstrap iteration.

    Parameters
    ----------
    A               : (n, n_refs) design matrix for one reference subset
    b               : (n,) observed spectrum values
    fitted          : (n,) full-data fitted values
    residuals       : (n,) full-data residuals (fitted - b)
    holdout_masks   : (n_bootstrap, n) bool array from select_holdout_blocks
    sampled_starts  : (n_bootstrap, n_blocks_needed) int array from select_holdout_blocks
    block_length    : int from select_holdout_blocks

    Returns
    -------
    bootstrap_coefs : (n_bootstrap, n_refs) float array
    bootstrap_pes   : (n_bootstrap,) float array — per-iteration holdout RMSE
    """
    n = len(b)
    n_bootstrap = len(holdout_masks)
    bootstrap_coefs = np.zeros((n_bootstrap, A.shape[1]))
    # Instead of computing each iteration's holdout RMSE inline with
    # np.sqrt(np.mean(np.square(...))) (a np.square/np.mean/np.sqrt trio per iteration,
    # ~2.32M times over the notebook's workload), accumulate each iteration's holdout
    # sum-of-squares and held-out point count as scalars and take the RMSE for all
    # iterations at once after the loop. This moves the mean/sqrt reduction out of the
    # hot loop (~10-15% faster here) and, because it stores scalars rather than a
    # fixed-width residual array, stays correct when the number of held-out points
    # varies between iterations (e.g. select_holdout_blocks_v4/v5).
    holdout_sum_of_squares = np.zeros(n_bootstrap)
    holdout_point_counts = np.zeros(n_bootstrap)

    # Vectorized replacement for the per-iteration block gather this loop used to do. The
    # original code, inside `for i in range(n_bootstrap):`, called ~2.32M times total across
    # 2324 reference combinations x 1000 bootstrap iterations, was:
    #
    #     bootstrap_residuals = np.concatenate(
    #         [residuals[s:s + block_length] for s in sampled_starts[i]]
    #     )[:n]
    #     bootstrap_b = fitted + bootstrap_residuals
    #
    # sampled_starts (shape (n_bootstrap, n_blocks_needed)) is fully known before the loop
    # starts, so instead of re-gathering blocks one bootstrap iteration at a time, gather all
    # of them at once with a single fancy-indexing operation:
    #
    #   1. block_offsets = [0, 1, ..., block_length - 1].
    #   2. Broadcasting sampled_starts[:, :, None] (n_bootstrap, n_blocks_needed, 1) against
    #      block_offsets[None, None, :] (1, 1, block_length) gives block_indices of shape
    #      (n_bootstrap, n_blocks_needed, block_length), where block_indices[i, j] is the
    #      block_length run of consecutive residual indices for bootstrap i's j-th sampled
    #      block (i.e. sampled_starts[i, j] + block_offsets).
    #   3. residuals[block_indices] gathers those residual values, same shape.
    #   4. .reshape(n_bootstrap, -1) flattens the (n_blocks_needed, block_length) axes into a
    #      single axis per bootstrap row, in the same block-by-block, then-within-block order
    #      that np.concatenate produced per iteration in the original code.
    #   5. [:, :n] truncates each row to n, matching the original per-iteration `[:n]` (the
    #      concatenated blocks run past n since n_blocks_needed * block_length >= n).
    #
    # all_bootstrap_residuals ends up with shape (n_bootstrap, n): row i is exactly what the
    # old code computed as bootstrap_residuals for iteration i. Moving the gather out of the
    # hot loop cut ~27% off this function's time during profiling.
    block_offsets = np.arange(block_length)
    block_indices = sampled_starts[:, :, None] + block_offsets[None, None, :]
    all_bootstrap_residuals = residuals[block_indices].reshape(n_bootstrap, -1)[:, :n]

    for i in range(n_bootstrap):
        holdout_mask = holdout_masks[i]
        train_mask = ~holdout_mask

        bootstrap_b = fitted + all_bootstrap_residuals[i]

        bootstrap_coef, _ = scipy.optimize.nnls(A[train_mask], bootstrap_b[train_mask])
        bootstrap_coefs[i] = bootstrap_coef

        holdout_residuals = A[holdout_mask] @ bootstrap_coef - b[holdout_mask]
        holdout_sum_of_squares[i] = holdout_residuals @ holdout_residuals
        holdout_point_counts[i] = holdout_residuals.shape[0]

    # per-iteration holdout RMSE, reduced for all iterations at once
    bootstrap_pes = np.sqrt(holdout_sum_of_squares / holdout_point_counts)

    return bootstrap_coefs, bootstrap_pes


def reconstruct_from_sampled_starts(residuals, sampled_starts, block_length):
    """Rebuild the residual series each bootstrap iteration actually fits.

    Identical to the gather inside do_moving_block_holdout_bootstrap, so what this
    measures is what the bootstrap really used, not a re-derivation of it.
    """
    n = len(residuals)
    block_offsets = np.arange(block_length)
    block_indices = sampled_starts[:, :, None] + block_offsets[None, None, :]
    return residuals[block_indices].reshape(len(sampled_starts), -1)[:, :n]

In [ ]:
def do_ref_subsets_moving_block_holdout_bootstrap(b, A, M, rng, select_holdout_blocks_fn,
                                                  n_bootstrap=1000, block_length='auto',
                                                  resample_block_length=None):
    """Bootstrap prediction error for every combination of references at each size in M.

    select_holdout_blocks_fn is called once and its draws are shared across all
    combinations so that per-combination prediction errors are directly comparable.

    Parameters
    ----------
    b                        : (n,) observed spectrum normalized fluorescence values
    A                        : (n, n_refs) interpolated reference spectrum fluorescence values
    M                        : list of combination sizes to evaluate (each between 1 and n_refs)
    rng                      : numpy random Generator
    select_holdout_blocks_fn : callable with signature
                               (n, rng, n_bootstrap=..., block_length=...) ->
                               (holdout_masks, sampled_starts, block_length, n_holdout_blocks)
    n_bootstrap              : number of bootstrap iterations
    resample_block_length    : passed through to the selector; None ties the resample
                               blocks to the holdout blocks, as every earlier version did
    block_length             : 'auto' (default) tunes the block length from the residuals
                               with choose_block_length; an int pins it; None leaves the
                               selector on its own round(n ** (1/3)) rule of thumb

    Returns
    -------
    results : dict of numpy arrays, one row per combination across all sizes in M:
        'M'               : (n_combinations,) int
        'ref_indices'     : (n_combinations, max_M) float — column indices into A, NaN for unused
        'bootstrap_coefs' : (n_combinations, n_bootstrap, max_M) float — NaN for unused coefficients
        'bootstrap_pes'   : (n_combinations, n_bootstrap) float
        'coef'            : (n_combinations, max_M) float — full-data NNLS coefficients, NaN for unused
        'fitted'          : (n_combinations, n) float
        'residuals'       : (n_combinations, n) float
        'lags'            : (n_lags+1,) int — ACF lag indices (shared across all combinations)
        'acf_values'      : (n_combinations, n_lags+1) float — ACF of full-data residuals
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int
    n_holdout_blocks : int
    """
    n, n_refs = A.shape
    for m in M:
        if not (1 <= m <= n_refs):
            raise ValueError(f'M value {m} must be between 1 and n_refs={n_refs}')

    max_M = max(M)
    combos = [(m, ref_indices) for m in sorted(M) for ref_indices in combinations(range(n_refs), m)]
    n_combinations = len(combos)
    n_lags = min(40, n // 2)

    all_M = np.zeros(n_combinations, dtype=int)
    all_ref_indices = np.full((n_combinations, max_M), np.nan)
    all_coef = np.full((n_combinations, max_M), np.nan)
    all_fitted = np.zeros((n_combinations, n))
    all_residuals = np.zeros((n_combinations, n))

    for i, (m, ref_indices) in enumerate(combos):
        coef, fitted, residuals = fit_nnls(A[:, ref_indices], b)
        all_M[i] = m
        all_ref_indices[i, :m] = ref_indices
        all_coef[i, :m] = coef
        all_fitted[i] = fitted
        all_residuals[i] = residuals

    all_acf_values = np.zeros((n_combinations, n_lags + 1))
    ci_95 = 1.96 / np.sqrt(n)
    max_sig_lags = []

    for i in range(n_combinations):
        lags, acf_values = calculate_acf(all_residuals[i])
        all_acf_values[i] = acf_values
        sig_mask = np.abs(acf_values[1:]) > ci_95
        max_sig_lags.append(int(np.where(sig_mask)[0][-1]) + 1 if sig_mask.any() else 0)

    print(f'Significant autocorrelation lags across all subsets: {min(max_sig_lags)}–{max(max_sig_lags)}')

    # The residuals are already in hand, so the block length can be sized from the
    # dependence they actually carry rather than from n alone. See choose_block_length
    # for why the aggregate over combinations is a low quantile.
    if isinstance(block_length, str):
        assert block_length == 'auto', f'block_length must be an int, None or "auto", got {block_length!r}'
        block_length = choose_block_length(all_residuals)['block_length']

    holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
        select_holdout_blocks_fn(n, rng, n_bootstrap=n_bootstrap,
                                 block_length=block_length,
                                 resample_block_length=resample_block_length)

    all_bootstrap_coefs = np.full((n_combinations, n_bootstrap, max_M), np.nan)
    all_bootstrap_pes = np.zeros((n_combinations, n_bootstrap))

    for i, (m, ref_indices) in enumerate(combos):
        bootstrap_coefs, bootstrap_pes = do_moving_block_holdout_bootstrap(
            A[:, ref_indices], b, all_fitted[i], all_residuals[i],
            holdout_masks, sampled_starts, block_length,
        )
        all_bootstrap_coefs[i, :, :m] = bootstrap_coefs
        all_bootstrap_pes[i] = bootstrap_pes

    results = {
        'M': all_M,
        'ref_indices': all_ref_indices,
        'bootstrap_coefs': all_bootstrap_coefs,
        'bootstrap_pes': all_bootstrap_pes,
        'coef': all_coef,
        'fitted': all_fitted,
        'residuals': all_residuals,
        'lags': lags,
        'acf_values': all_acf_values,
    }
    return results, holdout_masks, sampled_starts, block_length, n_holdout_blocks

In [ ]:
@names_its_figures
def plot_bootstrap_summary(
    energies, b, fitted, residuals,
    lags, acf_values,
    bootstrap_coefs, bootstrap_pes,
    coef, ref_names,
    spectrum_name, n_bootstrap,
    title_prefix=None, clusterings=None,
):
    """Summarize one fitted combination: the fit, its residuals, and its coefficients.

    One figure per row rather than one shared gridspec. A single grid forces every row onto
    the same column edges, which does two bad things here: a two-panel row splits at
    n_cols // 2, so it is lopsided whenever n_cols is odd (two references gives a 1:2
    split), and the per-reference panels end up narrow with wide gutters because their
    width is set by a grid the wide rows also have to live in. Giving each row its own
    figure lets its panels divide that row evenly, and the rows still read as one unit
    because they share a width.

    Each bootstrap distribution is summarized by its median rather than its mean. A
    prediction error distribution is bounded below by zero and has a long right tail, and a
    coefficient distribution piles up against the non-negativity constraint at zero, so in
    both cases the mean sits above the bulk of the draws and reports a typical value the
    distribution rarely takes. The median is also what the combination search ranks on, so
    the marker on these figures is now the same statistic that chose the combination.

    Parameters
    ----------
    energies : ndarray, shape (n_energies,)
        The energy grid the fit was computed on.
    b : ndarray, shape (n_energies,)
        The measured spectrum, normalized.
    fitted : ndarray, shape (n_energies,)
        The fitted spectrum.
    residuals : ndarray, shape (n_energies,)
        `fitted - b`, the convention fit_nnls uses.
    lags, acf_values : ndarray, shape (n_lags,)
        Autocorrelation of `residuals` and the lags it was evaluated at.
    bootstrap_coefs : ndarray, shape (n_bootstrap, n_refs)
        One row per bootstrap iteration, one column per reference, in `ref_names` order.
    bootstrap_pes : ndarray, shape (n_bootstrap,)
        The holdout prediction error (RMSE) of each bootstrap iteration.
    coef : ndarray, shape (n_refs,)
        The coefficients of the fit to the real spectrum, drawn as `observed` against the
        bootstrap distributions.
    ref_names : sequence of str
        The references in this combination, ordered to match the columns of
        `bootstrap_coefs` and the entries of `coef`.
    spectrum_name : str
        The sample, used as the title of the fit panel.
    n_bootstrap : int
        Iterations behind the distributions, reported in the first figure's suptitle.
    title_prefix : str, optional
        Prepended to every suptitle, e.g. '1st best 3-component fit'. None leaves the
        suptitles bare, which is what the single-fit demo wants.
    clusterings : dict, optional
        {metric: cluster_reference_spectra(...)} over the whole reference pool. Each tree
        becomes a panel in one extra figure, with this combination's references
        highlighted, so the output answers "how good is this fit" and "how distinctive are
        the references it chose" together. None drops that figure.

    Returns
    -------
    tuple of matplotlib.figure.Figure
        The figures, in the order they are meant to be read: the fit, the reference trees
        (only when `clusterings` is given, so the tuple is four figures or five), the
        residual diagnostics, the coefficient histograms, and the coefficient violins.

        Nothing is drawn here -- the caller displays them, typically
        `for fig in plot_bootstrap_summary(...): display(fig)`. Each is closed with
        `plt.close` before being returned, because the inline backend draws every figure
        still open at the end of a cell: left open, each figure would appear twice, once
        from that flush and once from the caller. Closing does not discard anything, and a
        closed figure still renders when displayed, saved, or further edited.
    """
    n = len(residuals)
    n_refs = len(ref_names)
    n_cols = n_refs + 1
    # the mean here is the definition of RMSE, not a summary of a bootstrap distribution,
    # so it stays a mean
    rmse = np.sqrt(np.mean(residuals ** 2))

    # Every row shares this width so the stack lines up. Two dozen leaf labels need about
    # 11 inches per tree, which sets the floor whenever the trees are drawn -- n_cols is 2
    # for a one-reference fit, which would otherwise leave each tree four inches wide.
    fig_width = max(4 * n_cols, 22) if clusterings else 4 * n_cols

    def suptitle_for(what):
        return f'{title_prefix} — {what}' if title_prefix is not None else what

    # ---- the fit itself
    fig_fit, ax_fit = plt.subplots(figsize=(fig_width, 4.5))
    plot_spectrum_fit(energies, b, fitted, residuals, ax=ax_fit)
    ax_fit.set_title(spectrum_name)
    fig_fit.suptitle(
        suptitle_for(f'Moving Block Holdout Bootstrap Distributions ({n_bootstrap} iterations)'),
        fontsize=13,
    )
    # tight_layout doesn't account for suptitle; the rect reserves space so it doesn't overlap
    fig_fit.tight_layout(rect=[0, 0, 1, 0.93])

    # ---- where this combination sits in the reference set
    fig_trees = None
    if clusterings:
        fig_trees, axs = plt.subplots(1, len(clusterings), figsize=(fig_width, 9))
        for ax, (metric, clustering) in zip(np.atleast_1d(axs), clusterings.items()):
            plot_highlighted_reference_dendrogram(
                clustering, highlight={'this fit': list(ref_names)}, ax=ax,
                legend_loc='inside', title=f'{metric} distance',
            )
        fig_trees.suptitle(suptitle_for('Reference Trees'), fontsize=13)
        fig_trees.tight_layout(rect=[0, 0, 1, 0.96])

    # ---- residual diagnostics
    fig_resid, (ax_acf, ax_resid_hist) = plt.subplots(1, 2, figsize=(fig_width, 4.5))
    plot_acf(lags, acf_values, n, ax=ax_acf)
    ax_acf.set_title('Residual Autocorrelation Function')
    ax_acf.legend(fontsize=12)

    plot_residuals_histogram(residuals, ax=ax_resid_hist)
    ax_resid_hist.set_title('Residuals Histogram')
    ax_resid_hist.legend(fontsize=12)
    fig_resid.suptitle(suptitle_for('Residual Diagnostics'), fontsize=13)
    fig_resid.tight_layout(rect=[0, 0, 1, 0.93])

    # ---- bootstrap coefficients, one column per reference plus the prediction error.
    # Two figures with the same width and the same number of columns, so a reference's
    # histogram still sits directly above its violin even though they are separate figures.
    fig_hist, hist_axs = plt.subplots(1, n_cols, figsize=(fig_width, 4.5), squeeze=False)
    fig_violin, violin_axs = plt.subplots(1, n_cols, figsize=(fig_width, 4.5), squeeze=False)
    hist_axs, violin_axs = hist_axs[0], violin_axs[0]

    # columns run from the largest median coefficient to the smallest, so the references
    # carrying the fit are read first
    for j, (coef_median, coef_i, name) \
            in enumerate(sorted(zip(np.median(bootstrap_coefs, axis=0), range(n_refs), ref_names), reverse=True)):

        p2_5 = np.percentile(bootstrap_coefs[:, coef_i], 2.5)
        p97_5 = np.percentile(bootstrap_coefs[:, coef_i], 97.5)

        hist_axs[j].hist(bootstrap_coefs[:, coef_i], bins=40, color='steelblue', alpha=0.7, edgecolor='white')
        hist_axs[j].axvline(coef[coef_i], color='red', linestyle='--', label=f'observed={coef[coef_i]:.3f}')
        hist_axs[j].axvline(coef_median, color='blue', linestyle='--', label=f'median={coef_median:.3f}')
        hist_axs[j].axvline(p2_5, color='green', linestyle=':', label=f'2.5%={p2_5:.3f}')
        hist_axs[j].axvline(p97_5, color='green', linestyle=':', label=f'97.5%={p97_5:.3f}')
        hist_axs[j].set_title(name, fontsize=9)
        hist_axs[j].set_xlabel('Coefficient')
        hist_axs[j].legend(fontsize=8)

        violin_axs[j].violinplot(bootstrap_coefs[:, coef_i])
        violin_axs[j].scatter(
            [0.95], [coef[coef_i]], color='red', zorder=5, marker='o', s=60,
            edgecolors='black', linewidths=0.8, label=f'observed={coef[coef_i]:.3f}',
        )
        violin_axs[j].scatter(
            [1.05], [coef_median], color='blue', zorder=5, marker='D', s=60,
            edgecolors='black', linewidths=0.8, label=f'median={coef_median:.3f}',
        )
        violin_axs[j].set_title(name, fontsize=9)
        violin_axs[j].legend(fontsize=8)

    pes_median = np.median(bootstrap_pes)
    pes_p2_5 = np.percentile(bootstrap_pes, 2.5)
    pes_p97_5 = np.percentile(bootstrap_pes, 97.5)

    hist_axs[n_refs].hist(bootstrap_pes, bins=40, color='darkorange', alpha=0.7, edgecolor='white')
    hist_axs[n_refs].axvline(rmse, color='red', linestyle='--', label=f'RMSE={rmse:.4f}')
    hist_axs[n_refs].axvline(pes_median, color='blue', linestyle='--', label=f'median={pes_median:.4f}')
    hist_axs[n_refs].axvline(pes_p2_5, color='green', linestyle=':', label=f'2.5%={pes_p2_5:.4f}')
    hist_axs[n_refs].axvline(pes_p97_5, color='green', linestyle=':', label=f'97.5%={pes_p97_5:.4f}')
    hist_axs[n_refs].set_title('Holdout Prediction Error')
    hist_axs[n_refs].set_xlabel('Prediction Error (RMSE)')
    hist_axs[n_refs].legend(fontsize=8)

    parts = violin_axs[n_refs].violinplot(bootstrap_pes)
    for pc in parts['bodies']:
        pc.set_facecolor('darkorange')
        pc.set_alpha(0.7)
    violin_axs[n_refs].scatter(
        [0.95], [rmse], color='red', zorder=5, marker='o', s=60,
        edgecolors='black', linewidths=0.8, label=f'RMSE={rmse:.4f}',
    )
    violin_axs[n_refs].scatter(
        [1.05], [pes_median], color='blue', zorder=5, marker='D', s=60,
        edgecolors='black', linewidths=0.8, label=f'median={pes_median:.4f}',
    )
    violin_axs[n_refs].set_title('Holdout Prediction Error')
    violin_axs[n_refs].legend(fontsize=8)

    coef_violin_axes = violin_axs[:n_refs]
    all_ylims = [ax.get_ylim() for ax in coef_violin_axes]
    global_ymin = min(lo for lo, hi in all_ylims)
    global_ymax = max(hi for lo, hi in all_ylims)
    for ax in coef_violin_axes:
        ax.set_ylim(global_ymin, global_ymax)

    for fig, what in ((fig_hist, 'Bootstrap Coefficient Distributions'),
                      (fig_violin, 'Bootstrap Coefficient Distributions (violins)')):
        fig.suptitle(suptitle_for(what), fontsize=13)
        fig.tight_layout(rect=[0, 0, 1, 0.93])

    figures = tuple(fig for fig in (fig_fit, fig_trees, fig_resid, fig_hist, fig_violin)
                    if fig is not None)
    for fig in figures:
        plt.close(fig)
    return figures

<a id="pipeline-selection"></a>

## 7. Choosing between combinations

Two ideas, both of which came out of the development notebook and both of which are
interrogated by [a study below](#study-selection).

**Rank within iterations, not across them.** Every combination was scored on the same held-out
energies on every iteration, so on each iteration they can simply be put in order.
`subset_mean_ranks` averages that position. Ranking this way never compares numbers from
different iterations, which matters because a third of the iterations are much harder than the
rest — see [the diagnostic window](#study-whiteline).

**Report ties, not a winner.** `peci_tie_table` compares each combination against the best one
at its size by a *paired* difference — subtracting two combinations' errors iteration by
iteration cancels whatever made that iteration easy or hard for both — and reports every
combination the data cannot separate from the best as equally good.

In [ ]:
# A confidence interval on a median, three ways.
#
# The tie rules below need one of these, and which to use is not obvious: this repository
# calls three different things a "95% CI". The study *Which interval, and what counts as
# equally good* measures them; the short answer is that they agree on every verdict, that
# scipy's BCa frequently returns nothing at all on these arrays, and that the order statistic
# is calibrated, never fails and costs nothing, so it is the default.
#
# What they estimate is the precision of the median, not the spread of the draws. The draws
# scatter because every bootstrap iteration held out a different set of energies; the interval
# says how well n such draws locate the number in the middle, which is tighter by roughly
# sqrt(n).


# Pre-compute 95% CIs (done once so plots and printed table are consistent)
def bootstrap_ci(values, stat_fn, n_resamples=5000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    idx = rng.integers(0, len(values), size=(n_resamples, len(values)))
    stats = stat_fn(values[idx], axis=1)
    return np.percentile(stats, [2.5, 97.5])


def median_ci_percentile_bootstrap(values, rng, n_resamples=5000):
    """Percentile bootstrap of the median: resample the draws, take the middle 95%.

    Reuses bootstrap_ci from the v1-v5 comparison above so the notebook keeps one
    definition of this interval rather than two that drift apart.
    """
    lo, hi = bootstrap_ci(np.asarray(values), np.median, n_resamples=n_resamples, rng=rng)
    return float(lo), float(hi)


def median_ci_scipy_bca(values, rng):
    """scipy's bias-corrected and accelerated interval, at the defaults.

    The same call mrfitty/prediction_error_fit.py makes to choose a component count, so
    picking this one would make the notebook and the package agree exactly. BCa shifts and
    stretches the percentile interval to correct for skew in the draws, which is the reason
    to prefer it -- prediction errors are bounded below by zero and have a long right tail.
    """
    # BCa fails on these arrays often enough that the warning would bury everything else
    # the study prints -- and the failures are counted and reported rather than ignored, so
    # silencing the warning loses nothing. See the findings below for why they happen.
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        result = scipy.stats.bootstrap(
            data=(np.asarray(values),), statistic=np.median, rng=rng,
        )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


def median_ci_order_statistic(values, rng=None):
    """A distribution-free interval on the median, read off the sorted draws.

    The number of draws below the true median is Binomial(n, 1/2), so counting
    1.96 * sqrt(n) / 2 draws out from the middle in each direction brackets it about 95% of
    the time whatever shape the draws have. No resampling and no random numbers, which is
    why `rng` is accepted and ignored -- it exists so the three estimators can be called
    interchangeably.

    The endpoints are actual draws, so the interval cannot be narrower than the gap between
    neighboring draws. At n = 1000 that granularity is far below the interval's width.
    """
    sorted_values = np.sort(np.asarray(values))
    n = len(sorted_values)
    half_width = 1.96 * np.sqrt(n) / 2.0
    lo_index = max(0, int(np.floor(n / 2.0 - half_width)))
    hi_index = min(n - 1, int(np.ceil(n / 2.0 + half_width)))
    return float(sorted_values[lo_index]), float(sorted_values[hi_index])


MEDIAN_CI_METHODS = {
    'percentile bootstrap': median_ci_percentile_bootstrap,
    'scipy BCa': median_ci_scipy_bca,
    'order statistic': median_ci_order_statistic,
}

In [ ]:
def subset_mean_ranks(bootstrap_pes):
    """Each subset's average finishing position across the bootstrap iterations.

    On every iteration all the subsets were scored on the same held-out energies, so on
    that iteration they can simply be put in order, first to last. A subset's mean rank is
    where it typically lands in that ordering. Lowest mean rank is the one that is usually
    near the front.

    Why rank rather than the median prediction error, which is the obvious choice and is
    what the search reported before: a subset's 1000 prediction errors come from very
    different holdout draws -- about a third of them held out the whiteline and are much
    harder than the rest -- so that column of numbers is a mixture of two groups, and its
    median lands in the sparse gap between them where a small change moves it a long way.

    Ranking avoids that because it only ever compares subsets *within* one iteration, where
    they faced the same holdout and the comparison is like for like. It also avoids a
    subtler trap: medians do not subtract. The median of (A - B) is not the median of A
    minus the median of B, so with the old anchor a subset could carry the higher median
    prediction error while beating the chosen best subset on most iterations -- the table
    and the comparison deciding ties could disagree about who was ahead. A mean rank is an
    average of within-iteration comparisons, so it cannot disagree with them that way.

    Ties in a column share the average of the positions they span, which is what
    scipy.stats.rankdata does by default. Exact ties are rare in continuous prediction
    errors but cost nothing to handle.

    Parameters
    ----------
    bootstrap_pes : ndarray, shape (n_subsets, n_bootstrap)
        The prediction errors of the subsets being ranked against each other. Pass only the
        subsets of one size -- ranks are meaningful against the field they were computed in.

    Returns
    -------
    ndarray, shape (n_subsets,) -- mean rank, 1 being best
    """
    return scipy.stats.rankdata(bootstrap_pes, axis=0).mean(axis=1)


# Which energies the fit is actually deciding on, and what the holdout does to them.


def contiguous_runs(mask):
    """[(first, last)] index pairs for each run of True in a boolean mask."""
    runs, start = [], None
    for i, value in enumerate(mask):
        if value and start is None:
            start = i
        elif not value and start is not None:
            runs.append((start, i - 1))
            start = None
    if start is not None:
        runs.append((start, len(mask) - 1))
    return runs


def diagnostic_window(energies, A, height_fraction=0.5):
    """The energies the references disagree about most -- what the fit is deciding on.

    Spread across the reference pool at each energy, the standard deviation along a row of
    the design matrix, picks out where choosing one reference over another changes the
    prediction. Everywhere else the references agree and holding an energy out asks the fit
    nothing it cannot answer from its neighbours.

    Every energy whose spread clears `height_fraction` of the maximum is in, so the window
    is a mask and not an interval. On this data it comes back as two runs rather than one,
    which is the point: the references part company twice, near 11869.7 eV where reduced
    arsenic absorbs and near 11875.2 eV where arsenate does, and a holdout that takes one
    leaves the other to identify the species from. An interval would have had to either span
    the quiet gap between them or throw one away.

    Defined from the references rather than from the unknown, which is a correction rather
    than a preference. The earlier version took the tallest peak of the *unknown* and walked
    outward while it stayed above half that height. On a spectrum that is half one species
    and half the other the walk never comes back down: for `Ott3_73_AsXANES_spot6_000` it
    ran from 11867.9 to 12097.0 eV, 200 of the 198-point grid's energies, and for its own
    repeat scan `_001` it stopped at 78. It also jumped between the two peaks from one
    unknown to the next, reporting positions 4.86 eV apart for those two scans of one spot.
    The references do not move from one unknown to the next, so this window does not either.

    Parameters
    ----------
    energies        : ndarray, the fitted energy grid
    A               : ndarray (n_energies, n_references), the design matrix
    height_fraction : float, the share of the maximum spread an energy must clear

    Returns
    -------
    dict -- 'mask' (bool array over the grid), 'runs' ([(lo_energy, hi_energy)] per run),
        'peak_energy' (where the spread is greatest), 'n_points', and 'spread' (the
        per-energy reference spread it was derived from, for plotting)
    """
    spread = np.asarray(A).std(axis=1)
    mask = spread > height_fraction * spread.max()
    return {
        'mask': mask,
        'runs': [(float(energies[lo]), float(energies[hi]))
                 for lo, hi in contiguous_runs(mask)],
        'peak_energy': float(energies[int(np.argmax(spread))]),
        'n_points': int(mask.sum()),
        'spread': spread,
    }


def window_absorption(energies, b, window):
    """What the unknown absorbs at the peak of each diagnostic run.

    The quantity that tells these unknowns apart -- reduced arsenic against arsenate -- and
    the one the old unknown-derived window was unstable on. Reading a height per run says
    what a single argmax could not.
    """
    b = np.asarray(b)
    peaks = {}
    for lo, hi in contiguous_runs(window['mask']):
        inside = np.arange(lo, hi + 1)
        at = inside[int(np.argmax(window['spread'][inside]))]
        peaks[float(energies[at])] = float(b[at])
    return peaks


def holdout_regimes(holdout_masks, window, threshold=0.5):
    """Split the bootstrap iterations by whether they held out most of the window.

    An iteration counts as "held out" when more than `threshold` of the window's energies
    were in its holdout. The split is deliberately coarse -- the quantity it stands in for is
    nearly binary, because the window is narrower than one holdout block, so an iteration
    tends to cover almost all of it or almost none.

    Returns
    -------
    dict -- 'held_out' (bool array, one entry per iteration), 'fraction' (how much of the
        window each iteration held out), 'threshold', 'n_held_out', 'n_retained'
    """
    fraction = holdout_masks[:, window['mask']].mean(axis=1)
    held_out = fraction > threshold
    return {
        'held_out': held_out,
        'fraction': fraction,
        'threshold': threshold,
        'n_held_out': int(held_out.sum()),
        'n_retained': int((~held_out).sum()),
    }


def regime_variance_explained(bootstrap_pes, regimes):
    """How much of each subset's prediction error variance is just the regime split.

    The variance within the two groups, weighted by their sizes, against the variance over
    all iterations. What is left over is what knowing the regime removes -- the usual
    "variance explained" of a one-way split, computed here for every subset at once.

    A value near 1 means a subset's prediction error is close to two numbers, one per regime,
    and that summarizing it with a single number describes neither.
    """
    held_out = regimes['held_out']
    n = len(held_out)
    within = (bootstrap_pes[:, held_out].var(axis=1) * held_out.sum()
              + bootstrap_pes[:, ~held_out].var(axis=1) * (~held_out).sum()) / n
    return 1.0 - within / bootstrap_pes.var(axis=1)

In [ ]:
def tie_by_paired_median_ci(differences, median_ci, rng):
    """Tied when a confidence interval on the *median* difference contains zero.

    Not the default; tie_by_paired_distribution is. Kept because it is the more literal
    reading of "compare the prediction error confidence intervals", and because it is the
    comparison mrfitty/prediction_error_fit.py makes when it chooses a component count.

    This asks how precisely the center of the difference distribution is known. Its width
    falls as 1 / sqrt(n_bootstrap), because more iterations locate a median more precisely
    -- so it measures how long the bootstrap ran as much as it measures the data, and
    running longer will eventually separate any two subsets that differ at all. See
    plot_tie_rule_sensitivity.
    """
    lo, hi = median_ci(differences, rng)
    return bool(lo <= 0.0 <= hi), lo, hi


def tie_by_paired_distribution(differences, median_ci=None, rng=None):
    """Tied when the middle 95% of the differences themselves straddles zero. The default.

    What the distribution is
    ------------------------
    `differences` holds one number per bootstrap iteration:

        differences[k] = bootstrap_pes[subset][k] - bootstrap_pes[best][k]

    and each `bootstrap_pes[...][k]` is that subset's holdout prediction error on
    iteration k -- the RMSE of its fit against the measured spectrum at the energies
    iteration k held out (see do_moving_block_holdout_bootstrap). Because
    do_ref_subsets_moving_block_holdout_bootstrap draws the holdout blocks once and scores
    every combination on those same draws, entry k of both arrays refers to the same held
    out energies, so the subtraction is a like-for-like comparison rather than a difference
    of two independent numbers.

    One entry is therefore one answer to "on this particular set of held out energies, by
    how much did this subset predict worse than the best one?" -- negative where it
    predicted better. The n_bootstrap entries together are the distribution of that answer
    over the holdout draws, and its spread is how much the answer depends on which energies
    happened to be held out.

    How the interval is taken
    ------------------------
    The 2.5th and 97.5th percentiles of those entries, via np.percentile, which sorts them
    and interpolates linearly between the two order statistics bracketing each percentile.
    This is the range the differences themselves occupy, *not* a confidence interval on
    their median: no resampling happens here, and `median_ci` and `rng` are accepted only
    so that the two tie rules can be called interchangeably.

    Why this is the default
    -----------------------
    Straddling zero means the subset beat the best one on a non-negligible share of the
    holdout draws -- roughly, at least 2.5% of them -- so the two change places often
    enough that the data does not establish an order between them. That is a property of
    the spectrum and the references. Widening the interval on the *median* instead measures
    how precisely the center is pinned down, which improves as 1 / sqrt(n_bootstrap): it
    reports the iteration count as much as the data, and running the bootstrap longer will
    eventually separate any two subsets that differ at all. The percentiles here do not
    move with the iteration count, only become better estimated. See
    plot_tie_rule_sensitivity, which measures exactly that difference.

    The cost is that this rule is lenient: a subset that loses 90% of the time is still
    called tied. It answers "is the order between these two established?", not "which one
    is better on average" -- the d_win_rate column peci_tie_table records is the reading to
    reach for when the question is the latter.
    """
    lo, hi = np.percentile(differences, [2.5, 97.5])
    return bool(lo <= 0.0 <= hi), float(lo), float(hi)


def peci_tie_table(results, ref_names, tie_rule=tie_by_paired_distribution,
                   median_ci=median_ci_order_statistic, regimes=None, seed=0):
    """One row per combination: its prediction error, and whether it ties the best at its size.

    Ties are decided by a *paired* difference rather than by comparing two subsets'
    intervals for overlap, and the reason is in how the search was run.
    do_ref_subsets_moving_block_holdout_bootstrap calls select_holdout_blocks_fn once and
    reuses those draws for every combination, so on iteration k every subset was scored on
    the same held-out energies. Subtracting two subsets' prediction errors iteration by
    iteration therefore cancels whatever made iteration k easy or hard for both of them,
    and what is left is the difference between the subsets themselves.

    Comparing each subset's own interval against the best subset's interval instead would
    throw that pairing away. Two intervals can overlap comfortably while every paired
    difference has the same sign, so the overlap test calls subsets tied that the data
    plainly separates -- it answers a question nobody asked ("could these two medians be
    equal if the subsets had been scored independently?") rather than the question the
    search poses ("is this subset worse than the best one on the same holdouts?").

    Parameters
    ----------
    results   : dict returned by do_ref_subsets_moving_block_holdout_bootstrap
    ref_names : list of reference spectrum names, indexed by results['ref_indices']
    regimes   : optional dict from holdout_regimes. When given, four more columns report
                the same comparison separately for the iterations that held out the
                whiteline and those that did not.
    tie_rule  : callable (differences, median_ci, rng) -> (tied, lo, hi). The default
                brackets the middle 95% of the paired differences themselves -- read its
                docstring, which sets out how that distribution is formed.
                tie_by_paired_median_ci puts a confidence interval on their median instead;
                on this data the two disagree about almost everything, and
                plot_tie_rule_sensitivity is the measurement behind preferring the default.
    median_ci : callable (values, rng) -> (lo, hi), used by whichever tie_rule wants one.
                The default is the order statistic interval, which the study above measured
                as calibrated on both the prediction errors and the paired differences,
                never failing, and fast enough to run on every combination without thinking
                about it. A resampling estimator works here too and costs minutes rather
                than milliseconds.
    seed      : int. Each combination gets its own generator derived from this, so a row
                does not depend on how many rows were computed before it.

    Returns
    -------
    pandas.DataFrame -- one row per combination, columns:
        'combination'  : row index into the results arrays
        'M'            : subset size
        'ref_indices'  : tuple of column indices into the design matrix
        'ref_names'    : tuple of reference names
        'mean_rank'    : average finishing position across iterations, 1 being best; the
                         anchor and the reported order, see subset_mean_ranks
        'pe_median'    : median bootstrap prediction error, reported but no longer selected
                         on
        'pe_ci_lo/hi'  : interval on that median
        'is_best'      : lowest mean_rank at this subset size -- the subset the tie set is
                         anchored on
        'd_median'     : median paired difference against the best subset at this size
        'd_lo/hi'      : the interval tie_rule put around that difference -- what it means
                         depends on the rule, which is why these are not named 'ci'
        'd_win_rate'   : fraction of iterations where this subset beat the best one, a
                         reading of the same comparison that no interval convention
                         mediates
        'tied'         : whether this subset is as good as the best one
        'rank'         : position within the subset size, by mean_rank ascending
        'pe_median_held_out', 'pe_median_retained', 'd_win_rate_held_out',
        'd_win_rate_retained' : present only when `regimes` is given
    """
    rows = []
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        # the anchor, and the order the table is reported in, is the mean rank across
        # iterations rather than the median prediction error -- see subset_mean_ranks
        mean_ranks = subset_mean_ranks(results['bootstrap_pes'][m_indices])
        order_within_m = np.argsort(mean_ranks)
        order = m_indices[order_within_m]
        best = int(order[0])
        best_prediction_errors = results['bootstrap_pes'][best]

        for rank, (i, mean_rank) in enumerate(zip(order, mean_ranks[order_within_m]), start=1):
            i = int(i)
            prediction_errors = results['bootstrap_pes'][i]
            rng = np.random.default_rng([seed, i])
            pe_ci_lo, pe_ci_hi = median_ci(prediction_errors, rng)

            if i == best:
                # the best subset against itself is a column of exact zeros, which has no
                # spread for a rule to work with -- scipy's BCa returns NaN on it. It ties
                # itself by definition, so say so rather than asking.
                tied, d_median, d_lo, d_hi, win_rate = True, 0.0, 0.0, 0.0, 0.0
            else:
                differences = prediction_errors - best_prediction_errors
                tied, d_lo, d_hi = tie_rule(differences, median_ci, rng)
                d_median = float(np.median(differences))
                win_rate = float((differences < 0).mean())

            # results['ref_indices'] rows are float and NaN-padded to max_M, so the row is
            # sliced to its own m before int() -- int(nan) raises
            ref_idx = tuple(int(j) for j in results['ref_indices'][i, :m])
            rows.append({
                'combination': i,
                'M': int(m),
                'ref_indices': ref_idx,
                'ref_names': tuple(ref_names[j] for j in ref_idx),
                'mean_rank': float(mean_rank),
                'pe_median': float(np.median(prediction_errors)),
                'pe_ci_lo': pe_ci_lo,
                'pe_ci_hi': pe_ci_hi,
                'is_best': i == best,
                'd_median': d_median,
                'd_lo': d_lo,
                'd_hi': d_hi,
                'd_win_rate': win_rate,
                'tied': tied,
                'rank': rank,
            })
            if regimes is not None:
                # the same comparison inside each regime, because a subset can be good at
                # reconstructing the whiteline, good at interpolating the rest, or both,
                # and the pooled numbers above cannot tell those apart
                held_out = regimes['held_out']
                differences = prediction_errors - best_prediction_errors
                rows[-1].update({
                    'pe_median_held_out': float(np.median(prediction_errors[held_out])),
                    'pe_median_retained': float(np.median(prediction_errors[~held_out])),
                    'd_win_rate_held_out': float((differences[held_out] < 0).mean()),
                    'd_win_rate_retained': float((differences[~held_out] < 0).mean()),
                })

    return pd.DataFrame(rows)


def peci_tie_counts(tie_table):
    """How many combinations tie the best one at each subset size, against how many exist."""
    return tie_table.groupby('M').agg(
        tied=('tied', 'sum'), combinations=('tied', 'size'),
    ).astype(int)


def tie_rule_sensitivity(results, ref_names, n_bootstrap_grid, seed=0):
    """Tie-set size under both rules as the bootstrap is allowed to run longer.

    The point of the sweep is that one of these rules is a statement about the data and the
    other is partly a statement about the iteration count. Truncating the existing draws to
    the first n of them is exactly a shorter run, since the iterations are independent and
    identically distributed, so no refitting is needed.
    """
    rules = {'CI on the median difference': tie_by_paired_median_ci,
             'middle 95% of the differences': tie_by_paired_distribution}
    rows = []
    for n_bootstrap in n_bootstrap_grid:
        truncated = dict(results)
        truncated['bootstrap_pes'] = results['bootstrap_pes'][:, :n_bootstrap]
        for rule_name, tie_rule in rules.items():
            counts = peci_tie_counts(
                peci_tie_table(truncated, ref_names, tie_rule=tie_rule, seed=seed)
            )
            rows.extend({'n_bootstrap': n_bootstrap, 'rule': rule_name, 'M': m,
                         'tied': int(counts.loc[m, 'tied']),
                         'combinations': int(counts.loc[m, 'combinations'])}
                        for m in counts.index)
    return pd.DataFrame(rows)

In [ ]:
def _ordinal(n):
    if 11 <= n % 100 <= 13:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f'{n}{suffix}'


def tied_subset_reference_shares(tie_table, m):
    """{reference: fraction of this size's tied subsets that contain it}.

    A share rather than the bare union, because the union is usually everything: at M=2 and
    M=3 over 24 references the tie set runs to a hundred subsets or more and between them
    they use every reference in the pool, so "appears in a tied subset" separates nothing.
    How *often* a reference appears does separate them -- a reference in four fifths of the
    tied subsets is one the data insists on, whatever else varies around it, and a flat set
    of shares says the opposite: that the tie set has no favourites.

    References that appear in no tied subset are left out rather than recorded as zero, so
    a caller can tell "not used" from "used rarely" without comparing against zero.
    """
    tied = tie_table[(tie_table['M'] == m) & tie_table['tied']]
    shares = collections.Counter(name for names in tied['ref_names'] for name in names)
    return {name: count / len(tied) for name, count in shares.items()}


def plot_peci_tie_panel(tie_table, m, ax, n_show=40):
    """The tie structure at one subset size: paired difference against the best subset.

    Everything is drawn relative to the best subset, which therefore sits at exactly zero.
    A combination whose error bar touches the dashed line is one the data cannot separate
    from the best -- the tie set is read straight off the figure, without consulting the
    table.

    The error bars are whatever interval peci_tie_table's tie_rule put around the
    difference, so the figure shows the rule that was actually applied rather than a second
    convention drawn alongside it.

    Only the n_show lowest-error combinations are drawn; at 24 references there are 2,024
    of them at M=3, and the ones far to the right are not close calls. The title says how
    many there are in total.
    """
    rows = tie_table[tie_table['M'] == m].sort_values('rank')
    n_total = len(rows)
    n_tied = int(rows['tied'].sum())
    shown = rows.head(n_show)

    x = np.arange(len(shown))
    center = shown['d_median'].to_numpy()
    yerr = [center - shown['d_lo'].to_numpy(), shown['d_hi'].to_numpy() - center]
    tied = shown['tied'].to_numpy()

    for mask, color, label in ((tied, 'mediumseagreen', f'tied with the best ({n_tied})'),
                               (~tied, 'steelblue', 'worse than the best')):
        if not mask.any():
            continue
        ax.errorbar(
            x[mask], center[mask], yerr=[yerr[0][mask], yerr[1][mask]],
            fmt='o', color=color, capsize=3, linewidth=1.2, markersize=4,
            linestyle='none', label=label,
        )

    ax.axhline(0.0, color='red', linestyle='--', linewidth=1.0, label='the best subset')
    ax.set_xlabel(f'Reference subset, by mean rank across iterations '
                  f'({len(shown)} of {n_total} shown)')
    ax.set_ylabel('Median paired difference in PE')
    ax.set_title(f'M={m}: {n_tied} of {n_total} subsets are as good as the best')
    ax.legend(fontsize=8)


@names_its_figures
def plot_best_peci_subset_bootstrap_summaries(
    results, ref_names, energies, b, n_bootstrap, spectrum_name, tie_table,
    max_subsets_per_size=3, n_show=40, clusterings=None,
):
    """Call plot_bootstrap_summary for the subsets that are as good as the best one.

    The sibling of plot_best_subset_bootstrap_summaries, differing in what it means by
    "best" twice over. That function takes the three lowest median prediction errors at
    each subset size and labels them 1st, 2nd and 3rd, which asserts an ordering the
    numbers may not support. This one anchors on the subset with the lowest mean rank
    across iterations (see subset_mean_ranks) and reports every subset whose prediction
    error is not distinguishable from it (see peci_tie_table), saying so in the titles so
    nothing implies a ranking within the tie set.

    The two need not pick the same anchor. best_subsets_by_size still selects on the median
    prediction error, so where the dendrograms it highlights disagree with the tie sets
    here, this is why.

    Parameters
    ----------
    results              : dict returned by do_ref_subsets_moving_block_holdout_bootstrap
    ref_names            : list of reference spectrum names
    energies             : energy grid array
    b                    : observed spectrum values
    n_bootstrap          : number of bootstrap iterations, passed to plot_bootstrap_summary
    spectrum_name        : str, used in plot titles
    tie_table            : the DataFrame from peci_tie_table, computed by the caller. It is
                           a separate argument rather than computed here because a tie set
                           is worth looking at on its own, and because a resampling
                           estimator makes it a minutes-long computation that should not be
                           hidden inside a plotting call.
    max_subsets_per_size : how many tied subsets to draw summaries for at each subset size,
                           lowest mean rank first. This is the only thing
                           bounding the output: a tie set can hold hundreds of subsets and
                           each one costs four or five figures. None draws all of them.
    n_show               : combinations per tie-structure panel, passed through
    clusterings          : optional {metric: clustering} over the whole reference pool.
                           Adds a figure of trees after each tie-structure panel, shading
                           each reference by how much of that size's tie set uses it, and
                           is forwarded to plot_bootstrap_summary so each per-subset figure
                           also shows where its own combination sits. Without it no tree
                           figure is produced.

    Returns
    -------
    tuple of matplotlib.figure.Figure
        For each subset size, the tie structure at that size, then the reference trees as
        a second figure when `clusterings` is given, then the summaries of the subsets
        drawn from it. Every figure is closed before being returned, for the
        reason plot_bootstrap_summary gives; the caller displays them, typically
        `for fig in plot_best_peci_subset_bootstrap_summaries(...): display(fig)`.
    """
    figures = []
    for m in sorted(set(tie_table['M'])):
        rows = tie_table[tie_table['M'] == m].sort_values('rank')
        tied_rows = rows[rows['tied']]
        n_tied = len(tied_rows)

        # 11 x 9 inches per tree is what plot_cluster_metric_comparison settled on for a
        # 24-name leaf column; the tie structure takes the same width so the two figures
        # stack cleanly.
        figure_width = max(11 * len(clusterings), 12) if clusterings else 12

        fig_ties, ax_ties = plt.subplots(figsize=(figure_width, 4.5))
        plot_peci_tie_panel(tie_table, m, ax_ties, n_show=n_show)
        fig_ties.suptitle(
            f'Subsets not distinguishable from the best {m}-component fit — {spectrum_name}',
            fontsize=13,
        )
        # tight_layout doesn't account for suptitle; the rect reserves space for it
        fig_ties.tight_layout(rect=[0, 0, 1, 0.91])
        plt.close(fig_ties)
        figures.append(fig_ties)

        if clusterings:
            # A figure of its own rather than a second row of the one above. In a single
            # figure the tie panel spans the full width while each tree spans only its
            # half, and once the leaf labels and their bars have taken their room the axes
            # no longer line up with the panel above -- which is what the eye goes looking
            # for. Separate figures of the same width do not invite the comparison.
            shares = tied_subset_reference_shares(tie_table, m)
            fig_trees, tree_axs = plt.subplots(
                1, len(clusterings), figsize=(figure_width, 9), squeeze=False,
            )
            for ax_tree, (metric, clustering) in zip(tree_axs[0], clusterings.items()):
                plot_weighted_reference_dendrogram(
                    clustering, leaf_weights=shares,
                    leaf_weight_label=f'share of the tie set containing the reference '
                                      f'(longest = {max(shares.values()):.0%})',
                    # 'below' would hang the legend off the bottom of the figure
                    ax=ax_tree, legend_loc='inside', title=f'{metric} distance',
                )
            fig_trees.suptitle(
                f'Where the {n_tied} tied subsets draw from: every reference, shaded and '
                f'barred by how much of the tie set uses it',
                fontsize=13,
            )
            fig_trees.tight_layout(rect=[0, 0, 1, 0.94])

            # Line the two figures up. Being separate figures is not enough on its own:
            # each is laid out alone, and the trees keep a wide margin outside their axes
            # for the leaf labels and the bars beside them, so their plot boxes would stop
            # well short of the panel above. Both figures are the same width, so a fraction
            # means the same inches in either, and setting both to one span makes the boxes
            # start and end together. The right edge is the trees' -- they are the ones
            # with something outside the axes to make room for -- and the left is whichever
            # of the two needs more, so the tie panel keeps room for its y axis labels.
            fig_ties.draw_without_rendering()
            fig_trees.draw_without_rendering()
            tree_boxes = [ax_tree.get_position() for ax_tree in fig_trees.axes]
            left = max(fig_ties.axes[0].get_position().x0, min(box.x0 for box in tree_boxes))
            right = max(box.x1 for box in tree_boxes)
            fig_ties.subplots_adjust(left=left, right=right)
            fig_trees.subplots_adjust(left=left, right=right)

            plt.close(fig_trees)
            figures.append(fig_trees)

        selected = tied_rows if max_subsets_per_size is None \
            else tied_rows.head(max_subsets_per_size)

        for rank, row in enumerate(selected.itertuples(), start=1):
            i = row.combination
            if n_tied == 1:
                title_prefix = f'the only {m}-component fit not beaten by the best'
            else:
                # "of n tied" rather than "2nd best": the order within a tie set is the
                # mean rank, and these subsets were just found not to differ on the
                # quantity that ranking is built from
                title_prefix = f'{_ordinal(rank)} of {n_tied} tied {m}-component fits'
            figures.extend(plot_bootstrap_summary(
                energies, b,
                results['fitted'][i],
                results['residuals'][i],
                results['lags'],
                results['acf_values'][i],
                results['bootstrap_coefs'][i, :, :m],
                results['bootstrap_pes'][i],
                results['coef'][i, :m],
                list(row.ref_names),
                spectrum_name=spectrum_name,
                n_bootstrap=n_bootstrap,
                title_prefix=title_prefix,
                clusterings=clusterings,
            ))

    return tuple(figures)


@names_its_figures
def plot_tie_rule_sensitivity(sensitivity, spectrum_name):
    """How each tie rule's answer moves as the bootstrap is allowed to run longer.

    One panel per subset size, tie-set size against n_bootstrap, one line per rule. A rule
    whose line drifts downward is reporting the iteration count: run it longer and fewer
    subsets survive, until only the winner is left. A rule whose line is flat is reporting
    the data, and would give the same answer at any iteration count.

    Returns
    -------
    matplotlib.figure.Figure
        The one figure this builds, closed, for the caller to display.
    """
    subset_sizes = sorted(set(sensitivity['M']))
    rules = list(dict.fromkeys(sensitivity['rule']))
    colors = {rule: color for rule, color in zip(rules, ('steelblue', 'mediumseagreen'))}

    fig, axs = plt.subplots(1, len(subset_sizes),
                            figsize=(max(10, 5 * len(subset_sizes)), 4.5), squeeze=False)
    for ax, m in zip(axs[0], subset_sizes):
        at_m = sensitivity[sensitivity['M'] == m]
        for rule in rules:
            rows = at_m[at_m['rule'] == rule].sort_values('n_bootstrap')
            ax.plot(rows['n_bootstrap'], rows['tied'], 'o-', color=colors[rule],
                    linewidth=1.5, markersize=5, label=rule)
        ax.set_xscale('log')
        ax.set_xlabel('Bootstrap iterations')
        ax.set_title(f'M={m} ({int(at_m["combinations"].iloc[0])} combinations)')
        ax.legend(fontsize=8)
    axs[0][0].set_ylabel('Subsets tied with the best')

    fig.suptitle(
        f'Does the tie rule describe the data or the iteration count? — {spectrum_name}',
        fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.91])
    plt.close(fig)
    return fig

In [ ]:
# ---------------------------------------------------------------------------
# Tests for the selection machinery: ranking, the tie rules, and the diagnostic
# window the regime split is built on.
# ---------------------------------------------------------------------------

def _paired_pe_results(offsets, n_bootstrap=1000, shared_scale=1.0, noise_scale=0.02,
                       seed=0, subset_sizes=None, n_energies=30):
    """A results dict whose prediction errors are paired the way a real search's are.

    Every combination gets the same per-iteration term -- standing in for the holdout draw
    all of them were scored on -- plus its own offset and a little independent noise. The
    offset is the truth the tie rules have to recover: a combination with a larger offset
    really is worse, by exactly the amount the rules are asked to detect.

    The arrays the tie rules never look at -- fits, residuals, autocorrelations,
    coefficients -- are filled in with the right shapes and NaN padding so that
    plot_bootstrap_summary can be driven from the same fixture.
    """
    rng = np.random.default_rng(seed)
    offsets = np.asarray(offsets, dtype=float)
    n_combinations = len(offsets)
    if subset_sizes is None:
        subset_sizes = np.ones(n_combinations, dtype=int)
    subset_sizes = np.asarray(subset_sizes, dtype=int)
    max_M = int(subset_sizes.max())

    shared = shared_scale * rng.random(n_bootstrap)
    prediction_errors = (
        shared[None, :]
        + offsets[:, None]
        + noise_scale * rng.standard_normal((n_combinations, n_bootstrap))
    )

    b = rng.normal(size=n_energies)
    residuals = rng.normal(size=(n_combinations, n_energies)) * 0.01
    results = {
        'M': subset_sizes,
        'ref_indices': np.full((n_combinations, max_M), np.nan),
        'coef': np.full((n_combinations, max_M), np.nan),
        'bootstrap_coefs': np.full((n_combinations, n_bootstrap, max_M), np.nan),
        'bootstrap_pes': prediction_errors,
        'fitted': b[None, :] + residuals,
        'residuals': residuals,
        'lags': np.arange(6),
        'acf_values': rng.random((n_combinations, 6)),
    }
    n_refs = max(max_M, 2)
    for i, m in enumerate(subset_sizes):
        # a distinct combination of the reference pool for each row, wrapped so that any
        # number of rows can be built without running out
        results['ref_indices'][i, :m] = [(i + j) % n_refs for j in range(m)]
        results['coef'][i, :m] = rng.random(m)
        results['bootstrap_coefs'][i, :, :m] = rng.random((n_bootstrap, m))

    ref_names = [f'ref{i}.e' for i in range(n_refs)]
    return results, ref_names, b


def _two_peak_design_matrix(n_energies=120):
    """References that part company in two places, as the arsenic pool does.

    Half the references carry a peak at index 30 and half at index 80, so the spread across
    the pool is large at both and small everywhere else.
    """
    grid = np.arange(n_energies)
    reduced = np.exp(-0.5 * ((grid - 30) / 3.0) ** 2)
    oxidised = np.exp(-0.5 * ((grid - 80) / 3.0) ** 2)
    background = 0.3 * np.tanh((grid - 20) / 10.0)
    columns = [background + (reduced if i % 2 else oxidised) for i in range(8)]
    return grid.astype(float), np.column_stack(columns)


def test_the_window_finds_both_places_the_references_part():
    """Two runs, not one interval spanning the quiet gap between them."""
    energies, A = _two_peak_design_matrix()
    window = diagnostic_window(energies, A)

    assert len(window['runs']) == 2, f'expected two runs, got {window["runs"]}'
    (first_lo, first_hi), (second_lo, second_hi) = window['runs']
    assert first_lo <= 30 <= first_hi and second_lo <= 80 <= second_hi
    # the quiet middle is out
    assert not window['mask'][55]
    assert window['n_points'] == int(window['mask'].sum())


def test_the_window_does_not_depend_on_the_unknown():
    """The correction: it is a property of the references, so it cannot move between samples.

    The earlier version walked outward from the tallest peak of the *unknown* and so jumped
    between the two peaks, and on a half-and-half spectrum ran away across the whole grid.
    """
    energies, A = _two_peak_design_matrix()
    mostly_reduced = A[:, 1] * 3.0 + A[:, 0] * 0.2
    mostly_oxidised = A[:, 0] * 3.0 + A[:, 1] * 0.2

    window = diagnostic_window(energies, A)
    for unknown in (mostly_reduced, mostly_oxidised):
        # window_absorption reads the unknown, but the window itself never sees it
        heights = window_absorption(energies, unknown, window)
        assert len(heights) == 2, 'one height per run'
    reduced_heights = window_absorption(energies, mostly_reduced, window)
    oxidised_heights = window_absorption(energies, mostly_oxidised, window)
    assert reduced_heights != oxidised_heights, \
        'the heights should tell the two unknowns apart even though the window does not move'


def test_contiguous_runs_finds_the_runs():
    assert contiguous_runs(np.array([0, 1, 1, 0, 0, 1, 0], dtype=bool)) == [(1, 2), (5, 5)]
    assert contiguous_runs(np.zeros(4, dtype=bool)) == []
    assert contiguous_runs(np.ones(3, dtype=bool)) == [(0, 2)]


def test_median_ci_estimators_bracket_the_median():
    """Whatever else they disagree about, an interval has to contain the point estimate."""
    values = np.random.default_rng(0).lognormal(size=500)
    for name, median_ci in MEDIAN_CI_METHODS.items():
        lo, hi = median_ci(values, np.random.default_rng(0))
        assert lo <= np.median(values) <= hi, f'{name} produced [{lo}, {hi}]'


def test_order_statistic_ci_survives_a_constant_sample():
    """The best subset's difference against itself is all zeros, and must not blow up.

    scipy's BCa returns NaN here, which is why peci_tie_table special-cases the best
    subset rather than trusting whatever the estimator says.
    """
    lo, hi = median_ci_order_statistic(np.zeros(100), None)
    assert (lo, hi) == (0.0, 0.0)


def test_peci_tie_table_has_one_row_per_combination():
    results, ref_names, _ = _paired_pe_results([0.0, 0.05, 0.10, 0.15])
    tie_table = peci_tie_table(results, ref_names)

    assert len(tie_table) == len(results['M'])
    assert sorted(tie_table['rank']) == [1, 2, 3, 4]
    assert set(tie_table['combination']) == set(range(4))


def test_mean_rank_anchor_ignores_how_hard_an_iteration_was():
    """Why the anchor is a rank: a mixture of easy and hard iterations must not sway it.

    Subset A wins on every single iteration by a whisker. Subset B loses every one, but
    its losses fall in the hard iterations, where all the errors are large. A summary
    taken down the column can be swayed by that; a rank cannot, because it only compares
    the two inside an iteration, and A is ahead in all of them.
    """
    n_bootstrap = 400
    hard = np.zeros(n_bootstrap, dtype=bool)
    hard[: n_bootstrap // 3] = True          # a third of the iterations are much harder
    a = np.where(hard, 1.0, 0.1)
    b_pes = a + 0.001                        # B is worse on every iteration, always

    prediction_errors = np.vstack([a, b_pes])
    mean_ranks = subset_mean_ranks(prediction_errors)

    assert mean_ranks[0] < mean_ranks[1], 'the subset that wins every iteration must rank first'
    assert mean_ranks[0] == 1.0 and mean_ranks[1] == 2.0


def test_mean_rank_handles_exact_ties():
    """Tied prediction errors share the average of the positions they span."""
    prediction_errors = np.vstack([np.ones(10), np.ones(10), np.full(10, 2.0)])
    np.testing.assert_allclose(subset_mean_ranks(prediction_errors), [1.5, 1.5, 3.0])


def test_peci_anchor_is_the_lowest_mean_rank():
    """is_best, rank order and mean_rank all agree, and the offsets decide the order."""
    results, ref_names, _ = _paired_pe_results([0.02, 0.0, 0.04], noise_scale=0.001)
    tie_table = peci_tie_table(results, ref_names)

    assert tie_table['mean_rank'].is_monotonic_increasing, 'rows come back in anchor order'
    best = tie_table[tie_table['is_best']]
    assert len(best) == 1 and best['rank'].iloc[0] == 1
    # the middle combination has the smallest offset, so it is the one that should anchor
    assert best['combination'].iloc[0] == 1


def test_holdout_regimes_split_on_the_window():
    """An iteration is in the held-out regime when most of the window is in its holdout."""
    n_energies, n_bootstrap = 20, 4
    window = {'mask': np.zeros(n_energies, dtype=bool)}
    window['mask'][5:9] = True               # a four-point window

    holdout_masks = np.zeros((n_bootstrap, n_energies), dtype=bool)
    holdout_masks[0, 5:9] = True             # all of the window
    holdout_masks[1, 5:8] = True             # three quarters of it
    holdout_masks[2, 5:7] = True             # half -- not more than half, so retained
    holdout_masks[3, 0:4] = True             # none of it

    regimes = holdout_regimes(holdout_masks, window)
    np.testing.assert_array_equal(regimes['held_out'], [True, True, False, False])
    assert regimes['n_held_out'] == 2 and regimes['n_retained'] == 2


def test_regime_variance_explained_recognizes_a_two_level_signal():
    """All spread between the regimes gives 1; none gives 0."""
    held_out = np.array([True] * 50 + [False] * 50)
    regimes = {'held_out': held_out}

    two_levels = np.where(held_out, 0.10, 0.02)[None, :]
    assert regime_variance_explained(two_levels, regimes)[0] > 0.999

    rng = np.random.default_rng(0)
    unrelated = rng.normal(size=(1, 100))
    assert abs(regime_variance_explained(unrelated, regimes)[0]) < 0.1


def test_peci_tie_table_regime_columns_are_opt_in():
    """The stratified columns appear only when regimes are supplied, and they add up."""
    results, ref_names, _ = _paired_pe_results([0.0, 0.05, 0.1], n_bootstrap=200)
    stratified = ['pe_median_held_out', 'pe_median_retained',
                  'd_win_rate_held_out', 'd_win_rate_retained']

    plain = peci_tie_table(results, ref_names)
    assert not any(column in plain.columns for column in stratified)

    held_out = np.zeros(200, dtype=bool)
    held_out[:80] = True
    regimes = {'held_out': held_out}
    split = peci_tie_table(results, ref_names, regimes=regimes)
    assert all(column in split.columns for column in stratified)

    # the two regime win rates, weighted by how many iterations each holds, are the overall one
    weighted = (split['d_win_rate_held_out'] * 80 + split['d_win_rate_retained'] * 120) / 200
    np.testing.assert_allclose(weighted, split['d_win_rate'], atol=1e-12)


def test_peci_best_subset_ties_itself():
    results, ref_names, _ = _paired_pe_results([0.0, 0.05, 0.10])
    tie_table = peci_tie_table(results, ref_names)
    best = tie_table[tie_table['is_best']]

    assert len(best) == 1, 'exactly one combination is the best at this subset size'
    assert best['rank'].iloc[0] == 1
    assert best['d_median'].iloc[0] == 0.0
    assert bool(best['tied'].iloc[0]), 'a subset is always as good as itself'


def test_peci_excludes_a_clearly_worse_subset():
    """An offset far larger than the noise is not a tie under either rule."""
    results, ref_names, _ = _paired_pe_results([0.0, 1.0], noise_scale=0.01)
    for tie_rule in (tie_by_paired_median_ci, tie_by_paired_distribution):
        tie_table = peci_tie_table(results, ref_names, tie_rule=tie_rule)
        loser = tie_table[~tie_table['is_best']]
        assert not loser['tied'].iloc[0], f'{tie_rule.__name__} called a 1.0 offset a tie'
        assert loser['d_win_rate'].iloc[0] == 0.0


def test_peci_paired_comparison_is_sharper_than_overlapping_intervals():
    """The argument for pairing, as a test rather than a comment.

    Both combinations are dominated by the same per-iteration term, so their prediction
    errors scatter over a wide range and their own intervals overlap almost completely.
    The paired difference removes that shared term and separates them anyway.
    """
    results, ref_names, _ = _paired_pe_results([0.0, 0.05], shared_scale=10.0, noise_scale=0.01)
    tie_table = peci_tie_table(results, ref_names)
    best, other = tie_table.iloc[0], tie_table.iloc[1]

    overlapping = (best['pe_ci_lo'] <= other['pe_ci_hi']
                   and other['pe_ci_lo'] <= best['pe_ci_hi'])
    assert overlapping, 'the setup is only interesting if the unpaired intervals overlap'
    assert not other['tied'], 'the paired comparison should separate what the overlap cannot'


def test_tie_rules_differ_in_what_more_iterations_buy():
    """The property that decides which rule to select on.

    A confidence interval on a median narrows as sqrt(n), so running the bootstrap longer
    eventually separates any two subsets that differ at all -- its answer is partly a
    statement about the iteration count. Bracketing the differences themselves measures how
    far apart the two subsets are across holdout draws, which is a property of the data and
    does not sharpen with more iterations.

    The widths are what is asserted, not the tie verdicts: a verdict flips only once the
    width crosses the offset, and where that happens for one particular draw is luck.
    """
    results, ref_names, _ = _paired_pe_results(
        [0.0, 0.05], shared_scale=1.0, noise_scale=0.5, n_bootstrap=4000,
    )
    widths = {}
    for tie_rule in (tie_by_paired_median_ci, tie_by_paired_distribution):
        for n_bootstrap in (250, 4000):
            truncated = dict(results)
            truncated['bootstrap_pes'] = results['bootstrap_pes'][:, :n_bootstrap]
            tie_table = peci_tie_table(truncated, ref_names, tie_rule=tie_rule)
            other = tie_table[~tie_table['is_best']].iloc[0]
            widths[(tie_rule.__name__, n_bootstrap)] = other['d_hi'] - other['d_lo']

    # sixteen times the iterations should quarter the width of an interval on a median;
    # halving is the loose version of that, and is what is asserted
    assert widths[('tie_by_paired_median_ci', 4000)] \
        < 0.5 * widths[('tie_by_paired_median_ci', 250)], \
        f'CI on the median did not narrow with more iterations: {widths}'

    # the spread of the differences is a property of the data, so it should barely move
    width_ratio = (widths[('tie_by_paired_distribution', 4000)]
                   / widths[('tie_by_paired_distribution', 250)])
    assert 0.8 < width_ratio < 1.25, \
        f'bracketing the differences moved by {width_ratio:.2f}x with more iterations'


def test_tie_rule_sensitivity_reports_both_rules_at_every_size():
    """The sweep the finding above is read off: one row per (iteration count, rule, size)."""
    results, ref_names, _ = _paired_pe_results(
        [0.0, 0.01, 0.02, 0.0, 0.01, 0.02], noise_scale=0.1, subset_sizes=[1, 1, 1, 2, 2, 2],
    )
    sensitivity = tie_rule_sensitivity(results, ref_names, n_bootstrap_grid=[100, 1000])

    assert len(sensitivity) == 2 * 2 * 2, 'two iteration counts, two rules, two subset sizes'
    assert set(sensitivity['combinations']) == {3}
    assert (sensitivity['tied'] >= 1).all(), 'the best subset always ties itself'
    assert (sensitivity['tied'] <= sensitivity['combinations']).all()


def test_tied_subset_reference_shares_counts_the_tie_set():
    """How much of the tie set each reference is in, ignoring the subsets that are not tied."""
    tie_table = pd.DataFrame([
        {'M': 2, 'tied': True,  'ref_names': ('a.e', 'b.e')},
        {'M': 2, 'tied': True,  'ref_names': ('b.e', 'c.e')},   # b.e is in both
        {'M': 2, 'tied': True,  'ref_names': ('b.e', 'c.e')},
        {'M': 2, 'tied': False, 'ref_names': ('d.e', 'e.e')},   # not tied, so not counted
        {'M': 1, 'tied': True,  'ref_names': ('z.e',)},         # a different subset size
    ])
    shares = tied_subset_reference_shares(tie_table, 2)

    assert shares == {'a.e': 1 / 3, 'b.e': 1.0, 'c.e': 2 / 3}
    assert 'd.e' not in shares, 'an untied subset contributes nothing'
    assert tied_subset_reference_shares(tie_table, 1) == {'z.e': 1.0}


def test_the_trees_are_their_own_figure_and_line_up_with_the_panel():
    """Two figures, not two rows -- and their plot boxes start and end together."""
    results, ref_names, b = _paired_pe_results(
        [0.0, 0.002, 0.004], noise_scale=0.05, n_bootstrap=200,
    )
    energies = np.linspace(11800, 12000, results['residuals'].shape[1])
    tie_table = peci_tie_table(results, ref_names)
    rng = np.random.default_rng(0)
    A = rng.random((len(energies), len(ref_names)))
    clusterings = {
        metric: cluster_reference_spectra(A, ref_names, np.random.default_rng(1),
                                          metric=metric, resample_count=20, verbose=False)
        for metric in ('correlation', 'cosine')
    }

    def figures(**kwargs):
        return plot_best_peci_subset_bootstrap_summaries(
            results, ref_names, energies, b, n_bootstrap=200, spectrum_name='SYNTH.e',
            tie_table=tie_table, max_subsets_per_size=0, **kwargs,
        )

    alone = figures()
    assert len(alone) == 1 and len(alone[0].axes) == 1, 'no clusterings, no tree figure'

    tie_figure, tree_figure = figures(clusterings=clusterings)
    assert len(tie_figure.axes) == 1, 'the tie structure keeps a figure to itself'
    assert len(tree_figure.axes) == len(clusterings)
    assert [ax.get_title() for ax in tree_figure.axes] == \
        ['correlation distance', 'cosine distance']
    assert all(not plt.fignum_exists(figure.number) for figure in (tie_figure, tree_figure))

    # the same width, so a fraction is the same distance in both
    assert tie_figure.get_size_inches()[0] == tree_figure.get_size_inches()[0]
    tie_box = tie_figure.axes[0].get_position()
    tree_boxes = [ax.get_position() for ax in tree_figure.axes]
    assert abs(tie_box.x0 - min(box.x0 for box in tree_boxes)) < 1e-6, 'left edges should line up'
    assert abs(tie_box.x1 - max(box.x1 for box in tree_boxes)) < 1e-6, 'right edges too'


def test_plot_best_peci_returns_closed_figures_and_respects_the_cap():
    """Figures come back for the caller to display, and the cap is what bounds how many."""
    # offsets well inside the noise at both subset sizes, so there is a tie set with more
    # than one member for the cap to actually bite on
    results, ref_names, b = _paired_pe_results(
        [0.0, 0.002, 0.004, 0.0, 0.002, 0.004], noise_scale=0.05, n_bootstrap=200,
        subset_sizes=[1, 1, 1, 2, 2, 2],
    )
    energies = np.linspace(11800, 12000, results['residuals'].shape[1])
    tie_table = peci_tie_table(results, ref_names)
    tied_per_size = peci_tie_counts(tie_table)['tied']
    assert (tied_per_size > 1).any(), 'the cap is only exercised if something ties'

    figure_counts = {}
    for max_subsets_per_size in (1, 2, None):
        figures = plot_best_peci_subset_bootstrap_summaries(
            results, ref_names, energies, b,
            n_bootstrap=results['bootstrap_pes'].shape[1],
            spectrum_name='SYNTH.e', tie_table=tie_table,
            max_subsets_per_size=max_subsets_per_size,
        )
        # one tie-structure figure per subset size, then four per summary -- five when
        # clusterings are given, which they are not here
        drawn = tied_per_size if max_subsets_per_size is None \
            else tied_per_size.clip(upper=max_subsets_per_size)
        assert len(figures) == len(tied_per_size) + 4 * drawn.sum()
        assert all(not plt.fignum_exists(figure.number) for figure in figures), \
            'every figure should be closed before it is returned'
        figure_counts[max_subsets_per_size] = len(figures)

    assert figure_counts[1] <= figure_counts[2] <= figure_counts[None]

_selection_test_fns = [
    test_median_ci_estimators_bracket_the_median,
    test_order_statistic_ci_survives_a_constant_sample,
    test_peci_tie_table_has_one_row_per_combination,
    test_mean_rank_anchor_ignores_how_hard_an_iteration_was,
    test_mean_rank_handles_exact_ties,
    test_peci_anchor_is_the_lowest_mean_rank,
    test_holdout_regimes_split_on_the_window,
    test_regime_variance_explained_recognizes_a_two_level_signal,
    test_peci_tie_table_regime_columns_are_opt_in,
    test_peci_best_subset_ties_itself,
    test_peci_excludes_a_clearly_worse_subset,
    test_peci_paired_comparison_is_sharper_than_overlapping_intervals,
    test_tie_rules_differ_in_what_more_iterations_buy,
    test_tie_rule_sensitivity_reports_both_rules_at_every_size,
    test_tied_subset_reference_shares_counts_the_tie_set,
    test_the_trees_are_their_own_figure_and_line_up_with_the_panel,
    test_plot_best_peci_returns_closed_figures_and_respects_the_cap,
    test_the_window_finds_both_places_the_references_part,
    test_the_window_does_not_depend_on_the_unknown,
    test_contiguous_runs_finds_the_runs,
]
for _fn in _selection_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_selection_test_fns)} tests passed')

<a id="pipeline-fit-files"></a>

## 8. Fit files, and the study cache

A search takes about 24 seconds and produces 24 MB of draws. `write_fit_results` puts
everything the figures draw from into one Parquet file and `read_fit_results` rebuilds exactly
the dict a live fit returns, so a figure can be redrawn without repeating the search. The
table is one row per reference combination, which is the axis most questions about a fit are
asked along, and the holdout masks ride along as a column of their own.

In [ ]:
# Writing a fit to one file, and reading it back into what the plotting functions expect.
#
# The figures in this notebook are all drawn from a live fit, so redrawing anything means
# re-running the 2,324-combination bootstrap. These two functions end that: the fit goes
# into a Parquet file, and the reader hands back exactly the dict `do_fits_and_plot_summaries`
# returns, so a file-driven session is indistinguishable from a fresh one.
#
# Parquet rather than a bag of arrays because the interesting axis of this data *is* tabular
# -- one row per reference combination -- and a table lets the ranking be queried from
# pandas, DuckDB, Polars or R without unpacking anything. The bootstrap draws ride along as
# list columns, which Parquet stores variable-length, so the NaN padding a live fit carries
# is not written at all.
#
# Needs pyarrow, which is declared in requirements.txt and so comes with `pip install -e .`
# -- declared there rather than only here so these two functions can move into the mrfitty
# package without a dependency change.
import base64
import datetime
import json

import pyarrow as pa
import pyarrow.parquet as pq

# The version is stamped on every file and checked on the way back in. There is one version,
# and the reader refuses anything else rather than guessing: a file it does not recognize is
# one it cannot reconstruct, not one missing a label. If the layout ever has to change after
# this is released, that is the point at which reading older files becomes worth the code --
# until then there are no older files to read.
RESULTS_SCHEMA_VERSION = 1
RESULTS_METADATA_KEY = b'mrfitty_fit_results'

# Read by plot_reference_dendrogram and worth keeping; the two large clustering arrays
# (chance_merge_heights, cophenetic_distances) are read only by plot_cluster_metric_comparison,
# which is outside what this file is meant to redraw, so they are left out and restored as
# None rather than silently missing.
_CLUSTERING_SCALARS = ('metric', 'method', 'percentile', 'resample_count', 'surrogate',
                       'cutoff_distance', 'cophenetic_correlation', 'n_clusters',
                       'first_permutation_digest')
_CLUSTERING_ARRAYS = ('Z', 'labels', 'distances')
_CLUSTERING_OMITTED = ('chance_merge_heights', 'cophenetic_distances')


def _encode_array(array):
    """A numpy array as JSON-safe base64 plus the shape and dtype needed to rebuild it."""
    array = np.ascontiguousarray(array)
    return {'b64': base64.b64encode(array.tobytes()).decode('ascii'),
            'dtype': array.dtype.str,          # includes byte order, so a big-endian reader is safe
            'shape': list(array.shape)}


def _decode_array(encoded):
    return np.frombuffer(base64.b64decode(encoded['b64']),
                         dtype=np.dtype(encoded['dtype'])).reshape(encoded['shape'])



def write_fit_results(path, fit_summary, block_length, n_holdout_blocks, elapsed_time, seed):
    """Write everything the fit summaries draw from into one Parquet file.

    Parameters
    ----------
    path : str
        Destination `.parquet` file.
    fit_summary : dict
        What `do_fits_and_plot_summaries` returns: 'results', 'holdout_masks', 'A', 'b',
        'ref_names', 'valid_energies', 'sample_energies', 'spectrum_name' and 'clusterings'.
        All of them are required except 'clusterings'.
    block_length, n_holdout_blocks, elapsed_time, seed
        Provenance the fit produced but does not carry in `results`. `block_length` matters
        most: under the default `block_length='auto'` it is tuned from the residuals, so it
        is a result of the fit rather than a setting, and cannot be recovered later.

    The file also records the mrfitty version that wrote it and the time it was written,
    so a file found later says which code produced it and when -- neither is recoverable
    from the fit itself, and the version is what tells a reader whether the numbers in the
    file predate a change in how they are computed.

    The bootstrap draws are stored as float32. They are 75 MB of the 83 MB at float64, and
    nothing drawn from them -- medians, percentiles, violins -- resolves anywhere near
    float32's precision. Everything else keeps full precision.

    The holdout masks go in the table as a `holdout_mask` column of booleans, one row per
    bootstrap iteration -- eight kilobytes for a thousand iterations over two hundred
    energies, because Parquet already stores BOOLEAN as bits and zstd takes it from there.
    Nothing has to pack them by hand. They are what `holdout_regimes` needs, and they cannot be recovered
    from anything else in the file: the selector is seeded, but reproducing the draws would
    mean knowing which selector ran and re-running it, which the file does not record.

    Because that column is per iteration and the rest of the table is per combination, the
    table is as long as whichever there are more of and every column is null past its own
    end. A query over combinations should say `WHERE M IS NOT NULL`; the reader does the
    equivalent using the counts in the metadata.
    """
    results = fit_summary['results']
    n_combinations = len(results['M'])
    n_bootstrap = results['bootstrap_pes'].shape[1]
    holdout_masks = np.asarray(fit_summary['holdout_masks'], dtype=bool)

    # The holdout masks are per *iteration*, not per combination, so they are a column of a
    # different length. Parquet columns in one table share a length, so the table is as long
    # as the longer of the two and each column is padded with nulls past its own end. In a
    # real search there are far more combinations than iterations -- 2,324 against 1,000 --
    # so it is the mask column that is padded and the combination rows are untouched. Only a
    # search over a handful of references inverts that.
    #
    # The masks go in as list(bool) rather than anything hand-packed: Parquet stores BOOLEAN
    # as bits already, so the format does the packing that numpy has no dtype for.
    n_rows = max(n_combinations, n_bootstrap)

    # One row per combination, each list column holding only that row's own M values: the
    # NaN padding to max_M that the in-memory arrays carry is an artifact of rectangular
    # numpy, and the reader puts it back.
    rows = {
        'M': [], 'ref_indices': [], 'coef': [], 'bootstrap_coefs': [],
        'bootstrap_pes': [], 'residuals': [], 'acf_values': [], 'median_pe': [],
    }
    for i in range(n_combinations):
        m = int(results['M'][i])
        rows['M'].append(m)
        rows['ref_indices'].append(results['ref_indices'][i, :m].astype(np.int16).tolist())
        rows['coef'].append(results['coef'][i, :m].tolist())
        rows['bootstrap_coefs'].append(
            results['bootstrap_coefs'][i, :, :m].astype(np.float32).reshape(-1).tolist())
        rows['bootstrap_pes'].append(results['bootstrap_pes'][i].astype(np.float32).tolist())
        rows['residuals'].append(results['residuals'][i].tolist())
        rows['acf_values'].append(results['acf_values'][i].tolist())
        # stored so the ranking every summary figure is built on can be read off the table
        # without unnesting a thousand draws per row
        rows['median_pe'].append(float(np.median(results['bootstrap_pes'][i])))

    for column in rows.values():
        column.extend([None] * (n_rows - n_combinations))
    rows['holdout_mask'] = ([[bool(value) for value in mask] for mask in holdout_masks]
                            + [None] * (n_rows - n_bootstrap))

    table = pa.table(rows, schema=pa.schema([
        ('M', pa.int8()),
        ('ref_indices', pa.list_(pa.int16())),
        ('coef', pa.list_(pa.float64())),
        ('bootstrap_coefs', pa.list_(pa.float32())),
        ('bootstrap_pes', pa.list_(pa.float32())),
        ('residuals', pa.list_(pa.float64())),
        ('acf_values', pa.list_(pa.float64())),
        ('median_pe', pa.float64()),
        ('holdout_mask', pa.list_(pa.bool_())),
    ]))

    # Everything that is not per-combination goes in the file's key-value metadata as one
    # JSON document -- the same place GeoParquet keeps its spec -- so the table itself stays
    # a clean per-combination table for anyone querying it.
    clusterings = {}
    for metric, clustering in (fit_summary.get('clusterings') or {}).items():
        clusterings[metric] = {
            'ref_names': list(clustering['ref_names']),
            **{key: clustering[key] for key in _CLUSTERING_SCALARS},
            **{key: _encode_array(clustering[key]) for key in _CLUSTERING_ARRAYS},
        }

    metadata = {
        'schema_version': RESULTS_SCHEMA_VERSION,
        'mrfitty_version': mrfitty.__version__,
        # UTC with an explicit offset, so the timestamp means the same thing to a reader in
        # another timezone and sorts lexicographically
        'written_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'spectrum_name': fit_summary['spectrum_name'],
        'ref_names': list(fit_summary['ref_names']),
        'n_bootstrap': int(n_bootstrap),
        'n_combinations': int(n_combinations),
        'max_M': int(results['M'].max()),
        'block_length': int(block_length),
        'n_holdout_blocks': int(n_holdout_blocks),
        'elapsed_time': float(elapsed_time),
        'seed': int(seed),
        'omitted': list(_CLUSTERING_OMITTED),
        # the sample's own measured grid, wider than the fitted window; two kilobytes, and
        # storing it whole means a caller can hand it straight to plot_ref_subsets_summary
        'sample_energies': _encode_array(fit_summary['sample_energies']),
        'valid_energies': _encode_array(fit_summary['valid_energies']),
        'b': _encode_array(fit_summary['b']),
        'A': _encode_array(fit_summary['A']),
        'lags': _encode_array(results['lags']),
        'clusterings': clusterings,
    }
    table = table.replace_schema_metadata({RESULTS_METADATA_KEY: json.dumps(metadata).encode()})
    pq.write_table(table, path, compression='zstd')
    return path


def read_fit_results(path):
    """Rebuild the dict a live fit would have returned, from a file written by write_fit_results.

    The returned dict has the same keys `do_fits_and_plot_summaries` returns, so every
    plotting function takes it unchanged, plus the provenance the writer recorded.

    A file stamped with any other schema version is refused rather than read as far as it
    goes, because there is only one layout and a file that does not carry it is not one this
    reader can rebuild a fit from.
    """
    table = pq.read_table(path)
    raw = table.schema.metadata.get(RESULTS_METADATA_KEY)
    if raw is None:
        raise ValueError(f'{path} carries no {RESULTS_METADATA_KEY.decode()} metadata; '
                         'it was not written by write_fit_results')
    metadata = json.loads(raw)
    if metadata['schema_version'] != RESULTS_SCHEMA_VERSION:
        raise ValueError(f'{path} is schema version {metadata["schema_version"]}, '
                         f'this reader understands {RESULTS_SCHEMA_VERSION}')

    columns = {name: table.column(name).to_pylist() for name in table.column_names}
    # the counts come from the metadata rather than from the table's length, which is the
    # longer of the two and so says nothing about either on its own
    n_combinations = metadata['n_combinations']
    # the per-combination columns run to the table's length and are null past the last
    # combination, so they are cut once here and nothing below has to know about the padding
    for name in ('M', 'ref_indices', 'coef', 'bootstrap_coefs', 'bootstrap_pes',
                 'residuals', 'acf_values', 'median_pe'):
        columns[name] = columns[name][:n_combinations]
    max_M = metadata['max_M']
    n_bootstrap = metadata['n_bootstrap']
    n_energies = len(columns['residuals'][0])
    n_lags = len(columns['acf_values'][0])

    # Rebuild the rectangular, NaN-padded arrays a live fit produces, so nothing downstream
    # can tell the difference -- including the `[:, :m]` slicing the summary plots do.
    results = {
        'M': np.array(columns['M'], dtype=int),
        'ref_indices': np.full((n_combinations, max_M), np.nan),
        'coef': np.full((n_combinations, max_M), np.nan),
        'bootstrap_coefs': np.full((n_combinations, n_bootstrap, max_M), np.nan, dtype=np.float32),
        'bootstrap_pes': np.zeros((n_combinations, n_bootstrap), dtype=np.float32),
        'fitted': np.zeros((n_combinations, n_energies)),
        'residuals': np.zeros((n_combinations, n_energies)),
        'lags': _decode_array(metadata['lags']),
        'acf_values': np.zeros((n_combinations, n_lags)),
    }
    b = _decode_array(metadata['b'])
    for i in range(n_combinations):
        m = int(columns['M'][i])
        results['ref_indices'][i, :m] = columns['ref_indices'][i]
        results['coef'][i, :m] = columns['coef'][i]
        results['bootstrap_coefs'][i, :, :m] = np.asarray(
            columns['bootstrap_coefs'][i], dtype=np.float32).reshape(n_bootstrap, m)
        results['bootstrap_pes'][i] = columns['bootstrap_pes'][i]
        results['residuals'][i] = columns['residuals'][i]
        results['acf_values'][i] = columns['acf_values'][i]
        # fit_nnls defines residuals = fitted - b, so the fitted spectrum is recoverable
        # exactly and is not worth the 3.7 MB it would take to store
        results['fitted'][i] = b + results['residuals'][i]

    clusterings = {}
    for metric, stored in metadata['clusterings'].items():
        clusterings[metric] = {
            'ref_names': list(stored['ref_names']),
            **{key: stored[key] for key in _CLUSTERING_SCALARS},
            **{key: _decode_array(stored[key]) for key in _CLUSTERING_ARRAYS},
            # present and empty rather than absent, so a caller that wants them sees why
            **{key: None for key in metadata['omitted']},
        }

    return {
        'results': results,
        # the column runs to the table's length, so it is cut back to the iterations
        'holdout_masks': np.array(columns['holdout_mask'][:n_bootstrap], dtype=bool),
        'A': _decode_array(metadata['A']),
        'b': b,
        'ref_names': list(metadata['ref_names']),
        'valid_energies': _decode_array(metadata['valid_energies']),
        'spectrum_name': metadata['spectrum_name'],
        'clusterings': clusterings,
        'sample_energies': _decode_array(metadata['sample_energies']),
        'block_length': metadata['block_length'],
        'n_holdout_blocks': metadata['n_holdout_blocks'],
        'elapsed_time': metadata['elapsed_time'],
        'seed': metadata['seed'],
        'schema_version': metadata['schema_version'],
        'mrfitty_version': metadata['mrfitty_version'],
        'written_at': metadata['written_at'],
    }

In [ ]:
# Running a study once and keeping the answer.
#
# The studies below are built on synthetic replicates, each of which is a full combination
# search, so several of them run for tens of minutes. Their *results* are a few kilobytes: a
# summary table per study. Those are cached to Parquet so that re-opening the notebook, or
# re-running one section, does not mean re-running all of them.
#
# The cache is not committed. A fresh clone recomputes everything on its first run, which is
# why every Findings section below states its numbers in the prose rather than pointing at a
# table that may not have been drawn yet.

STUDY_METADATA_KEY = b'mrfitty_study_result'


def cached_study(name, compute, recompute=False, **scale):
    """Return a study's summary table, computing it only if it is not already on disk.

    Parameters
    ----------
    name      : str -- the file stem under study_results/
    compute   : callable () -> pandas.DataFrame
    recompute : bool -- ignore any cached copy and redraw
    **scale   : the parameters the result depends on (replicate counts, grids). Stored
                alongside, and compared on read: a cached file drawn at a different scale is
                reported rather than silently returned as though it answered the same
                question.

    Returns
    -------
    pandas.DataFrame
    """
    path = os.path.join(STUDY_RESULTS_DIR, f'{name}.parquet')

    if not recompute and os.path.exists(path):
        table = pq.read_table(path)
        stored = json.loads(table.schema.metadata[STUDY_METADATA_KEY])
        print(f'{name}: from cache, computed {stored["written_at"]} '
              f'by mrfitty {stored["mrfitty_version"]}')
        if stored['scale'] != scale:
            print(f'  NOTE: cached at scale {stored["scale"]}, asked for {scale}. '
                  f'Pass recompute=True to redraw.')
        return table.to_pandas()

    started = time.perf_counter()
    result = compute()
    elapsed = time.perf_counter() - started

    os.makedirs(STUDY_RESULTS_DIR, exist_ok=True)
    table = pa.Table.from_pandas(result, preserve_index=False)
    table = table.replace_schema_metadata({
        **(table.schema.metadata or {}),
        STUDY_METADATA_KEY: json.dumps({
            'study': name,
            'scale': scale,
            'seconds': round(elapsed, 1),
            'mrfitty_version': mrfitty.__version__,
            'written_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        }).encode(),
    })
    pq.write_table(table, path, compression='zstd')
    print(f'{name}: computed in {elapsed:.0f}s and cached to {os.path.relpath(path)}')
    return result

In [ ]:
# ---------------------------------------------------------------------------
# Tests for write_fit_results / read_fit_results
#
# Built on a miniature fit -- 3 references, 5 combinations, 20 bootstrap iterations --
# so the file is written and read in milliseconds, but with the same shapes, the same
# NaN padding and real clusterings, which is what the round trip has to preserve.
# ---------------------------------------------------------------------------
import tempfile

def _synthetic_fit_summary(seed=0, n_energies=30, n_bootstrap=20):
    """A miniature fit: 3 references, combinations of sizes 1 and 2, NaN padding and all."""
    rng = np.random.default_rng(seed)
    ref_names = ['a.e', 'b.e', 'c.e']
    combos = [(1, [0]), (1, [1]), (1, [2]), (2, [0, 1]), (2, [1, 2])]
    max_M = max(m for m, _ in combos)
    n = len(combos)
    b = rng.normal(size=n_energies)
    results = {
        'M': np.array([m for m, _ in combos], dtype=int),
        'ref_indices': np.full((n, max_M), np.nan),
        'coef': np.full((n, max_M), np.nan),
        'bootstrap_coefs': np.full((n, n_bootstrap, max_M), np.nan),
        'bootstrap_pes': rng.random((n, n_bootstrap)),
        'fitted': np.zeros((n, n_energies)),
        'residuals': rng.normal(size=(n, n_energies)) * 0.01,
        'lags': np.arange(6),
        'acf_values': rng.random((n, 6)),
    }
    for i, (m, indices) in enumerate(combos):
        results['ref_indices'][i, :m] = indices
        results['coef'][i, :m] = rng.random(m)
        results['bootstrap_coefs'][i, :, :m] = rng.random((n_bootstrap, m))
        results['fitted'][i] = b + results['residuals'][i]

    # holdout masks with the shape and the roughly one-third holdout of a real draw, so
    # the regime split can be exercised on the reloaded copy
    holdout_masks = rng.random((n_bootstrap, n_energies)) < 0.35

    A = rng.random((n_energies, len(ref_names)))
    clusterings = {
        metric: cluster_reference_spectra(A, ref_names, np.random.default_rng(1),
                                          metric=metric, resample_count=20, verbose=False)
        for metric in ('correlation', 'cosine')
    }
    return {
        'results': results, 'holdout_masks': holdout_masks,
        'A': A, 'b': b, 'ref_names': ref_names,
        'valid_energies': np.linspace(11800, 12000, n_energies),
        'spectrum_name': 'SYNTH.e', 'clusterings': clusterings,
        'sample_energies': np.linspace(11790, 12010, n_energies + 7),
    }


def _round_trip(fit_summary, **kwargs):
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42, **kwargs)
        return read_fit_results(path), os.path.getsize(path)


def test_holdout_masks_survive_exactly():
    """The masks are what holdout_regimes needs, and a bit is either right or it is not."""
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)

    assert loaded['holdout_masks'].dtype == bool
    assert loaded['holdout_masks'].shape == fit_summary['holdout_masks'].shape
    np.testing.assert_array_equal(loaded['holdout_masks'], fit_summary['holdout_masks'])


def test_regimes_are_recoverable_from_the_file():
    """The reason the masks are stored: a reloaded fit can still be split by regime."""
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)

    window = diagnostic_window(fit_summary['valid_energies'], fit_summary['A'])
    original = holdout_regimes(fit_summary['holdout_masks'], window)
    reloaded = holdout_regimes(loaded['holdout_masks'], window)

    np.testing.assert_array_equal(reloaded['held_out'], original['held_out'])
    assert reloaded['n_held_out'] == original['n_held_out']


def test_holdout_masks_are_a_column_of_the_table():
    """They belong in the table, not in the file's metadata blob."""
    import pandas as pd
    fit_summary = _synthetic_fit_summary()
    n_bootstrap = fit_summary['results']['bootstrap_pes'].shape[1]
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42)
        frame = pd.read_parquet(path, columns=['holdout_mask'])
        stored = json.loads(pq.read_table(path).schema.metadata[RESULTS_METADATA_KEY])

    assert 'holdout_masks' not in stored, 'the masks should not be in the metadata'
    masks = frame['holdout_mask'].dropna()
    assert len(masks) == n_bootstrap, 'one row per bootstrap iteration, and no more'
    np.testing.assert_array_equal(np.array(masks.tolist(), dtype=bool),
                                  fit_summary['holdout_masks'])


def test_the_table_is_as_long_as_whichever_there_are_more_of():
    """Per-combination and per-iteration columns share a table, so both are null-padded."""
    import pandas as pd
    fit_summary = _synthetic_fit_summary()
    n_combinations = len(fit_summary['results']['M'])
    n_bootstrap = fit_summary['results']['bootstrap_pes'].shape[1]
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42)
        frame = pd.read_parquet(path)
        # and the reader is not fooled by the padding
        loaded = read_fit_results(path)

    assert len(frame) == max(n_combinations, n_bootstrap)
    assert frame['M'].notna().sum() == n_combinations
    assert frame['holdout_mask'].notna().sum() == n_bootstrap
    assert len(loaded['results']['M']) == n_combinations
    assert loaded['holdout_masks'].shape[0] == n_bootstrap


def test_results_round_trip():
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)
    original = fit_summary['results']

    for key in original:
        assert loaded['results'][key].shape == original[key].shape, key

    np.testing.assert_array_equal(loaded['results']['M'], original['M'])
    np.testing.assert_array_equal(loaded['results']['lags'], original['lags'])
    for key in ('ref_indices', 'coef', 'residuals', 'acf_values'):
        np.testing.assert_allclose(loaded['results'][key], original[key], atol=0, rtol=0,
                                   err_msg=f'{key} should survive exactly')
    # the draws are stored as float32, which is the one deliberate loss
    for key in ('bootstrap_coefs', 'bootstrap_pes'):
        np.testing.assert_allclose(loaded['results'][key], original[key], rtol=1e-6,
                                   err_msg=f'{key} should survive to float32 precision')


def test_padding_and_fitted_are_reconstructed():
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)
    original = fit_summary['results']

    # the file stores only each row's own M values; the padding has to come back
    for i, m in enumerate(original['M']):
        assert np.isnan(loaded['results']['ref_indices'][i, m:]).all()
        assert np.isnan(loaded['results']['coef'][i, m:]).all()
        assert np.isnan(loaded['results']['bootstrap_coefs'][i, :, m:]).all()
        assert np.isfinite(loaded['results']['ref_indices'][i, :m]).all()

    # fitted is not stored at all -- it is b + residuals, exactly
    np.testing.assert_allclose(loaded['results']['fitted'], original['fitted'], atol=1e-12)


def test_inputs_and_provenance_survive():
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)

    np.testing.assert_array_equal(loaded['A'], fit_summary['A'])
    np.testing.assert_array_equal(loaded['b'], fit_summary['b'])
    np.testing.assert_array_equal(loaded['valid_energies'], fit_summary['valid_energies'])
    assert loaded['ref_names'] == fit_summary['ref_names']
    assert loaded['spectrum_name'] == 'SYNTH.e'
    np.testing.assert_array_equal(loaded['sample_energies'], fit_summary['sample_energies'])
    assert (loaded['block_length'], loaded['n_holdout_blocks']) == (10, 6)
    assert loaded['seed'] == 42 and loaded['elapsed_time'] == 1.5
    assert loaded['schema_version'] == RESULTS_SCHEMA_VERSION


def test_the_file_says_what_wrote_it_and_when():
    before = datetime.datetime.now(datetime.timezone.utc)
    loaded, _ = _round_trip(_synthetic_fit_summary())

    assert loaded['mrfitty_version'] == mrfitty.__version__
    # parsed rather than compared as text, so a malformed timestamp fails here
    written_at = datetime.datetime.fromisoformat(loaded['written_at'])
    assert written_at.tzinfo is not None, 'the timestamp should carry its UTC offset'
    assert before <= written_at <= datetime.datetime.now(datetime.timezone.utc)


def _rewrite_metadata(path, **changes):
    """Copy a results file with its metadata document edited; None deletes a key.

    How a file this reader should refuse is manufactured without keeping one around: the
    table is untouched, so what the reader sees differs from a good file only in the
    metadata.
    """
    table = pq.read_table(path)
    metadata = json.loads(table.schema.metadata[RESULTS_METADATA_KEY])
    for key, value in changes.items():
        if value is None:
            metadata.pop(key, None)
        else:
            metadata[key] = value
    table = table.replace_schema_metadata({RESULTS_METADATA_KEY: json.dumps(metadata).encode()})
    older_path = path.replace('.parquet', '_rewritten.parquet')
    pq.write_table(table, older_path, compression='zstd')
    return older_path


def test_a_fit_without_holdout_masks_is_refused():
    """The masks are required, so their absence is an error and not a quietly emptier file."""
    fit_summary = _synthetic_fit_summary()
    del fit_summary['holdout_masks']
    try:
        _round_trip(fit_summary)
    except KeyError as error:
        assert 'holdout_masks' in str(error)
    else:
        raise AssertionError('a fit summary with no holdout masks should not write')


def test_an_unreadable_schema_version_is_still_refused():
    fit_summary = _synthetic_fit_summary()
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42)
        future_path = _rewrite_metadata(path, schema_version=RESULTS_SCHEMA_VERSION + 1)
        try:
            read_fit_results(future_path)
        except ValueError as error:
            assert 'schema version' in str(error)
        else:
            raise AssertionError('a version this reader cannot reconstruct should be refused')


def test_clusterings_survive_and_still_draw():
    fit_summary = _synthetic_fit_summary()
    loaded, _ = _round_trip(fit_summary)

    for metric, original in fit_summary['clusterings'].items():
        restored = loaded['clusterings'][metric]
        np.testing.assert_array_equal(restored['Z'], original['Z'])
        np.testing.assert_array_equal(restored['labels'], original['labels'])
        np.testing.assert_allclose(restored['distances'], original['distances'])
        assert restored['cutoff_distance'] == original['cutoff_distance']
        assert restored['surrogate'] == original['surrogate']
        # the two big arrays are deliberately not written; say so rather than omit the key
        for key in ('chance_merge_heights', 'cophenetic_distances'):
            assert restored[key] is None

    # the dendrogram reads none of the omitted keys, so it draws from the reloaded dict
    fig, ax = plt.subplots()
    plot_highlighted_reference_dendrogram(loaded['clusterings']['correlation'],
                                          highlight=['a.e'], ax=ax)
    plt.close(fig)


def test_the_table_is_queryable_without_unnesting():
    # the reason for Parquet over a bag of arrays: the ranking every summary figure is built
    # on has to be readable from the table itself
    import pandas as pd
    fit_summary = _synthetic_fit_summary()
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'fit.parquet')
        write_fit_results(path, fit_summary, block_length=10, n_holdout_blocks=6,
                          elapsed_time=1.5, seed=42)
        # the combination rows are the ones with an M; in a real search there are more
        # combinations than iterations and nothing is padded, but the fixture here is small
        frame = pd.read_parquet(path, columns=['M', 'median_pe']).dropna(subset=['M'])

    live_medians = np.median(fit_summary['results']['bootstrap_pes'], axis=1)
    np.testing.assert_allclose(frame['median_pe'].to_numpy(), live_medians, rtol=1e-6)
    np.testing.assert_array_equal(np.argsort(frame['median_pe'].to_numpy()),
                                  np.argsort(live_medians))
    np.testing.assert_array_equal(frame['M'].to_numpy(), fit_summary['results']['M'])


def test_a_foreign_parquet_is_rejected():
    import pandas as pd
    with tempfile.TemporaryDirectory() as directory:
        path = os.path.join(directory, 'other.parquet')
        pd.DataFrame({'x': [1, 2, 3]}).to_parquet(path)
        try:
            read_fit_results(path)
        except ValueError as error:
            assert 'mrfitty_fit_results' in str(error)
        else:
            raise AssertionError('a file this writer did not produce should be refused')


_results_io_test_fns = [
    test_results_round_trip,
    test_holdout_masks_survive_exactly,
    test_regimes_are_recoverable_from_the_file,
    test_a_fit_without_holdout_masks_is_refused,
    test_holdout_masks_are_a_column_of_the_table,
    test_the_table_is_as_long_as_whichever_there_are_more_of,
    test_padding_and_fitted_are_reconstructed,
    test_inputs_and_provenance_survive,
    test_the_file_says_what_wrote_it_and_when,
    test_an_unreadable_schema_version_is_still_refused,
    test_clusterings_survive_and_still_draw,
    test_the_table_is_queryable_without_unnesting,
    test_a_foreign_parquet_is_rejected,
]

for _fn in _results_io_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_results_io_test_fns)} tests passed')

In [ ]:
# ---------------------------------------------------------------------------
# Tests for the study cache.
# ---------------------------------------------------------------------------

def _counting_study():
    """A compute function that records how often it was actually run."""
    calls = []

    def compute():
        calls.append(1)
        return pd.DataFrame({'answer': [len(calls)], 'label': ['x']})

    return compute, calls


def test_a_study_is_computed_once_and_read_back_after():
    global STUDY_RESULTS_DIR
    compute, calls = _counting_study()
    original_dir = STUDY_RESULTS_DIR
    with tempfile.TemporaryDirectory() as directory:
        STUDY_RESULTS_DIR = directory
        try:
            first = cached_study('demo', compute, n_replicates=8)
            second = cached_study('demo', compute, n_replicates=8)
        finally:
            STUDY_RESULTS_DIR = original_dir

    assert len(calls) == 1, 'the second call should have read the cache'
    pd.testing.assert_frame_equal(first, second)
    assert first['answer'].iloc[0] == 1


def test_recompute_ignores_the_cache_and_the_scale_is_recorded():
    global STUDY_RESULTS_DIR
    compute, calls = _counting_study()
    original_dir = STUDY_RESULTS_DIR
    with tempfile.TemporaryDirectory() as directory:
        STUDY_RESULTS_DIR = directory
        try:
            cached_study('demo', compute, n_replicates=8)
            again = cached_study('demo', compute, recompute=True, n_replicates=8)
            stored = json.loads(
                pq.read_table(os.path.join(directory, 'demo.parquet'))
                  .schema.metadata[STUDY_METADATA_KEY]
            )
            # a cached file drawn at another scale is reported rather than passed off as an
            # answer to the question that was asked
            cached_study('demo', compute, n_replicates=64)
        finally:
            STUDY_RESULTS_DIR = original_dir

    assert len(calls) == 2, 'recompute should redraw; the mismatched scale should not'
    assert again['answer'].iloc[0] == 2
    assert stored['scale'] == {'n_replicates': 8}
    assert stored['mrfitty_version'] == mrfitty.__version__


_cache_test_fns = [
    test_a_study_is_computed_once_and_read_back_after,
    test_recompute_ignores_the_cache_and_the_scale_is_recorded,
]
for _fn in _cache_test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_cache_test_fns)} tests passed')

<a id="pipeline-all-five"></a>

## 9. Running all five

One search per unknown, cached to disk. About two minutes from cold, seconds thereafter.

In [ ]:
# The two places the references part company, in the order diagnostic_window reports them.
# Naming them is a statement about this data rather than about the method: on an edge
# other than arsenic K there would be a different number of runs meaning different things.
DIAGNOSTIC_RUN_LABELS = ('absorption, reduced As (~11870 eV)',
                         'absorption, arsenate (~11875 eV)')


# One search per unknown, cached. Fitting is the pipeline; drawing it is not, so this returns
# the fit summaries and the figures are made from them afterwards.


def fit_unknown(spectrum, reference_spectra, n_bootstrap=1000, M=(1, 2, 3),
                block_length='auto', resample_block_length=None, seed=42,
                interpolant=make_cubic_spline_interpolant, use_cache=True, recompute=False):
    """Search every reference combination against one unknown, reading a cached fit if there is one.

    The cache is keyed on the file name alone, so anything that changes the fit -- a different
    block length, a different interpolant -- needs `recompute=True` or a cleared cache. That is
    a deliberate simplicity: the studies below vary those parameters, and they do it on
    synthetic spectra rather than through this function.

    Returns the dict `write_fit_results` and `read_fit_results` round-trip, plus the reference
    clusterings, so downstream figures take the same input whether the fit was just made or
    read off disk.
    """
    cache_path = os.path.join(FIT_CACHE_DIR, f'{spectrum.file_name}.parquet')
    if use_cache and not recompute and os.path.exists(cache_path):
        fit_summary = read_fit_results(cache_path)
        print(f'{spectrum.file_name}: read from {os.path.relpath(cache_path)}')
        return fit_summary

    valid_energies, A, b, *limiters = interpolate_references_at_sample_energies(
        reference_spectra=reference_spectra, sample_spectrum=spectrum,
        make_interpolant=interpolant, verbose=False,
    )

    started = time.perf_counter()
    results, holdout_masks, sampled_starts, resample_length, n_holdout_blocks = \
        do_ref_subsets_moving_block_holdout_bootstrap(
            b, A, M=list(M), rng=np.random.default_rng(seed=seed),
            select_holdout_blocks_fn=select_holdout_blocks,
            n_bootstrap=n_bootstrap, block_length=block_length,
            resample_block_length=resample_block_length,
        )
    elapsed = time.perf_counter() - started

    # Cluster the pool once, here, rather than inside the plotting functions: they draw the
    # same two trees on every figure, and this way one place fixes the seed and the parameters.
    clusterings = {
        metric: cluster_reference_spectra(A, [r.file_name for r in reference_spectra],
                                          np.random.default_rng(seed), metric=metric,
                                          verbose=False)
        for metric in ('correlation', 'cosine')
    }

    fit_summary = {
        'results': results,
        'holdout_masks': holdout_masks,
        'A': A,
        'b': b,
        'ref_names': [r.file_name for r in reference_spectra],
        'valid_energies': valid_energies,
        'sample_energies': spectrum.data_df.index.values,
        'spectrum_name': spectrum.file_name,
        'clusterings': clusterings,
        'block_length': resample_length,
        'n_holdout_blocks': n_holdout_blocks,
        'elapsed_time': elapsed,
        'seed': seed,
    }

    if use_cache:
        os.makedirs(FIT_CACHE_DIR, exist_ok=True)
        write_fit_results(cache_path, fit_summary, block_length=resample_length,
                          n_holdout_blocks=n_holdout_blocks, elapsed_time=elapsed, seed=seed)
        print(f'{spectrum.file_name}: {len(results["M"])} combinations in {elapsed:.0f}s, '
              f'cached to {os.path.relpath(cache_path)}')
    return fit_summary


def fit_all_unknowns(spectra, reference_spectra, **kwargs):
    """{file name: fit summary} for each unknown, in the order given."""
    return {s.file_name: fit_unknown(s, reference_spectra, **kwargs) for s in spectra}


def summarize_unknowns(fit_summaries, window_of):
    """One row per unknown: what it is made of, and what the search chose for it.

    `window_of` maps a fit summary to its diagnostic window, so the absorption at each
    diagnostic run can be reported -- that pair of heights is what distinguishes these
    unknowns from one another, and it is the axis the five were chosen along.
    """
    rows = []
    for name, fit in fit_summaries.items():
        window = window_of(fit)
        peaks = window_absorption(fit['valid_energies'], fit['b'], window)
        results = fit['results']
        row = {'unknown': name, 'energies': len(fit['valid_energies']),
               'combinations': len(results['M']), 'tuned block length': fit['block_length']}
        # Labelled by which run they are, not by energy: the five unknowns sit on grids
        # offset from one another by a few tenths of an eV, so naming the columns after the
        # peak energy would give each unknown its own column and nothing would line up.
        for label, (energy, height) in zip(DIAGNOSTIC_RUN_LABELS, peaks.items()):
            row[label] = round(height, 3)
        for m in sorted(set(results['M'])):
            at_m = np.where(results['M'] == m)[0]
            best = int(at_m[np.argmin(subset_mean_ranks(results['bootstrap_pes'][at_m]))])
            row[f'best M={m}'] = ', '.join(
                fit['ref_names'][int(j)] for j in results['ref_indices'][best, :m]
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index('unknown')

In [ ]:
# About two minutes from cold, seconds once the fit cache is warm.
fit_summaries = fit_all_unknowns(unknown_spectra, reference_spectra)

# The diagnostic window is a property of the references, so it is the same window for every
# unknown up to each one's energy grid -- which is the point of defining it this way.
windows = {name: diagnostic_window(fit['valid_energies'], fit['A'])
           for name, fit in fit_summaries.items()}
regimes = {name: holdout_regimes(fit['holdout_masks'], windows[name])
           for name, fit in fit_summaries.items()}

primary_window = windows[PRIMARY_UNKNOWN]

display(summarize_unknowns(fit_summaries, lambda fit: windows[fit['spectrum_name']]))

print('diagnostic window, per unknown:')
for name, window in windows.items():
    runs = ', '.join(f'{lo:.1f}-{hi:.1f} eV' for lo, hi in window['runs'])
    held = regimes[name]['n_held_out'] / len(regimes[name]['held_out'])
    print(f'  {name:32} {window["n_points"]:3d} points in {runs}   '
          f'mostly held out on {held:.1%} of iterations')

# The block length is tuned per unknown from its own residuals, and it varies a great deal:
# see how far apart the "tuned block length" column above is across these five. The study
# below asks whether that matters.
tuned = {name: fit['block_length'] for name, fit in fit_summaries.items()}
print(f'\ntuned block lengths: {tuned}')
print(f'  range {min(tuned.values())} to {max(tuned.values())} -- '
      f'the largest is well outside the 3-15 band the development notebook validated')

# Part 2 — The studies

Six sections, each interrogating one choice the pipeline above makes. Every one asks its
question twice: against **synthetic data, where the right answer is known and recovery can be
scored**, and against the **five real unknowns, where it matters**. Neither arm is sufficient
alone — synthetic spectra can be scored but are not this data, and the real spectra are this
data but have no known answer, so "the selection did not change" is the strongest thing they
can say by themselves.

Each ends in a **Findings** write-up that states its numbers in the prose, because the study
results are cached outside the repository and a fresh clone renders nothing until it has run.

<a id="study-interpolation"></a>

## Study 1 — Linear against cubic spline interpolation

Every reference has to be resampled onto the unknown's energy grid before anything can be
fitted, and the choice of interpolant sits underneath every number in this notebook.
`mrfitty.base.ReferenceSpectrum` hardcodes a cubic spline; this pipeline makes it a parameter
so the choice can be examined.

The real arm can only ask whether the two methods **select the same references**, and whichever
way that falls it is a null: agreement says the choice does not matter here, disagreement says
one of them is wrong without saying which. The synthetic arm can ask **which one is right**,
because it starts from a curve whose true shape is known.

That truth does not have to be invented. A measured reference spectrum *is* a XANES curve;
subsampling one and rebuilding it asks exactly the question the pipeline asks, with an answer
already in hand.

In [ ]:
# The synthetic arm: rebuild a known curve from fewer points, and see which interpolant gets
# it back. The curves are the real reference spectra, so the test is about XANES shapes --
# a sharp whiteline on a smooth edge step -- rather than about polynomials in the abstract.


def interpolant_recovery(spectra, strides=(2, 3, 4), methods=None):
    """Drop points from each reference, rebuild it, and measure the error against the original.

    Every `stride`-th energy is kept and the rest are rebuilt from them, so the error is
    measured at energies the interpolant never saw. Reported separately inside and outside the
    diagnostic window: the whiteline is where the curve is sharpest and where an interpolant
    is most likely to differ, and it is also the region the fit is deciding on, so an error
    there costs more than the same error in the flat wings.
    """
    if methods is None:
        methods = {'linear': make_linear_interpolant, 'cubic spline': make_cubic_spline_interpolant}

    rows = []
    for spectrum in spectra:
        energies = spectrum.data_df.index.values
        truth = spectrum.data_df.norm.values
        # the whiteline region of this spectrum, by its own sharpest feature
        gradient = np.abs(np.gradient(truth, energies))
        sharp = gradient > 0.5 * gradient.max()

        for stride in strides:
            kept = np.zeros(len(energies), dtype=bool)
            kept[::stride] = True
            kept[0] = kept[-1] = True          # keep the ends, or this becomes extrapolation
            rebuilt_at = ~kept

            for name, make_interpolant in methods.items():
                interpolant = make_interpolant(energies[kept], truth[kept])
                rebuilt = interpolant(energies[rebuilt_at])
                error = np.abs(rebuilt - truth[rebuilt_at])
                in_sharp = sharp[rebuilt_at]
                rows.append({
                    'reference': spectrum.file_name, 'stride': stride, 'method': name,
                    'points_used': int(kept.sum()),
                    'rms_error': float(np.sqrt(np.mean(error ** 2))),
                    'max_error': float(error.max()),
                    'rms_error_at_the_edge': float(np.sqrt(np.mean(error[in_sharp] ** 2)))
                                             if in_sharp.any() else np.nan,
                })
    return pd.DataFrame(rows)


def summarize_interpolant_recovery(recovery):
    """Per stride and method, and the paired count of references each method rebuilds better."""
    summary = recovery.groupby(['stride', 'method']).agg(
        rms_error=('rms_error', 'median'),
        rms_error_at_the_edge=('rms_error_at_the_edge', 'median'),
        max_error=('max_error', 'median'),
    )
    # paired, reference by reference: both methods rebuilt the same curve from the same points
    wins = []
    for stride, at_stride in recovery.groupby('stride'):
        errors = at_stride.pivot(index='reference', columns='method', values='rms_error')
        cubic_better = int((errors['cubic spline'] < errors['linear']).sum())
        wins.append({'stride': stride, 'references': len(errors),
                     'cubic spline rebuilds better': cubic_better,
                     'linear rebuilds better': len(errors) - cubic_better})
    return summary, pd.DataFrame(wins).set_index('stride')


@names_its_figures
def plot_interpolant_recovery(recovery, summary):
    """Where the two interpolants differ, and by how much, as points are taken away."""
    fig, (ax_overall, ax_edge, ax_worst) = plt.subplots(1, 3, figsize=(19, 4.5))
    colors = {'linear': 'darkorange', 'cubic spline': 'steelblue'}

    for column, ax, title in (
        ('rms_error', ax_overall, 'Rebuilding error over the whole curve'),
        ('rms_error_at_the_edge', ax_edge, 'Rebuilding error at the edge, where it is sharp'),
        ('max_error', ax_worst, 'Worst single energy'),
    ):
        for method, color in colors.items():
            at_method = summary.xs(method, level='method')[column]
            ax.plot(at_method.index, at_method.to_numpy(), 'o-', color=color, linewidth=1.6,
                    label=method)
        ax.set_xlabel('Stride (1 in N energies kept)')
        ax.set_ylabel('Median error, normalized absorption')
        ax.set_yscale('log')
        ax.set_title(title)
        ax.legend(fontsize=8)

    fig.suptitle('Rebuilding 24 reference spectra from a fraction of their own points',
                 fontsize=13)
    # tight_layout doesn't account for suptitle; the rect reserves space for it
    fig.tight_layout(rect=[0, 0, 1, 0.91])
    plt.close(fig)
    return fig

In [ ]:
# The synthetic arm: seconds, because it is interpolation and not fitting.
interpolant_recovery_table = cached_study(
    'interpolant_recovery',
    lambda: interpolant_recovery(reference_spectra),
    strides=[2, 3, 4], references=len(reference_spectra),
)
interpolant_summary, interpolant_wins = summarize_interpolant_recovery(interpolant_recovery_table)

display(plot_interpolant_recovery(interpolant_recovery_table, interpolant_summary))
display(interpolant_summary.round(5))
print('paired, reference by reference:')
display(interpolant_wins)

In [ ]:
# The real arm: fit all five unknowns both ways and ask whether the choice reaches the answer.
# Two searches per unknown, about four minutes from cold.


def interpolation_selection_agreement(spectra, reference_spectra, n_bootstrap=400, seed=42):
    """Does the interpolant change which references get selected, on real unknowns?

    A weaker question than the synthetic arm's -- it can only return agreement or
    disagreement, not a verdict -- but it is the one that decides whether the choice matters
    here. Fewer bootstrap iterations than the pipeline uses: this is comparing two rankings,
    not estimating an interval.
    """
    rows = []
    for spectrum in spectra:
        selections = {}
        for name, interpolant in (('linear', make_linear_interpolant),
                                  ('cubic spline', make_cubic_spline_interpolant)):
            energies, A, b, *_ = interpolate_references_at_sample_energies(
                reference_spectra=reference_spectra, sample_spectrum=spectrum,
                make_interpolant=interpolant, verbose=False,
            )
            with contextlib.redirect_stdout(io.StringIO()):
                results, *_ = do_ref_subsets_moving_block_holdout_bootstrap(
                    b, A, M=[1, 2, 3], rng=np.random.default_rng(seed),
                    select_holdout_blocks_fn=select_holdout_blocks, n_bootstrap=n_bootstrap,
                )
            selections[name] = results

        for m in sorted(set(selections['linear']['M'])):
            chosen = {}
            for name, results in selections.items():
                at_m = np.where(results['M'] == m)[0]
                order = np.argsort(subset_mean_ranks(results['bootstrap_pes'][at_m]))
                best = int(at_m[order[0]])
                chosen[name] = tuple(int(j) for j in results['ref_indices'][best, :m])
            ranks = {name: scipy.stats.rankdata(
                        subset_mean_ranks(r['bootstrap_pes'][np.where(r['M'] == m)[0]]))
                     for name, r in selections.items()}
            rows.append({
                'unknown': spectrum.file_name, 'M': m,
                'same_selection': chosen['linear'] == chosen['cubic spline'],
                'rank_correlation': float(scipy.stats.spearmanr(
                    ranks['linear'], ranks['cubic spline']).statistic),
            })
    return pd.DataFrame(rows)


interpolation_agreement = cached_study(
    'interpolation_selection',
    lambda: interpolation_selection_agreement(unknown_spectra, reference_spectra),
    unknowns=list(UNKNOWN_NAMES), n_bootstrap=400,
)
display(interpolation_agreement)
print(f"selections agree on {interpolation_agreement['same_selection'].sum()} of "
      f"{len(interpolation_agreement)} (unknown, subset size) pairs; "
      f"median rank correlation {interpolation_agreement['rank_correlation'].median():.4f}")

### Findings

**The cubic spline is measurably the better interpolant, and on this data the choice almost
never reaches the answer — but "almost" is doing work the old version of this study could not
see.**

*The synthetic arm settles which one is right.* Rebuilding each of the 24 references from a
fraction of its own points, the cubic spline is better at every stride tried, and the margin
widens where it matters. Keeping one energy in two, the median error over the whole curve is
0.0056 against linear's 0.0094; at the edge, where the curve is sharp and where the references
differ from one another, it is 0.0098 against 0.0200 — a factor of two. The worst single energy
is 0.022 against 0.057. Reference by reference, paired on the same points, the spline rebuilds
better on 22 of 24 at stride 2, 23 of 24 at stride 3, and 20 of 24 at stride 4.

That is not a close call, and it is the question the pipeline actually faces: every reference is
resampled onto a grid it was not measured on, so the interpolant is being asked to fill in
exactly these gaps.

*The real arm says it mostly does not matter, and names where it does.* Across five unknowns
and three subset sizes, the two methods select the same references on 12 of 15 combinations,
and the rank correlation between their 2,324-combination orderings never falls below 0.9896.
Where they disagree — `OTT3_55_spot0` at M = 3, `Ott3_73_AsXANES_spot5_000` at M = 3, and
`Ott3_74_AsXANES_spot0` at M = 2 — the orderings are still correlated above 0.99, so the
disagreement is between combinations that were nearly tied anyway.

**What changed by asking twice.** The development notebook ran only the real arm, on one
unknown, and could report only that the two methods agreed. That is a null: it says the choice
did not matter *there*, and offers no reason to prefer either. Adding an arm with a known answer
turns it into a recommendation — keep the spline, which `mrfitty.base.ReferenceSpectrum`
hardcodes anyway — and running five unknowns finds the three cases where the null does not hold.

**Caveat.** Subsampling a measured spectrum is not quite the pipeline's problem: the pipeline
interpolates from one grid onto another of similar density, while this drops points from a grid
to rebuild itself. It is the same operation asked of the same curves, but a stride of 2 is a
harsher test than the pipeline usually poses.

<a id="study-block-length"></a>

## Study 2 — Choosing the block length

`choose_block_length` estimates how long the residual dependence runs, using Politis–White
over every reference combination and taking a low quantile of the result. The pipeline uses
that number for both block lengths, and [the study below](#study-resample-length) takes them
apart; this one asks whether the estimate itself is any good.

It has a question to answer rather than a null to confirm, because running five unknowns
turned one up. The tuned length is **8, 10, 11, 12 and 32** across them. The development
notebook only ever validated 3 to 15 — it found the selected subset identical across that
range and moving at 20 — so `Ott3_73_AsXANES_spot5_000` is being fitted at a block length
nothing has checked, on a 248-point grid where a block of 32 is an eighth of the spectrum.

The synthetic arm asks whether the estimator recovers a dependence length that is known
because it was put there. The real arm asks what the outlying estimate is made of.

In [ ]:
# The synthetic arm: series whose dependence length is known because it was chosen.


def ar1_series(phi, n, rng):
    """An AR(1) series, whose autocorrelation is phi**lag by construction."""
    series = np.zeros(n)
    innovations = rng.standard_normal(n)
    for i in range(1, n):
        series[i] = phi * series[i - 1] + innovations[i]
    return series


def block_length_recovery(phis=(0.0, 0.2, 0.4, 0.6, 0.8), n=198, n_replicates=60, seed=0):
    """What Politis-White returns for a dependence that is known.

    There is no single right answer to compare against -- the optimal block length for a
    block bootstrap is a bias-variance tradeoff, not a property of the series alone -- so what
    is checked is the shape: the estimate should rise with phi, and it should be near 1 when
    the series is independent and there is nothing to preserve.

    The integral time scale, (1 + phi) / (1 - phi), is drawn alongside as the scale the
    dependence actually sets, not as a target the estimator is failing to hit.
    """
    rows = []
    for phi in phis:
        for replicate in range(n_replicates):
            rng = np.random.default_rng([seed, replicate, int(phi * 100)])
            estimate = politis_white_block_length(ar1_series(phi, n, rng))
            rows.append({'phi': phi, 'replicate': replicate,
                         'b_opt': float(estimate['b_opt']),
                         'integral_time_scale': (1 + phi) / (1 - phi) if phi < 1 else np.nan})
    return pd.DataFrame(rows)


@names_its_figures
def plot_block_length_recovery(recovery):
    """What the estimator returns as the dependence it is shown lengthens."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    summary = recovery.groupby('phi')['b_opt']
    ax.errorbar(summary.mean().index, summary.mean().to_numpy(),
                yerr=summary.std().to_numpy(), fmt='o-', color='steelblue', capsize=3,
                linewidth=1.6, label='Politis-White estimate')
    scale = recovery.groupby('phi')['integral_time_scale'].first()
    ax.plot(scale.index, scale.to_numpy(), 's--', color='0.5', linewidth=1.2,
            label='integral time scale (1+φ)/(1−φ)')
    ax.set_xlabel('AR(1) coefficient φ')
    ax.set_ylabel('Block length')
    ax.set_title('Estimating a dependence length that was chosen')
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.close(fig)
    return fig


block_length_recovery_table = cached_study(
    'block_length_recovery', block_length_recovery,
    phis=[0.0, 0.2, 0.4, 0.6, 0.8], n_replicates=60,
)
display(plot_block_length_recovery(block_length_recovery_table))
display(block_length_recovery_table.groupby('phi')[['b_opt', 'integral_time_scale']]
        .agg({'b_opt': ['mean', 'std'], 'integral_time_scale': 'first'}).round(2))

In [ ]:
# The real arm: what the outlying estimate is made of.
#
# choose_block_length takes a low quantile over one Politis-White estimate per reference
# combination, so an unusual answer is a statement about the distribution of those estimates
# rather than about one number.


def block_length_estimates(fit_summaries, percentile=10):
    """The per-combination estimates behind each unknown's tuned length."""
    rows = []
    for name, fit in fit_summaries.items():
        residual_matrix = fit['results']['residuals']
        estimates = np.array([politis_white_block_length(r)['b_opt'] for r in residual_matrix])
        capped = np.ceil(len(fit['valid_energies']) / 3)
        rows.append({
            'unknown': name,
            'energies': len(fit['valid_energies']),
            'tuned': fit['block_length'],
            'p1': float(np.percentile(estimates, 1)),
            'p10': float(np.percentile(estimates, 10)),
            'median': float(np.median(estimates)),
            'at the cap': float((estimates >= capped).mean()),
            'residual rms': float(np.sqrt((residual_matrix ** 2).mean())),
        })
    return pd.DataFrame(rows).set_index('unknown')


block_length_estimate_table = cached_study(
    'block_length_estimates',
    lambda: block_length_estimates(fit_summaries),
    unknowns=list(UNKNOWN_NAMES),
)
display(block_length_estimate_table.round(2))

### Findings

**The estimator responds correctly to dependence it can see, and the low quantile that is meant
to protect it fails on one of the five unknowns.**

*The synthetic arm says the estimator works.* On AR(1) series of the same length as the fitted
spectrum, Politis–White rises monotonically with the dependence it is shown: 1.8, 3.0, 4.7, 7.4
and 13.0 as φ goes 0.0, 0.2, 0.4, 0.6, 0.8. Independent data gets a block of about one, which is
the right answer when there is nothing to preserve. The estimates run consistently above the
integral time scale (1.0, 1.5, 2.3, 4.0, 9.0) by roughly forty per cent, which is expected — the
optimal block length for a block bootstrap is a bias–variance tradeoff and not the correlation
length itself — and they are noisy, with a standard deviation of 2 to 4 across replicates.

*The real arm found something the single-unknown version could not.* `choose_block_length` takes
the tenth percentile of one estimate per reference combination, on the argument that underfit
combinations leave the spectrum in their residuals and inflate the estimate, so a low quantile
reads the well-fitted ones. That argument holds for four of the five unknowns, whose estimate
distributions rise gently:

| unknown | energies | p1 | p10 (tuned) | median |
|---|---|---|---|---|
| `OTT3_55_spot0` | 198 | 8.0 | **10** | 25.9 |
| `Ott3_73_AsXANES_spot5_000` | 248 | 1.3 | **32** | 48.0 |
| `Ott3_73_AsXANES_spot1_avg` | 248 | 2.8 | **12** | 19.6 |
| `Ott3_73_AsXANES_spot6_000` | 248 | 6.4 | **11** | 20.1 |
| `Ott3_74_AsXANES_spot0` | 196 | 3.3 | **8** | 14.8 |

`Ott3_73_AsXANES_spot5_000` is the exception, and the shape of its distribution is the tell: the
first percentile is 1.3 and the tenth is 32. Between those two the estimate leaps by a factor of
twenty-five, which means the low quantile is not sitting on a population of well-fitted
combinations at all — nine tenths of the 2,324 combinations report dependence longer than 32,
and the median is 48. This is the most arsenate-dominated of the five, and the reference pool
evidently fits it poorly enough that almost every combination leaves structure in its residuals.
The quantile defends against *some* combinations being underfit. It has nothing to offer when
nearly all of them are.

Nothing is hitting the cap of ⌈n/3⌉, so this is not the estimator saturating; it is the
estimator correctly reporting long dependence in residuals that still contain the spectrum.

**What follows.** A block length of 32 on a 248-point grid is an eighth of the spectrum per
block, and the development notebook found the selection starting to move at 20 on a 198-point
grid. So one of these five is being fitted outside the range anything has validated. Two
responses suggest themselves and neither is taken here: cap the tuned length at some fraction of
the grid, or treat a leap between low percentiles as a diagnostic that the reference pool does
not span the sample. The second is more informative — it is a statement about the chemistry
rather than about the estimator — and it is testable, because the synthetic machinery in
[the study below](#study-resample-length) can plant a spectrum the pool cannot explain and see
whether the percentile gap opens.

<a id="study-resample-length"></a>

## Study 3 — Resample length against holdout length

The two block lengths in `select_holdout_blocks` are doing opposite jobs.

Holdout blocks want to be **short**. The development notebook found that shortening them from
10 to 3 raised exact recovery of a known combination from 48.7% to 53.7% over 320 synthetic
spectra — a 25-to-9 paired split, about nine times in a thousand by luck. A finely spread
holdout leaves every scored energy close to energies the fit was given.

Resample blocks want to be **long**. The same notebook measured what short blocks cost: mean
absolute autocorrelation error over lags 1–10 rising from 0.075 at length 10 to 0.126 at
length 3, with the bootstrap's own long-run variance estimate still climbing past 10. The
blocks exist to carry residual dependence into the bootstrap, and that dependence is what the
coefficient intervals and the prediction error spread are built on.

Every version of the selector before this one used **one number for both**, so those two
results were in direct conflict and neither could be acted on. Nothing about the method
requires that, and `select_holdout_blocks` now takes them separately. This study asks whether
taking them apart buys both.

There is a constraint in the way. A resample block may not overlap the holdout — otherwise the
bootstrap would be resampling residuals it is about to be scored against — so long resample
blocks need long clear runs, and a finely spread holdout is precisely what does not leave
them. The study measures that first, because it bounds everything else.

In [ ]:
BLOCK_LENGTH_SCOPE = 'every iteration'


def _paired_counts(better_mask, reference_mask):
    """How many replicates each side wins, and the chance of that split if neither is better.

    Every block length saw the same synthetic spectra -- the sweep hands each replicate the
    same seed at every length -- so the useful comparison is spectrum by spectrum. Replicates
    where the two agree say nothing either way and are set aside; only the disagreements are
    counted.
    """
    better = int((better_mask & ~reference_mask).sum())
    worse = int((~better_mask & reference_mask).sum())
    chance = (float(scipy.stats.binomtest(better, better + worse, 0.5).pvalue)
              if better + worse else np.nan)
    return better, worse, chance


def whiteline_recovery_arm(A, residuals, window_mask, select_holdout_blocks_fn, M,
                           n_components, n_bootstrap, block_length, seed,
                           noise_block_length=None, resample_block_length=None):
    """Build one synthetic unknown, search it, and ask each version of the test for an answer.

    The scopes score the same search: every iteration, only the iterations that held out
    most of the whiteline, only those that kept it, and two more drawn down to the same
    number of iterations as the held-out scope. Scoring one search several ways rather than
    running several searches is what makes the comparison fair -- the scopes see the same
    fits, the same draws and the same noise, and differ only in which iterations they are
    allowed to look at.

    Runs in a worker process (see run_arms), so everything it needs arrives as an argument
    and the search's own report is captured rather than printed.

    `resample_block_length` is passed to the search, so an arm can be run with the two
    block lengths set independently.

    `noise_block_length` separates the spectrum being built from the search being run on
    it. They are the same thing by default, but a study that varies the search's block
    length has to hold the synthetic spectrum fixed while it does so, or it changes the
    data and the estimator at once and cannot say which moved the answer.

    Returns
    -------
    dict -- 'true_indices', 'weights', how much of the whiteline the draws held out, and
        per scope: the chosen subset, whether it is the true one, the true subset's rank,
        and a split-half rank correlation
    """
    rng = np.random.default_rng(seed)
    b, true_indices, weights = synthetic_unknown(
        A, residuals, rng, n_components=n_components,
        block_length=block_length if noise_block_length is None else noise_block_length,
    )

    with contextlib.redirect_stdout(io.StringIO()):
        results, holdout_masks, _, _, _ = do_ref_subsets_moving_block_holdout_bootstrap(
            b, A, M=list(M), rng=np.random.default_rng(seed + 1),
            select_holdout_blocks_fn=select_holdout_blocks_fn,
            n_bootstrap=n_bootstrap, block_length=block_length,
            resample_block_length=resample_block_length,
        )

    window_share = holdout_masks[:, window_mask].mean(axis=1)
    held_out = window_share > 0.5
    n_held_out = int(held_out.sum())

    # The held-out scope sees only about a third of the iterations, so comparing it against
    # the full search would confound what those iterations contain with how many there are.
    # The two "matched" scopes are drawn down to exactly n_held_out iterations, which is what
    # makes the comparison about content. The unmatched scopes are kept because they are what
    # a person would actually run.
    all_iterations = np.arange(len(held_out))
    retained_iterations = all_iterations[~held_out]
    scopes = {
        'every iteration': all_iterations,
        'whiteline held out': all_iterations[held_out],
        'whiteline retained': retained_iterations,
        'every iteration, matched n':
            rng.choice(all_iterations, size=n_held_out, replace=False),
        'whiteline retained, matched n':
            rng.choice(retained_iterations, size=min(n_held_out, len(retained_iterations)),
                       replace=False),
    }

    at_size = np.where(results['M'] == n_components)[0]
    truth_row = next(
        i for i in at_size
        if tuple(int(j) for j in results['ref_indices'][i, :n_components]) == true_indices
    )

    arm = {'true_indices': true_indices, 'weights': weights.tolist(),
           # what the block length did to the whiteline, which is the thing a block length
           # sweep is really varying: how often the window is lost whole, and how often
           # some of it survives for the fit to lean on
           'window_all_held_out': float((window_share == 1.0).mean()),
           'window_none_held_out': float((window_share == 0.0).mean()),
           'window_mean_share': float(window_share.mean()),
           'n_iterations': {name: len(rows) for name, rows in scopes.items()}}
    for name, iterations in scopes.items():
        if len(iterations) < 2:
            # a scope with no iterations to rank in; possible only on a tiny run where the
            # window was never held out, and reported as missing rather than crashing
            arm[name] = {'chosen': None, 'correct': False, 'true_rank': np.nan,
                         'n_subsets': len(at_size), 'split_half_rho': np.nan}
            continue
        mean_ranks = subset_mean_ranks(results['bootstrap_pes'][np.ix_(at_size, iterations)])
        order = np.argsort(mean_ranks)
        chosen = int(at_size[order[0]])

        # split the scope's own iterations in half and rank in each: a scope whose two
        # halves disagree is not measuring something stable enough to select on
        half = len(iterations) // 2
        first = subset_mean_ranks(results['bootstrap_pes'][np.ix_(at_size, iterations[:half])])
        second = subset_mean_ranks(results['bootstrap_pes'][np.ix_(at_size, iterations[half:])])

        arm[name] = {
            'chosen': tuple(int(j) for j in results['ref_indices'][chosen, :n_components]),
            'correct': chosen == truth_row,
            'true_rank': int(np.where(at_size[order] == truth_row)[0][0]) + 1,
            'n_subsets': len(at_size),
            'split_half_rho': float(scipy.stats.spearmanr(first, second).statistic),
        }
    return arm


def block_resampled_noise(residuals, rng, block_length):
    """Noise with the autocorrelation of a real fit's residuals, by resampling them in blocks.

    Drawing white noise would make the synthetic spectra easier than real ones and would
    also make the moving-block machinery pointless -- blocks exist because neighboring
    residuals are correlated. Resampling contiguous blocks of the real residuals carries
    that correlation over without assuming a model for it: an AR(1) fitted to these
    residuals matches the lag-1 correlation and then decays much too fast.
    """
    residuals = np.asarray(residuals)
    n = len(residuals)
    n_blocks = int(np.ceil(n / block_length))
    starts = rng.integers(0, n, size=n_blocks)
    # wrap at the end rather than truncating the pool of start positions, so every residual
    # is equally likely to appear -- the same reason select_holdout_blocks_v3 wraps
    return np.concatenate([np.take(residuals, range(s, s + block_length), mode='wrap')
                           for s in starts])[:n]


def synthetic_unknown(A, residuals, rng, n_components=3, block_length=10, min_weight=0.15):
    """A spectrum built from a known combination of references, plus realistic noise.

    The weights are drawn from a Dirichlet and redrawn until none is below `min_weight`.
    A combination carrying a 2% component is not identifiable at this noise level by any
    method, so including such cases would measure the detection limit rather than the thing
    under test.

    Returns
    -------
    (b, true_indices, weights) -- the spectrum, the columns of A that went into it, and
        the coefficients they went in with
    """
    n_refs = A.shape[1]
    true_indices = tuple(sorted(rng.choice(n_refs, size=n_components, replace=False)))
    while True:
        weights = rng.dirichlet(np.full(n_components, 2.0))
        if weights.min() >= min_weight:
            break
    b = A[:, list(true_indices)] @ weights + block_resampled_noise(residuals, rng, block_length)
    return b, true_indices, weights


RECOVERY_SCOPES = ('every iteration', 'whiteline held out', 'whiteline retained',
                   'every iteration, matched n', 'whiteline retained, matched n')


def whiteline_recovery_study(A, residuals, window, select_holdout_blocks_fn,
                             n_replicates=32, n_components=3, n_bootstrap=400,
                             block_length=10, resample_block_length=None,
                             noise_block_length=None, seed=0, n_jobs=N_JOBS):
    """Recovery of a known combination under each version of the test.

    `noise_block_length` pins the spectra while `block_length` varies, for a sweep that
    wants to change only the search. See whiteline_recovery_arm.

    Returns
    -------
    (DataFrame, list) -- one row per (replicate, scope), and the raw arms
    """
    arms = run_arms([
        (whiteline_recovery_arm, {
            'A': A, 'residuals': residuals, 'window_mask': window['mask'],
            'select_holdout_blocks_fn': select_holdout_blocks_fn,
            'M': [n_components], 'n_components': n_components,
            'n_bootstrap': n_bootstrap, 'block_length': block_length,
            'noise_block_length': noise_block_length,
            'resample_block_length': resample_block_length,
            'seed': seed + 1000 * replicate,
        })
        for replicate in range(n_replicates)
    ], n_jobs=n_jobs)

    rows = []
    for replicate, arm in enumerate(arms):
        for scope in RECOVERY_SCOPES:
            rows.append({'replicate': replicate, 'scope': scope,
                         'true_indices': arm['true_indices'],
                         'min_weight': min(arm['weights']),
                         'window_all_held_out': arm['window_all_held_out'],
                         'window_none_held_out': arm['window_none_held_out'],
                         'window_mean_share': arm['window_mean_share'],
                         'n_iterations': arm['n_iterations'][scope],
                         **arm[scope]})
    return pd.DataFrame(rows), arms

In [ ]:
# The two lengths, measured three ways: what the draws can do, what they carry, and what the
# search does with them.


def resample_fidelity(residuals, sampled_starts, resample_block_length, n_lags=10,
                      n_series=200):
    """Mean absolute error in the autocorrelation the resampled series carry.

    The reconstruction is the one `do_moving_block_holdout_bootstrap` actually fits, so this
    measures what the bootstrap used rather than a re-derivation of it. Lags 1 to `n_lags`,
    because that is the range the block length can plausibly reach.
    """
    reconstructed = reconstruct_from_sampled_starts(
        residuals, sampled_starts[:n_series], resample_block_length,
    )
    _, original = calculate_acf(residuals)
    errors = [
        np.abs(calculate_acf(series)[1][1:n_lags + 1] - original[1:n_lags + 1]).mean()
        for series in reconstructed
    ]
    return float(np.mean(errors))


def two_length_geometry(n, residuals, holdout_lengths, resample_lengths, n_bootstrap=400,
                        n_seeds=5, seed=0):
    """Feasibility and autocorrelation fidelity over the grid of the two lengths.

    No fitting, so this is seconds rather than minutes, and it is worth having first: it says
    which corners of the grid can be drawn at all, and the recovery sweep below need not visit
    the ones that cannot.

    Feasibility is tried with several seeds rather than one, because near the boundary it is a
    property of the draw and not of the pair. A single unlucky iteration out of hundreds can
    leave no clear run long enough, so a pair that works at one seed can fail at the next --
    which is exactly what happened the first time this study was run. A pair counts as
    feasible here only if every seed managed it.
    """
    rows = []
    for holdout_length in holdout_lengths:
        for resample_length in resample_lengths:
            row = {'holdout_length': holdout_length, 'resample_length': resample_length}
            draws, failures = [], 0
            for attempt in range(n_seeds):
                try:
                    with contextlib.redirect_stdout(io.StringIO()):
                        draws.append(select_holdout_blocks(
                            n, np.random.default_rng(seed + attempt), n_bootstrap=n_bootstrap,
                            block_length=holdout_length,
                            resample_block_length=resample_length,
                        ))
                except ValueError:
                    failures += 1

            if not draws:
                rows.append({**row, 'feasible': False, 'draw_failures': failures,
                             'availability': 0.0, 'acf_error': np.nan})
                continue

            holdout_masks, sampled_starts, length, _ = draws[0]
            rows.append({
                **row,
                'feasible': failures == 0,
                'draw_failures': failures,
                'availability': available_start_fraction(holdout_masks, length),
                'acf_error': resample_fidelity(residuals, sampled_starts, length),
            })
    return pd.DataFrame(rows)


def two_length_recovery(A, residuals, window, pairs, n_replicates=48, n_components=3,
                        n_bootstrap=400, noise_block_length=10, seed=0, n_jobs=N_JOBS):
    """Recovery of a known combination at each (holdout length, resample length) pair.

    The spectra are held fixed while the search varies -- `noise_block_length` stays at the
    tuned value rather than tracking either length -- so a difference between pairs is the
    estimator and not the data. Every pair sees the same spectra, which makes the comparison
    paired the way the studies before it were.
    """
    frames = []
    for holdout_length, resample_length in pairs:
        try:
            recovery, _ = whiteline_recovery_study(
                A, residuals, window, select_holdout_blocks,
                n_replicates=n_replicates, n_components=n_components,
                n_bootstrap=n_bootstrap, block_length=holdout_length,
                resample_block_length=resample_length,
                noise_block_length=noise_block_length, seed=seed, n_jobs=n_jobs,
            )
        except ValueError as error:
            # Feasibility near the boundary is a property of the draw, not of the pair: the
            # geometry probe tries a handful of seeds and these arms use hundreds, so a pair
            # can clear the probe and still meet one unlucky iteration with no clear run.
            # Drop it and say so rather than losing the rest of the sweep to it.
            print(f'  holdout {holdout_length} / resample {resample_length}: dropped, '
                  f'not drawable at every seed ({error})')
            continue
        frames.append(recovery.assign(holdout_length=holdout_length,
                                      resample_length=resample_length))
    return pd.concat(frames, ignore_index=True)


def summarize_two_length(recovery, geometry, reference_pair=(10, 10), top_k=5,
                         scope=BLOCK_LENGTH_SCOPE):
    """One row per pair: what it recovered, what it carried, and whether it could be drawn.

    Paired against the tuned setting, where both lengths are the number `choose_block_length`
    returns -- which is what every version of this pipeline did before the two were separated.
    """
    at_scope = recovery[recovery['scope'] == scope]
    key = ['holdout_length', 'resample_length']
    ranks = at_scope.pivot_table(index='replicate', columns=key, values='true_rank',
                                 aggfunc='first')
    # aggfunc='first' rather than the default mean: averaging turns the boolean into a float,
    # and the paired counts below are set operations on booleans
    named = at_scope.pivot_table(index='replicate', columns=key, values='correct',
                                 aggfunc='first').astype(bool)
    if reference_pair not in named.columns:
        raise KeyError(
            f'the baseline pair {reference_pair} is not in the sweep, so there is nothing to '
            f'compare against. The grid must contain the tuned length on both axes; it has '
            f'{sorted(named.columns)}'
        )
    reference_named = named[reference_pair]
    reference_top_k = ranks[reference_pair] <= top_k

    geometry = geometry.set_index(key)
    rows = []
    for pair in sorted(named.columns):
        in_top_k = ranks[pair] <= top_k
        named_better, named_worse, named_chance = _paired_counts(named[pair], reference_named)
        rows.append({
            'holdout_length': pair[0], 'resample_length': pair[1],
            'availability': geometry.loc[pair, 'availability'],
            'acf_error': geometry.loc[pair, 'acf_error'],
            'named_the_truth': float(named[pair].mean()),
            f'truth_in_top_{top_k}': float(in_top_k.mean()),
            'named_better': named_better, 'named_worse': named_worse,
            'named_chance_if_equally_good': named_chance,
            'rank_on_worst_tenth': float(ranks[pair].quantile(0.90)),
        })
    return pd.DataFrame(rows).set_index(['holdout_length', 'resample_length'])

In [ ]:
def _annotated_grid(table, value, ax, title, fmt='{:.3f}', cmap='viridis', mask_infeasible=True):
    """One cell per (holdout length, resample length), labelled with its value."""
    grid = table.pivot(index='resample_length', columns='holdout_length', values=value)
    image = ax.imshow(grid.to_numpy(), cmap=cmap, aspect='auto', origin='lower')
    ax.set_xticks(range(len(grid.columns)), grid.columns)
    ax.set_yticks(range(len(grid.index)), grid.index)
    ax.set_xlabel('Holdout block length')
    ax.set_ylabel('Resample block length')
    ax.set_title(title)
    for row in range(grid.shape[0]):
        for column in range(grid.shape[1]):
            cell = grid.to_numpy()[row, column]
            text = '—' if np.isnan(cell) else fmt.format(cell)
            # white on the dark end of the ramp, black on the light end
            finite = grid.to_numpy()[np.isfinite(grid.to_numpy())]
            shade = 'white' if np.isfinite(cell) and cell < np.median(finite) else 'black'
            ax.text(column, row, text, ha='center', va='center', fontsize=8, color=shade)
    return image


@names_its_figures
def plot_two_length_study(geometry, summary, spectrum_name, top_k=5):
    """What each pair of block lengths can draw, what it carries, and what it recovers.

    Returns
    -------
    matplotlib.figure.Figure -- closed, for the caller to display.
    """
    recovered = summary.reset_index()

    fig, (ax_avail, ax_acf, ax_recovery) = plt.subplots(1, 3, figsize=(21, 5.5))
    _annotated_grid(geometry, 'availability', ax_avail,
                    'Resample starts the holdout leaves usable', fmt='{:.2f}')
    # reversed, so dark is good on both of the first two panels
    _annotated_grid(geometry, 'acf_error', ax_acf,
                    'Autocorrelation error, lags 1-10 (lower is better)',
                    fmt='{:.3f}', cmap='viridis_r')
    _annotated_grid(recovered, 'named_the_truth', ax_recovery,
                    f'Names the true combination ({int(summary["named_better"].max() + summary["named_worse"].max())} '
                    f'discordant spectra at most)', fmt='{:.2f}')

    fig.suptitle(
        f'Holdout blocks want to be short, resample blocks want to be long — '
        f'references from {spectrum_name}', fontsize=13,
    )
    # tight_layout doesn't account for suptitle; the rect reserves space for it
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    plt.close(fig)
    return fig

In [ ]:
# The cheap half first: which corners of the grid can be drawn at all, and what each one
# carries. Seconds, because nothing is fitted.
# The tuned length, 10 for this unknown, is on both axes on purpose: the pair (10, 10) is the
# setting every earlier version of this pipeline used, and the study is only meaningful as a
# comparison against it.
TUNED_LENGTH = fit_summaries[PRIMARY_UNKNOWN]['block_length']
HOLDOUT_LENGTHS = (3, 6, TUNED_LENGTH, 14)
RESAMPLE_LENGTHS = (4, 8, TUNED_LENGTH, 16)

two_length_grid = cached_study(
    'two_length_geometry',
    lambda: two_length_geometry(len(primary_energies), primary_residuals,
                                HOLDOUT_LENGTHS, RESAMPLE_LENGTHS),
    holdout_lengths=list(HOLDOUT_LENGTHS), resample_lengths=list(RESAMPLE_LENGTHS),
    n_seeds=5,
)
display(two_length_grid.pivot(index='resample_length', columns='holdout_length',
                              values='availability').round(3))

feasible_pairs = [(int(r.holdout_length), int(r.resample_length))
                  for r in two_length_grid.itertuples() if r.feasible]
marginal = two_length_grid[(two_length_grid['draw_failures'] > 0)
                           & (two_length_grid['availability'] > 0)]
print(f'{len(feasible_pairs)} of {len(two_length_grid)} pairs draw reliably')
if len(marginal):
    print('marginal -- drawable at some seeds and not others:')
    display(marginal[['holdout_length', 'resample_length', 'draw_failures',
                      'availability']])

In [ ]:
# The expensive half: every feasible pair, scored against spectra whose answer is known.
# About twenty-five minutes on 32 cores from cold, because each replicate is a full
# 2,324-combination search. Cached, so it is paid once.
TWO_LENGTH_REPLICATES = 48

two_length_recovery_table = cached_study(
    'two_length_recovery',
    lambda: two_length_recovery(primary_A, primary_residuals, primary_window, feasible_pairs,
                                n_replicates=TWO_LENGTH_REPLICATES),
    pairs=[list(p) for p in feasible_pairs], n_replicates=TWO_LENGTH_REPLICATES,
)

drawn_pairs = set(map(tuple, two_length_recovery_table[['holdout_length',
                                                        'resample_length']].to_numpy()))
if len(drawn_pairs) < len(feasible_pairs):
    print(f'{len(feasible_pairs) - len(drawn_pairs)} pair(s) cleared the probe but could '
          f'not be drawn at every replicate seed; see above')

two_length_summary = summarize_two_length(two_length_recovery_table, two_length_grid,
                                          reference_pair=(TUNED_LENGTH, TUNED_LENGTH))
display(plot_two_length_study(two_length_grid, two_length_summary,
                              primary_spectrum.file_name))
display(two_length_summary.round(4))

In [ ]:
# The grid above draws the shape; 48 spectra are not enough to call the one comparison it
# points at. This repeats the best decoupled pair against the tuned one over enough spectra to
# settle it -- about fifteen minutes on 32 cores, and the same discipline the development
# notebook's short-block result needed: at 96 spectra that answer sat at p = 0.096 and at 320
# it was p = 0.009, and the intermediate measure that looked like a cost turned out to be noise.
CONFIRM_REPLICATES = 256
CONFIRM_PAIRS = [(3, TUNED_LENGTH), (TUNED_LENGTH, TUNED_LENGTH)]

two_length_confirmation = cached_study(
    'two_length_confirmation',
    lambda: two_length_recovery(primary_A, primary_residuals, primary_window, CONFIRM_PAIRS,
                                n_replicates=CONFIRM_REPLICATES),
    pairs=[list(p) for p in CONFIRM_PAIRS], n_replicates=CONFIRM_REPLICATES,
)
display(summarize_two_length(two_length_confirmation, two_length_grid,
                             reference_pair=(TUNED_LENGTH, TUNED_LENGTH)).round(4))

### Findings

**The two lengths do come apart, and pulling them apart is worth doing: a short holdout block
with the tuned resample length names the true combination 52.7% of the time against the tuned
pair's 44.5%, while giving up almost nothing in autocorrelation fidelity.**

The development notebook had these two results and could not reconcile them, because one number
controlled both: short blocks recovered the truth more often, long blocks preserved the residual
structure better, and `select_holdout_blocks` used `block_length_min` for both jobs. Separating
them shows the two effects really are governed by different lengths.

*Each score tracks the length it should.* Autocorrelation error is a property of the resample
length and barely notices the holdout length — across the grid it sits at 0.154–0.160 at resample
4, 0.115–0.124 at resample 10, and 0.108–0.118 at resample 16, moving by less than 0.01 down any
column:

| holdout ↓ / resample → | 4 | 8 | 10 | 16 |
|---|---|---|---|---|
| **3** | 0.1578 | 0.1286 | 0.1238 | 0.1395 |
| **6** | 0.1602 | 0.1232 | 0.1191 | 0.1181 |
| **10** | 0.1578 | 0.1236 | 0.1150 | 0.1103 |
| **14** | 0.1543 | 0.1232 | 0.1165 | 0.1083 |

Recovery moves the other way, along the rows: at 48 replicates the holdout-3 row recovers
0.563 / 0.604 / 0.604 while the holdout-14 row recovers 0.417 / 0.458 / 0.438.

*Availability is what stops you having both in full.* A resample block may not overlap the
holdout, so it needs a clear run that long. Short holdout blocks scatter the holdout finely and
chop the spectrum into short clear runs, exactly when a long resample block needs a long one.
The fraction of usable starts falls from 0.647 at (14, 4) to 0.189 at (3, 16), and (3, 16) is the
one pair of sixteen that the five-seed feasibility probe passed but that then failed to draw at
some replicate seeds — the sweep reports it and drops it rather than dying. That is the corner of
the grid, and it is the practical limit on how far the two lengths can be separated.

*The confirmation run settles it.* 256 replicates on the two pairs that matter, paired so every
replicate sees both:

| pair | names the truth | truth in top 5 | median rank of truth |
|---|---|---|---|
| holdout 3, resample 10 | **0.527** | 0.887 | **1.0** |
| holdout 10, resample 10 | 0.445 | 0.879 | 2.0 |

Paired over the 256 spectra, (3, 10) wins where (10, 10) loses 24 times and loses where it wins
3 times — p = 5 × 10⁻⁵. The gain is in outright identification, not in coarse ranking: top-5
recovery is the same to within a percentage point, so the short holdout is not finding
combinations the long one missed entirely, it is separating the best from the near-best.

**What follows.** Use a short holdout block with the resample length left at the tuned value.
Against the old tied setting the cost is small and quantified: autocorrelation error 0.124 rather
than 0.115, a tenth of the gap between resample 10 and the resample-4 setting a tied short block
would have forced (0.158). That is the trade the single-number version could not offer — the old
"use L = 3" result bought its recovery gain at resample 4, and the fidelity loss was the reason
not to take it. `select_holdout_blocks` now takes `resample_block_length` separately, defaulting
to `block_length_min` so nothing already recorded moves, and the recommendation is to set it.

The caveat is the grid corner: at holdout 3 the resample length cannot go much past 10 before
draws start failing, and the guard raises a `ValueError` naming both lengths rather than
silently returning fewer blocks. On the one of the five unknowns whose tuned length is 32
([study 2](#study-block-length)) this ceiling will bind long before 32, which is one more reason
to treat that spectrum as outside the validated range rather than as a setting to honour.

<a id="study-whiteline"></a>

## Study 4 — What prediction error is measuring, and whether it is a fair test

Before selecting on prediction error it is worth asking what makes it move. Across the
bootstrap it moves mostly for one reason: whether the holdout took the energies the references
disagree about. Those energies are a small part of the spectrum — sixteen of about two hundred,
in two runs — and a holdout block is wide enough to swallow a run whole.

That splits the iterations into two populations asking different questions. When the
diagnostic energies are present, a subset only has to interpolate across gaps in a curve whose
most informative feature it can still see. When they are held out, it has to *reconstruct*
them, which is a far harder demand and one that separates the references sharply. Mixing the
two produces a prediction error distribution with two humps and a median sitting in the gap
between them.

Two questions follow, and the second is the one that decides anything.

1. **How large is the effect?** Measured directly, on all five unknowns.
2. **Is the reconstruction half a test worth running?** Answered against synthetic spectra
   whose answer is known, because on real spectra "the selection did not change" is the
   strongest thing that can be said.

In [ ]:
REGIME_COLORS = {'whiteline held out': 'indianred', 'whiteline retained': 'steelblue'}


def plot_diagnostic_window(energies, b, window, holdout_masks, ax):
    """The measured spectrum and the reference spread, with the diagnostic runs marked."""
    ax.plot(energies, b, color='black', linewidth=1.0, label='measured spectrum')
    for k, (lo, hi) in enumerate(window['runs']):
        ax.axvspan(lo, hi, color='indianred', alpha=0.20,
                   label=f'diagnostic runs ({window["n_points"]} points)' if k == 0 else None)
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Normalized absorption')

    # The spread the window is cut from, and the holdout frequency, on one twinned axis --
    # both are fractions of the same order. The frequency is flat across the runs, which is
    # the point: the window is not held out more often than anywhere else, it is just narrow.
    ax_right = ax.twinx()
    ax_right.plot(energies, window['spread'] / window['spread'].max(), color='slateblue',
                  linewidth=1.0, label='reference spread (scaled)')
    ax_right.plot(energies, holdout_masks.mean(axis=0), color='mediumseagreen',
                  linewidth=0.8, alpha=0.8, label='fraction of iterations held out')
    ax_right.set_ylabel('Fraction')
    ax_right.set_ylim(0, 1.05)
    ax_right.legend(fontsize=7, loc='upper right')
    ax.set_title('Where the references disagree, and how often it is held out')
    ax.legend(fontsize=8, loc='upper left')


def plot_pe_by_regime(bootstrap_pes, regimes, label, ax):
    """One subset's prediction errors, separated into the two regimes.

    Drawn as two overlaid histograms rather than one, because the single distribution is
    the thing being argued against: the two humps are two different questions, and the
    median of the pooled draws falls in the gap between them where almost no iteration
    actually lands.
    """
    held_out = regimes['held_out']
    for values, (name, color) in zip((bootstrap_pes[held_out], bootstrap_pes[~held_out]),
                                     REGIME_COLORS.items()):
        ax.hist(values, bins=40, alpha=0.65, color=color, edgecolor='white',
                label=f'{name} (n={len(values)}, median={np.median(values):.4f})')
    ax.axvline(np.median(bootstrap_pes), color='black', linestyle='--', linewidth=1.2,
               label=f'median of all draws={np.median(bootstrap_pes):.4f}')
    ax.set_xlabel('Holdout Prediction Error (RMSE)')
    ax.set_ylabel('Iterations')
    ax.set_title(f'Prediction error by regime — {label}')
    ax.legend(fontsize=8)


def plot_regime_variance_explained(variance_explained, ax):
    """How much of prediction error is the regime split, across every combination."""
    ax.hist(variance_explained, bins=40, color='slateblue', alpha=0.8, edgecolor='white')
    median = float(np.median(variance_explained))
    ax.axvline(median, color='red', linestyle='--', label=f'median={median:.1%}')
    ax.set_xlabel('Fraction of PE variance explained by the regime alone')
    ax.set_ylabel('Combinations')
    ax.set_title(f'Across all {len(variance_explained)} combinations')
    ax.legend(fontsize=8)


@names_its_figures
def plot_regime_summary(results, ref_names, energies, b, holdout_masks, window, regimes,
                        spectrum_name):
    """What prediction error is measuring, in one figure.

    Returns
    -------
    matplotlib.figure.Figure
        Closed, for the caller to display.
    """
    variance_explained = regime_variance_explained(results['bootstrap_pes'], regimes)

    # the best subset at the largest size, as the example -- the one the search would pick,
    # so the bimodality cannot be dismissed as a property of a poor fit
    largest = max(results['M'])
    at_largest = np.where(results['M'] == largest)[0]
    example = int(at_largest[np.argmin(np.median(results['bootstrap_pes'][at_largest], axis=1))])
    example_label = ' + '.join(
        ref_names[int(j)] for j in results['ref_indices'][example, :largest]
    )

    fig, (ax_spectrum, ax_regimes, ax_variance) = plt.subplots(1, 3, figsize=(21, 5))
    plot_diagnostic_window(energies, b, window, holdout_masks, ax_spectrum)
    plot_pe_by_regime(results['bootstrap_pes'][example], regimes,
                      f'best {largest}-component fit', ax_regimes)
    plot_regime_variance_explained(variance_explained, ax_variance)

    fig.suptitle(
        f'{regimes["n_held_out"]} of {len(regimes["held_out"])} iterations held out most of '
        f'the whiteline — {spectrum_name}',
        fontsize=13,
    )
    # tight_layout doesn't account for suptitle; the rect reserves space for it
    fig.tight_layout(rect=[0, 0, 1, 0.91])
    plt.close(fig)
    runs = ', '.join(f'{lo:.1f}-{hi:.1f}' for lo, hi in window['runs'])
    print(f'diagnostic window: {window["n_points"]} of {len(energies)} energies, '
          f'in {len(window["runs"])} runs at {runs} eV')
    print(f'held out in {regimes["n_held_out"] / len(regimes["held_out"]):.1%} of iterations; '
          f'example subset: {example_label}')
    print(f'regime explains a median of {np.median(variance_explained):.1%} of prediction '
          f'error variance across all {len(variance_explained)} combinations')
    return fig

In [ ]:
# The yardstick every scope is measured against: the same number of ordinary iterations,
# picked at random. Comparing against this rather than against the whole run is what makes
# a difference a statement about *which* iterations a scope sees and not how many.
MATCHED_BASELINE = 'every iteration, matched n'


def summarize_recovery(recovery, top_k=5):
    """How often each scope gets it right, how badly it misses, and how steady it is.

    Every scope scored the same searches, so the useful comparison is spectrum by spectrum:
    on how many spectra did this scope put the true answer near the top when the matched
    yardstick did not, and on how many was it the other way round? Spectra where both agree
    say nothing either way, so they are set aside and only the disagreements are counted.

    'chance_if_equally_good' turns those two counts into one number: if the scope and the
    yardstick were really as good as each other, this is how often a split at least as
    lopsided as the one observed would come up by luck alone. Small means the difference is
    unlikely to be an accident of these particular spectra.
    """
    ranks = recovery.pivot(index='replicate', columns='scope', values='true_rank')
    baseline_in_top_k = ranks[MATCHED_BASELINE] <= top_k

    rows = []
    for scope in RECOVERY_SCOPES:
        at_scope = recovery[recovery['scope'] == scope]
        in_top_k = ranks[scope] <= top_k
        only_scope = int((in_top_k & ~baseline_in_top_k).sum())
        only_baseline = int((~in_top_k & baseline_in_top_k).sum())
        discordant = only_scope + only_baseline
        rows.append({
            'scope': scope,
            'iterations': float(at_scope['n_iterations'].mean()),
            'named_the_truth': float(at_scope['correct'].mean()),
            f'truth_in_top_{top_k}': float(in_top_k.mean()),
            # where the true answer landed on this scope's worst tenth of spectra, and on
            # its single worst -- one number for the usual bad case, one for the tail
            'rank_on_worst_tenth': float(ranks[scope].quantile(0.90)),
            'rank_at_its_worst': int(ranks[scope].max()),
            'halves_agree': float(at_scope['split_half_rho'].mean()),
            'better_than_yardstick': only_scope,
            'worse_than_yardstick': only_baseline,
            'chance_if_equally_good': (
                float(scipy.stats.binomtest(only_scope, discordant, 0.5).pvalue)
                if discordant else np.nan
            ),
        })
    return pd.DataFrame(rows).set_index('scope')


SCOPE_COLORS = {
    'every iteration': 'dimgray',
    'whiteline held out': 'indianred',
    'whiteline retained': 'steelblue',
    'every iteration, matched n': 'black',
    'whiteline retained, matched n': 'cornflowerblue',
}


def _short_scope(scope):
    return scope.replace('whiteline ', '').replace(', matched n', '\n(matched n)')


def plot_recovery_rates(summary, top_k, ax):
    """How often each scope names the true combination, and how often it comes close."""
    scopes = list(summary.index)
    x = np.arange(len(scopes))
    ax.bar(x - 0.2, summary['named_the_truth'], 0.4, label='names the true combination',
           color=[SCOPE_COLORS[s] for s in scopes], alpha=0.9, edgecolor='white')
    ax.bar(x + 0.2, summary[f'truth_in_top_{top_k}'], 0.4,
           label=f'true combination in the top {top_k}',
           color=[SCOPE_COLORS[s] for s in scopes], alpha=0.45, hatch='//', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels([_short_scope(s) for s in scopes], fontsize=8)
    ax.set_ylabel('Fraction of replicates')
    ax.set_ylim(0, 1)
    ax.set_title('Does the scope find the references that went in?')
    ax.legend(fontsize=8, loc='upper right')


def plot_true_rank_distribution(recovery, ax):
    """Where the true combination lands, as a cumulative curve -- the tail is the point."""
    for scope in RECOVERY_SCOPES:
        ranks = np.sort(recovery[recovery['scope'] == scope]['true_rank'].to_numpy())
        ax.step(ranks, np.arange(1, len(ranks) + 1) / len(ranks), where='post',
                color=SCOPE_COLORS[scope], linewidth=1.6, label=_short_scope(scope).replace('\n', ' '))
    ax.set_xscale('log')
    ax.set_xlabel('Rank of the true combination (1 is best)')
    ax.set_ylabel('Fraction of replicates at or below')
    ax.set_title('How far off the ranking is when it is off')
    ax.legend(fontsize=8, loc='lower right')


def plot_split_half_stability(summary, ax):
    """Whether a scope gives the same answer twice, which is not the same as being right."""
    scopes = list(summary.index)
    ax.bar(np.arange(len(scopes)), summary['halves_agree'],
           color=[SCOPE_COLORS[s] for s in scopes], alpha=0.9, edgecolor='white')
    ax.set_xticks(np.arange(len(scopes)))
    ax.set_xticklabels([_short_scope(s) for s in scopes], fontsize=8)
    ax.set_ylabel('Agreement between the scope\'s own two halves')
    ax.set_ylim(0.9, 1.0)
    # every scope sits near 1, which is the point: each one reproduces its own ranking, so
    # the differences in the other two panels are differences in what is being ranked
    ax.set_title('Each scope agrees with itself')


@names_its_figures
def plot_whiteline_recovery_study(recovery, spectrum_name, top_k=5):
    """The study in one figure, with its summary table.

    Returns
    -------
    (matplotlib.figure.Figure, pandas.DataFrame) -- the figure, closed, and the summary
    """
    summary = summarize_recovery(recovery, top_k=top_k)

    fig, (ax_rates, ax_ranks, ax_stability) = plt.subplots(1, 3, figsize=(21, 5))
    plot_recovery_rates(summary, top_k, ax_rates)
    plot_true_rank_distribution(recovery, ax_ranks)
    plot_split_half_stability(summary, ax_stability)

    n_replicates = recovery['replicate'].nunique()
    n_subsets = int(recovery['n_subsets'].iloc[0])
    fig.suptitle(
        f'Recovering a known combination from {n_replicates} synthetic spectra, '
        f'{n_subsets} candidates each — references from {spectrum_name}',
        fontsize=13,
    )
    # tight_layout doesn't account for suptitle; the rect reserves space for it
    fig.tight_layout(rect=[0, 0, 1, 0.91])
    plt.close(fig)
    return fig, summary

In [ ]:
# The real arm: how much of each unknown's prediction error is just the regime, and how
# differently the two regimes rank the same combinations.


def regime_effect(fit_summaries, windows, regimes):
    """Per unknown: the variance the regime split accounts for, and how far the two disagree."""
    rows = []
    for name, fit in fit_summaries.items():
        results, regime = fit['results'], regimes[name]
        explained = regime_variance_explained(results['bootstrap_pes'], regime)
        held_out, retained = regime['held_out'], ~regime['held_out']

        # rank the combinations at the largest size within each regime, and compare
        at_m = np.where(results['M'] == max(results['M']))[0]
        ranks_out = subset_mean_ranks(results['bootstrap_pes'][np.ix_(at_m, np.where(held_out)[0])])
        ranks_in = subset_mean_ranks(results['bootstrap_pes'][np.ix_(at_m, np.where(retained)[0])])
        rows.append({
            'unknown': name,
            'iterations losing the window': regime['n_held_out'],
            'median variance explained': float(np.median(explained)),
            'subsets above 0.5 explained': float((explained > 0.5).mean()),
            'rank correlation between regimes': float(scipy.stats.spearmanr(
                ranks_out, ranks_in).statistic),
            'same winner': bool(at_m[np.argmin(ranks_out)] == at_m[np.argmin(ranks_in)]),
        })
    return pd.DataFrame(rows).set_index('unknown')


regime_effect_table = cached_study(
    'regime_effect',
    lambda: regime_effect(fit_summaries, windows, regimes),
    unknowns=list(UNKNOWN_NAMES),
)
display(regime_effect_table.round(3))

display(plot_regime_summary(
    fit_summaries[PRIMARY_UNKNOWN]['results'], ref_names, primary_energies, primary_b,
    fit_summaries[PRIMARY_UNKNOWN]['holdout_masks'], primary_window,
    regimes[PRIMARY_UNKNOWN], PRIMARY_UNKNOWN,
))

In [ ]:
# The synthetic arm: is the reconstruction half a test worth running?
#
# Each replicate is a full search, so this is the expensive one -- about four minutes on 32
# cores from cold. Every scope scores the *same* search, differing only in which iterations it
# may look at, and two of them are drawn down to the number of iterations the held-out scope
# has, because a scope with fewer iterations does worse whatever those iterations contain.
WINDOW_REPLICATES = 96

window_recovery_table = cached_study(
    'window_recovery',
    lambda: whiteline_recovery_study(
        primary_A, primary_residuals, primary_window, select_holdout_blocks,
        n_replicates=WINDOW_REPLICATES, n_components=3, n_bootstrap=400,
        block_length=fit_summaries[PRIMARY_UNKNOWN]['block_length'],
    )[0],
    n_replicates=WINDOW_REPLICATES, n_bootstrap=400,
)

window_recovery_figure, window_recovery_summary = plot_whiteline_recovery_study(
    window_recovery_table, PRIMARY_UNKNOWN,
)
display(window_recovery_figure)
display(window_recovery_summary.round(4))

### Findings

**Holding out the diagnostic window is a fair test — losing it does not stop the bootstrap
finding the right combination — but it is a *different* test, and on three of the five unknowns
the two regimes crown different winners.**

*The window itself is now stable.* Derived from where the references disagree rather than from
one unknown's tallest peak, it is 16 energies in two runs for every one of the five unknowns,
landing at 11867.6–11871.9 and 11873.6–11876.9 eV — the ±0.2 eV spread is each unknown's own
measurement grid, not a different answer. Those are the reduced-arsenic and arsenate white lines,
which is the answer the old unknown-derived window was reaching for and missed — it returned 200
points on a 198-point grid for `Ott3_73_AsXANES_spot6_000` and 78 for that spot's own repeat
scan. The absorption in the two runs still separates the chemistry, as a reported diagnostic
rather than as the window definition:

| unknown | 11867.7–11871.7 (reduced) | 11873.7–11876.7 (arsenate) |
|---|---|---|
| `OTT3_55_spot0` | 2.163 | 1.274 |
| `Ott3_73_AsXANES_spot5_000` | 0.707 | 3.831 |
| `Ott3_73_AsXANES_spot1_avg` | 2.510 | 1.108 |
| `Ott3_73_AsXANES_spot6_000` | 1.566 | 1.490 |
| `Ott3_74_AsXANES_spot0` | 0.419 | 3.197 |

*The synthetic arm says the test is fair.* On 96 spectra built from known three-reference
mixtures, restricting the bootstrap to the iterations that lost the window recovers the planted
combination 42.7% of the time against 47.9% for the iterations that kept it — 8 better, 4 worse
when paired, p = 0.39. There is no evidence that removing the most informative energies makes
the estimator pick the wrong answer. What it does cost is resolution below first place: the truth
lands in the top five on 75.0% of the window-held-out runs against 90.6% for a matched number of
unrestricted iterations, 2 better against 17 worse, p = 7 × 10⁻⁴. So the window is where the
separation between near-equal combinations lives. Take it away and the winner survives; the
order behind it scrambles.

*The real arm says the two regimes are not interchangeable.* The window is held out whole on
26–37% of iterations, and which regime an iteration is in explains a median of 24–56% of the
variance in its prediction error:

| unknown | iterations losing the window | median variance explained | subsets above 0.5 | rank corr. | same winner |
|---|---|---|---|---|---|
| `OTT3_55_spot0` | 297 | 0.466 | 26.1% | 0.953 | no |
| `Ott3_73_AsXANES_spot5_000` | 368 | 0.547 | 63.2% | 0.767 | no |
| `Ott3_73_AsXANES_spot1_avg` | 279 | 0.468 | 23.1% | 0.895 | no |
| `Ott3_73_AsXANES_spot6_000` | 299 | 0.565 | 76.9% | 0.959 | yes |
| `Ott3_74_AsXANES_spot0` | 259 | 0.236 | 0.0% | 0.937 | yes |

Rank correlation between the two regimes runs 0.77–0.96, so the regimes broadly agree on the
ordering — and still disagree on first place for three of the five. `Ott3_73_AsXANES_spot5_000`
is the extreme on both counts, with the lowest agreement (0.767) and the most
regime-driven combinations, which is consistent with it being the spectrum the pool fits worst
in [study 2](#study-block-length).

**What follows.** The window-held-out iterations are a legitimate part of the sample and should
stay in — the synthetic arm gives no reason to drop them, and dropping a third of the iterations
costs precision outright. But a single prediction error averages two measurements of different
things, and the average can name a combination that neither regime would name on its own. Report
the regime split alongside the selection, as the pipeline now does: when the two regimes agree,
the choice is robust to the question of whether the white lines were seen; when they disagree, as
on three of these five, the selection depends on how often the draw happened to hide them, and a
tie set is the honest answer rather than a winner.

<a id="study-selection"></a>

## Study 5 — Which interval, and what counts as equally good

Two questions that turn out to be one. The tie rules need a confidence interval on a median,
and this repository calls three different things a "95% CI" — so the first question is which
estimator to use, and it only matters because of what the second does with it.

The second question is what "equally good" means. Because every combination was scored on the
same held-out energies, the difference between two of them can be taken iteration by
iteration, which cancels whatever made an iteration easy or hard for both. What interval to put
around that paired difference is the choice that decides how many combinations the search
reports as indistinguishable from the best.

Both arms are synthetic here, in different senses: the estimators are measured against a median
known exactly, by treating an observed set of draws as its own population; the tie rules are
measured against planted differences. The real arm is what the rules do to the five unknowns.

In [ ]:
def _representative_combinations(results):
    """(label, row index, best row index at that size) for four fits spanning each size.

    The runner-up is the point that matters: it is the closest call the selector below ever
    has to make, and an estimator that disagrees with the others anywhere will disagree
    there. Best, median and worst are included so the comparison is not read off one
    borderline case -- the estimators have the easiest job where the draws are tightest.
    """
    targets = []
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        order = m_indices[np.argsort(np.median(results['bootstrap_pes'][m_indices], axis=1))]
        best = int(order[0])
        picks = [('best', order[0]), ('runner-up', order[min(1, len(order) - 1)]),
                 ('median', order[len(order) // 2]), ('worst', order[-1])]
        targets.extend((f'M={m} {rank}', int(i), best) for rank, i in picks)
    return targets


def ci_method_agreement(results, seed=0):
    """Every estimator applied to the same real arrays: do they disagree, and does it matter?

    Each row is one (combination, estimator) pair, carrying both intervals the selector
    below deals with -- the combination's own interval on its median prediction error, and
    the interval on its paired difference against the best combination at its size. The
    `tied` column is the verdict that interval produces, so a disagreement between
    estimators shows up as a changed decision rather than a changed decimal.
    """
    rows = []
    for label, i, best_i in _representative_combinations(results):
        prediction_errors = results['bootstrap_pes'][i]
        differences = prediction_errors - results['bootstrap_pes'][best_i]
        for method_index, (method_name, median_ci) in enumerate(MEDIAN_CI_METHODS.items()):
            # one rng per (combination, estimator), seeded from position rather than from
            # str.hash, which is salted per process and would make the table irreproducible
            rng = np.random.default_rng([seed, i, method_index])
            pe_lo, pe_hi = median_ci(prediction_errors, rng)
            d_lo, d_hi = median_ci(differences, rng)
            rows.append({
                'subset': label,
                'is_best': i == best_i,
                'method': method_name,
                'pe_median': float(np.median(prediction_errors)),
                'pe_ci_lo': pe_lo,
                'pe_ci_hi': pe_hi,
                'pe_ci_width': pe_hi - pe_lo,
                'd_median': float(np.median(differences)),
                'd_ci_lo': d_lo,
                'd_ci_hi': d_hi,
                'd_ci_width': d_hi - d_lo,
                # NaN propagates to False here, which is the honest reading: an estimator
                # that cannot produce an interval has not called anything tied
                'tied': bool(d_lo <= 0.0 <= d_hi),
            })
    return pd.DataFrame(rows)


def ci_coverage_arm(values, arm_label, method_name, median_ci, n_replicates, seed):
    """How often one estimator's interval covers a median that is known exactly.

    The observed draws are treated as the population. Its median is then known without
    assuming any distribution, and resampling n draws from it reproduces the real,
    right-skewed shape of the data the estimator will be used on -- which a normal or
    lognormal stand-in would not. An estimator that deserves the label "95%" should cover
    that known median in about 95% of the replicates.

    Runs in a worker process (see run_arms), so the estimator arrives as an argument rather
    than being looked up in a global, and nothing is printed.

    Parameters
    ----------
    values       : ndarray -- the draws standing in as the population
    arm_label    : str -- which array this is, carried through to the figure
    method_name  : str -- which estimator, carried through to the figure
    median_ci    : callable (values, rng) -> (lo, hi)
    n_replicates : int -- resampled data sets; the coverage estimate's own noise is
                   sqrt(0.95 * 0.05 / n_replicates), about 0.010 at 500
    seed         : int -- the arm's whole source of randomness

    Returns
    -------
    dict -- 'arm', 'method', 'coverage', 'failed', 'mean_width', 'n_replicates', 'truth'.
        'failed' is the fraction of replicates where the estimator returned no interval at
        all. Those count against coverage, because an estimator that cannot answer has not
        covered anything, and they are reported separately so the two failure modes -- an
        interval that misses, and no interval -- are not read as one number.
    """
    values = np.asarray(values)
    truth = float(np.median(values))
    n = len(values)
    rng = np.random.default_rng(seed)

    covered = 0
    failed = 0
    widths = np.empty(n_replicates)
    for replicate in range(n_replicates):
        resampled = values[rng.integers(0, n, size=n)]
        lo, hi = median_ci(resampled, rng)
        if not (np.isfinite(lo) and np.isfinite(hi)):
            failed += 1
            widths[replicate] = np.nan
            continue
        covered += lo <= truth <= hi
        widths[replicate] = hi - lo

    return {
        'arm': arm_label,
        'method': method_name,
        'coverage': covered / n_replicates,
        'failed': failed / n_replicates,
        'mean_width': float(np.nanmean(widths)) if failed < n_replicates else np.nan,
        'n_replicates': n_replicates,
        'truth': truth,
    }


def ci_method_timing(values, n_repeats=10, seed=0):
    """Seconds per interval, measured serially so the numbers are not distorted by load."""
    rows = []
    for method_name, median_ci in MEDIAN_CI_METHODS.items():
        rng = np.random.default_rng(seed)
        start = time.perf_counter()
        for _ in range(n_repeats):
            median_ci(values, rng)
        rows.append({
            'method': method_name,
            'seconds_per_interval': (time.perf_counter() - start) / n_repeats,
        })
    return pd.DataFrame(rows)


def compare_median_ci_methods(results, seed=0, n_replicates=500, n_jobs=N_JOBS):
    """Measure the three estimators on the fit above: agreement, coverage, and cost.

    Coverage is run for two arms, because the selector below uses these estimators on two
    differently shaped arrays: prediction errors, which are positive and right-skewed, and
    paired differences between two subsets' prediction errors, which are roughly symmetric
    and sit near zero. An estimator can be well calibrated on one and not the other.

    Returns
    -------
    dict of pandas.DataFrame -- 'agreement', 'coverage', 'timing', plus 'n_combinations'
    """
    targets = _representative_combinations(results)
    best_at_largest_m = targets[-1][2]          # every pick at a size carries its best
    runner_up = targets[-3][1]                  # the 'runner-up' pick at that size
    coverage_arrays = {
        'prediction error': results['bootstrap_pes'][best_at_largest_m],
        'paired difference': (results['bootstrap_pes'][runner_up]
                              - results['bootstrap_pes'][best_at_largest_m]),
    }

    arms = run_arms([
        (ci_coverage_arm, {
            'values': values, 'arm_label': arm_label, 'method_name': method_name,
            'median_ci': median_ci, 'n_replicates': n_replicates,
            'seed': seed + 1000 * arm_index + method_index,
        })
        for arm_index, (arm_label, values) in enumerate(coverage_arrays.items())
        for method_index, (method_name, median_ci) in enumerate(MEDIAN_CI_METHODS.items())
    ], n_jobs=n_jobs)

    return {
        'agreement': ci_method_agreement(results, seed=seed),
        'coverage': pd.DataFrame(arms),
        'timing': ci_method_timing(coverage_arrays['prediction error'], seed=seed),
        'n_combinations': len(results['M']),
    }

In [ ]:
# Which interval. The three estimators are measured against a median known exactly: an
# observed set of draws is treated as its own population, so the truth needs no distributional
# assumption and keeps the real right-skewed shape.
CI_REPLICATES = 500

ci_comparison = cached_study(
    'ci_estimators',
    lambda: compare_median_ci_methods(fit_summaries[PRIMARY_UNKNOWN]['results'],
                                      seed=7, n_replicates=CI_REPLICATES)['coverage'],
    n_replicates=CI_REPLICATES,
)
display(ci_comparison.round(4))

timing = pd.DataFrame(ci_method_timing(
    fit_summaries[PRIMARY_UNKNOWN]['results']['bootstrap_pes'][0]))
timing['minutes for the whole search'] = (
    timing['seconds_per_interval'] * 2 * len(fit_summaries[PRIMARY_UNKNOWN]['results']['M']) / 60
)
display(timing.round(5))

In [ ]:
# What counts as equally good. The two rules on the same paired differences, and what each
# does to the five unknowns.
tie_tables = {
    name: peci_tie_table(fit['results'], fit['ref_names'], regimes=regimes[name], seed=11)
    for name, fit in fit_summaries.items()
}
median_ci_tie_tables = {
    name: peci_tie_table(fit['results'], fit['ref_names'],
                         tie_rule=tie_by_paired_median_ci, seed=11)
    for name, fit in fit_summaries.items()
}

tie_counts = pd.concat(
    [peci_tie_counts(tie_tables[name]).rename(columns={'tied': 'middle 95% of differences'})
       .join(peci_tie_counts(median_ci_tie_tables[name])['tied'].rename('CI on the median'))
       .assign(unknown=name)
     for name in fit_summaries],
).set_index('unknown', append=True).reorder_levels(['unknown', 'M'])
display(tie_counts)

# Truncating the draws to the first n of them is exactly a shorter bootstrap run, so this
# sweep costs no refitting.
sensitivity = cached_study(
    'tie_rule_sensitivity',
    lambda: tie_rule_sensitivity(fit_summaries[PRIMARY_UNKNOWN]['results'], ref_names,
                                 n_bootstrap_grid=[50, 100, 250, 500, 1000], seed=11),
    n_bootstrap_grid=[50, 100, 250, 500, 1000],
)
display(plot_tie_rule_sensitivity(sensitivity, PRIMARY_UNKNOWN))

### Findings

**Use the order statistic for the interval and the distribution rule for the tie set. BCa fails
on exactly the quantity the tie rules are built from, and the median-CI rule reports a tie set
that shrinks toward a single combination as you buy more bootstrap draws.**

*Which interval.* Three estimators, 500 replicates, coverage of a nominal 95% interval:

| quantity | percentile | scipy BCa | order statistic |
|---|---|---|---|
| prediction error | 0.960 | 0.928 (3.2% failed) | 0.944 |
| paired difference | 0.944 | **0.718 (23.8% failed)** | 0.956 |

On the prediction error itself all three are usable. On the paired difference — the quantity
every tie rule actually consumes — BCa covers 71.8% of the time and refuses to produce an
interval at all on nearly a quarter of replicates. The reason is visible in the data rather than
in the method: paired differences are taken iteration by iteration against a shared set of
holdout draws, so for similar combinations a large share of the iterations give *identical*
differences. BCa's acceleration term is a jackknife skewness estimate, and it is undefined when
the jackknife replicates do not vary. The order statistic has no such term, costs nothing in
width (0.000225 against BCa's 0.000226, a fraction of a per cent), and covers correctly.

*What counts as equally good.* The two rules behave completely differently as the number of
bootstrap draws rises:

| draws | median-CI M=1 / 2 / 3 | distribution M=1 / 2 / 3 |
|---|---|---|
| 50 | 2 / 3 / 9 | 7 / 115 / 149 |
| 100 | 2 / 8 / 10 | 8 / 86 / 145 |
| 250 | 2 / 2 / 4 | 8 / 101 / 153 |
| 500 | 2 / 1 / 1 | 8 / 103 / 150 |
| 1000 | 2 / 1 / 1 | 8 / 102 / 136 |

The median-CI rule collapses: by 500 draws it ties exactly one combination at M = 2 and M = 3,
meaning it declares a unique winner. That is not a finding about the data, it is arithmetic — the
confidence interval on a median narrows like 1/√n, so any rule that asks "is zero inside this
interval" will eventually answer no for every pair, and the tie set is a statement about how long
the bootstrap ran. The distribution rule asks a question whose answer does not depend on n: the
middle 95% of the paired differences is a fixed quantile of a fixed distribution, and it settles
by 100 draws and stays there (8 / 86–103 / 136–153 across a twentyfold range of draws).

At 1000 draws the distribution rule returns tie sets of **8 / 102 / 136** on `OTT3_55_spot0`,
reproducing the development notebook's recorded numbers exactly — the continuity check for this
study.

**What follows.** `peci_tie_table` defaults to the order statistic and
`tie_by_paired_distribution`, and both defaults are now justified by something other than
preference. The median-CI rule is kept and exercised, because it answers a real question — "can
we resolve these two at all, given this many draws?" — but it is the wrong tool for reporting
which combinations a reader should regard as equally supported. A tie set of 102 at M = 2 is not
the method failing to decide; it is the honest width of what 1000 draws on this spectrum can
distinguish, and it does not get narrower by running longer.

<a id="study-distance"></a>

## Study 6 — Correlation against cosine reference distance

The reference tree decides whether a selected combination reads as spanning the pool or as
three names for one thing, and the tree depends on the distance. Correlation distance centres
each reference before comparing, so it ignores a constant offset; cosine does not.

The synthetic arm plants a cluster structure and asks which metric recovers it. The real arm
asks whether the two trees disagree about anything that changes how a selected combination is
read.

In [ ]:
# The synthetic arm: plant a cluster structure, then ask which distance recovers it.


def planted_reference_pool(rng, n_energies=200, n_groups=4, per_group=5, noise=0.05,
                           offset_scale=0.0):
    """References in known groups, optionally each shifted by a constant of its own.

    The offset is the whole point of the comparison. Correlation distance centres each
    reference before comparing, so a constant shift is invisible to it; cosine does not, so a
    shifted copy of a shape looks like a different shape. Raising `offset_scale` is therefore
    a dial that should leave correlation alone and degrade cosine, and if it does not then the
    two metrics do not differ in the way their definitions say they do.
    """
    grid = np.linspace(0, 1, n_energies)
    shapes = [np.exp(-0.5 * ((grid - centre) / 0.06) ** 2) + 0.4 * np.tanh((grid - 0.3) / 0.1)
              for centre in np.linspace(0.25, 0.75, n_groups)]

    columns, truth = [], []
    for group, shape in enumerate(shapes):
        for _ in range(per_group):
            offset = offset_scale * rng.standard_normal()
            columns.append(shape + noise * rng.standard_normal(n_energies) + offset)
            truth.append(group)
    names = [f'g{g}_{i}.e' for g in range(n_groups) for i in range(per_group)]
    return np.column_stack(columns), np.array(truth), names


def metric_recovers_planted_groups(offsets=(0.0, 0.25, 0.5, 1.0), n_replicates=40,
                                   n_groups=4, per_group=5, seed=0):
    """Adjusted Rand score of each metric against the planted grouping, as the offset grows.

    The tree is cut at the number of groups that were planted, so this measures the distance
    and the linkage rather than the cutoff rule.
    """
    rows = []
    for offset_scale in offsets:
        for replicate in range(n_replicates):
            rng = np.random.default_rng([seed, replicate, int(offset_scale * 1000)])
            A, truth, names = planted_reference_pool(rng, n_groups=n_groups,
                                                     per_group=per_group,
                                                     offset_scale=offset_scale)
            for metric in ('correlation', 'cosine'):
                distances = np.clip(pdist(np.ascontiguousarray(A.T), metric=metric), 0.0, None)
                labels = hc.fcluster(hc.linkage(distances, method='complete'),
                                     t=n_groups, criterion='maxclust')
                rows.append({'offset_scale': offset_scale, 'replicate': replicate,
                             'metric': metric,
                             'agreement_with_truth': float(adjusted_rand_score(truth, labels))})
    return pd.DataFrame(rows)


@names_its_figures
def plot_metric_recovery(recovery):
    """How well each distance recovers a planted grouping as constant offsets are added."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for metric, color in (('correlation', 'steelblue'), ('cosine', 'darkorange')):
        at_metric = recovery[recovery['metric'] == metric]
        summary = at_metric.groupby('offset_scale')['agreement_with_truth']
        means, errors = summary.mean(), summary.sem()
        ax.errorbar(means.index, means.to_numpy(), yerr=errors.to_numpy(), fmt='o-',
                    color=color, capsize=3, linewidth=1.6, label=metric)
    ax.set_xlabel('Scale of the constant offset added to each reference')
    ax.set_ylabel('Adjusted Rand score against the planted groups')
    ax.set_ylim(0, 1.05)
    ax.set_title('Recovering a grouping that was put there on purpose')
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.close(fig)
    return fig


metric_recovery_table = cached_study(
    'metric_recovers_planted_groups',
    metric_recovers_planted_groups,
    offsets=[0.0, 0.25, 0.5, 1.0], n_replicates=40,
)
display(plot_metric_recovery(metric_recovery_table))
display(metric_recovery_table.groupby(['offset_scale', 'metric'])['agreement_with_truth']
        .agg(['mean', 'std']).round(3))

In [ ]:
def compare_cluster_metrics(
    A, ref_names, highlight, metrics=('correlation', 'cosine'), method='complete',
    seed=42, resample_count=1000, percentile=95.0,
):
    """Cluster one design matrix under each metric and measure what the choice changes.

    Controlled the way `compare_interpolation_methods` is: one A, one linkage method, the
    same number of randomized copies behind each cutoff, and the same combinations located
    in both trees, so the metric is the only thing that differs. Each arm gets a freshly
    seeded generator, and because `rng.permuted` depends only on A's shape, both arms
    shuffle the references in exactly the same way -- so the two cutoffs are measured
    against the same randomized data. That is asserted here through the stored digest
    rather than assumed.
    """
    from scipy.stats import spearmanr
    from sklearn.metrics import adjusted_rand_score

    clusterings = {
        metric: cluster_reference_spectra(
            A, ref_names, np.random.default_rng(seed), metric=metric, method=method,
            resample_count=resample_count, percentile=percentile, verbose=False,
        )
        for metric in metrics
    }
    digests = {c['first_permutation_digest'] for c in clusterings.values()}
    assert len(digests) == 1, 'the metric arms did not shuffle the references the same way'

    groups = _normalize_highlight(highlight, ref_names)

    # nearest neighbor of every reference, under each metric
    nearest = {}
    for metric, clustering in clusterings.items():
        square = squareform(clustering['distances'])
        np.fill_diagonal(square, np.inf)
        nearest[metric] = [ref_names[j] for j in square.argmin(axis=1)]
    first, second = metrics
    changed_neighbor = [
        (ref_names[i], nearest[first][i], nearest[second][i])
        for i in range(len(ref_names)) if nearest[first][i] != nearest[second][i]
    ]

    # where each combination sits in each tree -- the question the section is asked to settle
    subtree_spread = {}
    for label, indices in groups.items():
        subtree_spread[label] = {}
        for metric, clustering in clusterings.items():
            group = smallest_enclosing_subtree(clustering['Z'], indices)
            subtree_spread[label][metric] = {
                **group,
                'height_fraction': group['height'] / clustering['Z'][-1, 2],
                'within_cutoff': group['height'] <= clustering['cutoff_distance'],
            }

    return {
        'metrics': tuple(metrics),
        'ref_names': list(ref_names),
        'clusterings': clusterings,
        'groups': groups,
        'nearest_neighbor': nearest,
        'changed_neighbor': changed_neighbor,
        'subtree_spread': subtree_spread,
        'distance_spearman': float(spearmanr(*[clusterings[m]['distances'] for m in metrics]).statistic),
        'adjusted_rand': float(adjusted_rand_score(*[clusterings[m]['labels'] for m in metrics])),
    }


def summarize_cluster_metric_comparison(comparison):
    """Print the comparison: the trees' own statistics, then what actually moved."""
    metrics = comparison['metrics']
    clusterings = comparison['clusterings']

    print(f"{'':<30}" + ''.join(f'{m:>16}' for m in metrics))
    rows = [
        ('largest distance', lambda c: f"{c['distances'].max():.4f}"),
        ('root merge height', lambda c: f"{c['Z'][-1, 2]:.4f}"),
        ('cutoff distance', lambda c: f"{c['cutoff_distance']:.4f}"),
        ('clusters at the cutoff', lambda c: f"{c['n_clusters']:d}"),
        ('cluster sizes', lambda c: '/'.join(str(s) for s in sorted(np.bincount(c['labels'])[1:], reverse=True))),
        ('cophenetic correlation', lambda c: f"{c['cophenetic_correlation']:.5f}"),
    ]
    for name, value_of in rows:
        print(f'{name:<30}' + ''.join(f'{value_of(clusterings[m]):>16}' for m in metrics))

    print(f'\npairwise distance rank correlation (Spearman): {comparison["distance_spearman"]:.5f}')
    print(f'flat clustering agreement (adjusted Rand):     {comparison["adjusted_rand"]:.4f}')
    print(f'references whose nearest neighbor changes:     '
          f'{len(comparison["changed_neighbor"])} of {len(comparison["ref_names"])}')
    for name, first_neighbor, second_neighbor in comparison['changed_neighbor']:
        print(f'  {name}\n      {metrics[0]}: {first_neighbor}\n      {metrics[1]}: {second_neighbor}')

    print('\nwhere each selected combination sits:')
    print(f"{'combination':<14}" + ''.join(f'{m:>34}' for m in metrics))
    for label, per_metric in comparison['subtree_spread'].items():
        cells = ''
        for metric in metrics:
            spread = per_metric[metric]
            verdict = 'within one cluster' if spread['within_cutoff'] else 'spans the tree'
            cells += (f"{spread['height']:>8.4f} ({spread['height_fraction']:>3.0%}) "
                      f"{spread['size']:>2d} leaves {verdict:>18}")
        print(f'{label:<14}' + cells)


@names_its_figures
def plot_cluster_metric_comparison(comparison, spectrum_name):
    """The two trees on one figure, then the two diagnostics on another.

    Two figures rather than one grid: 24 reference file names need a dendrogram panel's
    full width, and squeezing the diagnostics in beside them clips the labels.
    """
    metrics = comparison['metrics']
    first, second = metrics

    fig_trees, axs = plt.subplots(1, len(metrics), figsize=(11 * len(metrics), 9))
    for ax, metric in zip(np.atleast_1d(axs), metrics):
        plot_highlighted_reference_dendrogram(
            comparison['clusterings'][metric], highlight=comparison['groups'], ax=ax,
            title=f'{metric} distance',
        )
    fig_trees.suptitle(f'Reference clustering, {" vs ".join(metrics)} distance — {spectrum_name}',
                       fontsize=13)
    fig_trees.tight_layout()

    fig_diagnostics, (ax_scatter, ax_chance) = plt.subplots(1, 2, figsize=(13, 4.5))

    ax_scatter.scatter(comparison['clusterings'][first]['distances'],
                       comparison['clusterings'][second]['distances'],
                       s=12, alpha=0.5, color='tab:blue')
    ax_scatter.set_xlabel(f'{first} distance')
    ax_scatter.set_ylabel(f'{second} distance')
    ax_scatter.set_title(f'Every reference pair under both metrics\n'
                         f'Spearman rho = {comparison["distance_spearman"]:.5f}', fontsize=10)

    for metric, color in zip(metrics, ('tab:blue', 'tab:green')):
        clustering = comparison['clusterings'][metric]
        # each metric is scaled by its own root height, so two very different distance
        # scales can be read on one axis
        scale = clustering['Z'][-1, 2]
        ax_chance.hist(clustering['chance_merge_heights'] / scale, bins=60, histtype='step',
                       density=True, color=color, label=f'{metric}: randomized')
        ax_chance.axvline(clustering['cutoff_distance'] / scale, color=color, linestyle='--',
                          label=f'{metric}: cutoff')
        ax_chance.plot(clustering['Z'][:, 2] / scale, np.full(clustering['Z'].shape[0], -0.35),
                       marker='|', linestyle='none', color=color, markersize=8)
    ax_chance.set_xlabel('merge height / root merge height')
    ax_chance.set_ylabel('density')
    ax_chance.set_title('How tightly randomized references merge, and the cutoff that gives\n'
                        '(ticks below the axis are the real merges)', fontsize=10)
    ax_chance.legend(fontsize='small')
    fig_diagnostics.tight_layout()

    return fig_trees, fig_diagnostics

In [ ]:
# The real arm: do the two trees disagree about anything that changes how a selection reads?
metric_comparison = compare_cluster_metrics(
    primary_A, ref_names,
    highlight=best_subsets_by_size(fit_summaries[PRIMARY_UNKNOWN]['results']),
)
summarize_cluster_metric_comparison(metric_comparison)
display(plot_cluster_metric_comparison(metric_comparison, PRIMARY_UNKNOWN))

### Findings

**Correlation recovers planted reference groups perfectly at every baseline offset tested;
cosine falls apart as soon as the offsets are non-zero. On the real pool the two metrics rank
pairwise distances almost identically (Spearman 0.991) and still produce different clusters.**

*The synthetic arm is decisive.* References are built in known groups and given baseline offsets
of increasing size; agreement with the planted grouping, 40 replicates per cell:

| offset scale | correlation (mean / worst) | cosine (mean / worst) |
|---|---|---|
| 0.00 | 1.000 / 1.000 | 1.000 / 1.000 |
| 0.25 | **1.000 / 1.000** | 0.504 / 0.068 |
| 0.50 | **1.000 / 1.000** | 0.187 / −0.003 |
| 1.00 | **1.000 / 1.000** | 0.075 / −0.075 |

With no offset the two are identical, which is the check that the harness is fair — cosine is
correlation on centred data, so they must agree there. Add a constant to a spectrum and cosine
rotates its vector toward the all-ones direction, pulling every offset spectrum toward every
other regardless of shape; by an offset scale of 1.0 cosine's agreement with the truth is
indistinguishable from chance, with some replicates scoring below zero. Correlation subtracts the
mean first and never sees the offset at all: 1.000 in all 120 offset replicates, worst case
included.

This is the sharper test the old version of this study could not run. Comparing two metrics on
real references can only report that they differ; it cannot say which is right, because there is
no true grouping to check against.

*The real arm shows the difference is small in rank and large in outcome.* Across the 24-member
pool the two metrics agree closely on the ordering of pairwise distances (Spearman 0.991) and
only one reference changes its nearest neighbour — `arsenate_sorbed_anth_avg_als_cal.e` moves
from `Fh2l_sorbed_arsenate_pH7_10uM` under correlation to `arsenate_sorbed_ram_avg1_2` under
cosine. Yet the flat clusterings agree only moderately (adjusted Rand 0.315): at the cutoff,
correlation splits the pool 19/5 and cosine splits it 14/10. Correlation also gives the cleaner
tree, with cophenetic correlation 0.901 against 0.841. Small, consistent rank differences
accumulate into a different dendrogram, which is what hierarchical clustering does with ties.

The placement of the selected combinations differs accordingly. The best M = 2 and M = 3
combinations both span the full correlation tree (100% of the root height, all 24 leaves), so
under correlation the fit is drawing on chemically distinct references — the behaviour you want.
Under cosine the best M = 2 sits at 40% of the root height inside a single cluster, which would
have read as the fit leaning on near-duplicate references. That reading would have been an
artifact of the metric.

**What follows.** Keep correlation, and the reason is no longer that it looks better on this
pool. XANES references are normalised but not baseline-matched, and cosine is provably sensitive
to exactly that residual. The dendrogram is used here to judge whether a selected combination
spans the pool or hides inside one corner of it, and cosine answers that question wrongly on
synthetic data where the answer is known.